# EnsAff v17 - ucuncu mimari: E(n)-degismez graf agi


Makalenin Tartisma bolumunde su hipotezi one surduk: iki sinir agimizda da mesafe
kanali TOPLAMSAL giriyor ve havuzlamada mesafeye bagli esik yok, bu yuzden hicbir sey
bir temasi temas olarak islemeye zorlamiyor; temas-sayim modeli ise bastan sona mesafe
esiginden ibaret ve geometriyi kullanan tek model o.

**Bu defter o hipotezin dogrudan testidir.** Burada egitilen agda geometri toplamsal
degil: iki dugum arasindaki mesaj, aralarindaki uzakligin bir fonksiyonu olarak
uretilir, yani mesaj mesafeyle kapilanir. Ustelik kenarlar 10 A yaricap esigiyle
kurulur. Temas-sayim modelini geometriye duyarli kildigini dusundugumuz iki ozellik de
burada var.

Ayni taniyi verirse hipotez zayiflar ve bulgu guclenir (geometriye gomulu bir mimari
bile kullanmiyor). Vermezse "geometriyi hangi mimari sinifi kullanir" sorusunun cevabi
bulunmus olur ve makale elestirel olmaktan cikip yapici bir sonuc kazanir. **Her iki
cikti da ise yarar.**

Ag, butun kompleksin otelenmesi ve dondurulmesi altinda degismezdir. Bu bir iddia
degil, sandbox'ta 3e-8 hassasiyetinde dogrulanmis bir ozellik; sondalar bu yuzden
yalnizca BAGIL geometriyi bozanlardir.

Defter ayrica v16'dan kalan iki kucuk acigi kapatir: voksel karsilastirmasini adil
kosullarda tekrarlar ve RMSF kolunun seyrelme kontrolunu kosar.

---

### Donanim ve sure

| Bolum | Donanim | Yuksek RAM | Sure |
|---|---|---|---|
| 1 Kurulum + yapilar | CPU + internet | gerekmez | 0-30 dk (Drive onbellegi varsa dakikalar) |
| A1 Veri onbellegi | CPU | gerekmez | 20-30 dk, bir kez |
| A2 Gercek koordinatlar, 5 tohum | **A100 GPU** | gerekmez | ~1,5 sa |
| A3 Sabit olcu kolu, 3 tohum | **A100 GPU** | gerekmez | ~50 dk |
| A4 Karistirilmis konum kolu, 3 tohum | **A100 GPU** | gerekmez | ~50 dk |
| A5 Sonda bataryasi | **A100 GPU** | gerekmez | ~10 dk |
| B Kucuk duzeltmeler | CPU | gerekmez | ~15 dk |
| RAPOR | CPU | gerekmez | 1 dk |

**Toplam yaklasik 3,5-4,5 saat, A100 ile, yuksek RAM GEREKMEZ.** Sureler tohum basina
~16 dakikalik bir olcumden olceklendi; gercek sure bunun bir buçuk katina kadar
cikabilir. Her asama Drive'a yazilir ve tekrar calistirildiginda atlanir.

`Runtime -> Run all` yeterlidir.

In [ ]:
# =====================================================================
# 1/4 KURULUM + KURTARMA | Donanim: CPU yeterli (GPU kullanilmaz)
# Defterin tamami icin A100 + High-RAM secili olmali.
#
# BU HUCRE AYNI ZAMANDA KURTARMA HUCRESIDIR.
# Drive'in FUSE baglantisi uzun kosularda kopuyor ("Transport endpoint is not
# connected"); o anda calisma dizini Drive icindeyse os.getcwd() bile hata verir
# ve kosu olur. Bunun icin WORK artik YEREL diskte, Drive yalnizca yedek ve her
# asamadan sonra yazilir.
#
# Baglanti kosu sirasinda yine koparsa: RUNTIME'I YENIDEN BASLATMAYIN, sadece bu
# hucreyi tekrar calistirin (once /content'e cikar, Drive'i zorla yeniden baglar,
# calisma dizinini geri kurar), sonra kaldiginiz hucreden devam edin. Tamamlanmis
# kosular atlanir, bastan egitim yapilmaz.
# =====================================================================
import os
os.chdir("/content")        # cwd Drive'da kalmis ve mount dusmusse her sey patlar
BUILD = "2026-10-13-v14-pozitif-kontrol"
import shutil, subprocess, glob
print("BUILD:", BUILD)
subprocess.run("nvidia-smi -L", shell=True)
subprocess.run("pip -q install rdkit", shell=True)

WORK       = "/content/EnsAff"          # yerel: hizli, FUSE kopmasindan etkilenmez
CACHE      = "/content/emb_cache"
STRUCT     = "/content/pdbstruct"
DRIVE_ROOT = "/content/drive/MyDrive/EnsAff"
for d in (WORK, CACHE, STRUCT):
    os.makedirs(d, exist_ok=True)

def drive_ok():
    try:
        os.listdir(DRIVE_ROOT); return True
    except Exception:
        return False

def drive_mount(force=False):
    try: os.chdir("/content")
    except Exception: pass
    try:
        from google.colab import drive
    except Exception as e:
        print("[drive] Colab disi ortam:", e); return False
    if force:
        try: drive.flush_and_unmount()
        except Exception as e: print("[drive] unmount:", e)
    try:
        drive.mount("/content/drive", force_remount=force)
        os.makedirs(DRIVE_ROOT, exist_ok=True)
    except Exception as e:
        print("[drive] baglanamadi:", e); return False
    return drive_ok()

HAS_DRIVE = drive_ok() or drive_mount() or drive_mount(force=True)
print("[drive]", "bagli" if HAS_DRIVE else "YOK -> sonuclar yalnizca yerel diskte kalir")

def _copy(src, dst):
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copyfile(src, dst)

def _sync(src, dst, etiket):
    """Tek deneme + kopma halinde bir kez yeniden baglanip tekrar deneme."""
    global HAS_DRIVE
    for _ in (1, 2):
        if HAS_DRIVE and os.path.exists(src):
            try:
                _copy(src, dst); print(f"[drive] {etiket} tamam"); return True
            except Exception as e:
                print(f"[drive] {etiket} basarisiz ({e}); yeniden baglaniyor...")
                HAS_DRIVE = drive_mount(force=True)
                continue
        return False
    return False

def drive_pull(sub):
    return _sync(os.path.join(DRIVE_ROOT, sub), os.path.join(WORK, sub), f"{sub} -> yerel")

def drive_push(sub="runs"):
    """Yerel -> Drive. Her asamadan sonra cagrilir; BASARISIZ OLSA BILE kosu durmaz,
    cunku rapor her halukarda ekrana basiliyor."""
    if not _sync(os.path.join(WORK, sub), os.path.join(DRIVE_ROOT, sub), f"{sub} -> Drive"):
        print(f"[drive] UYARI: {sub} yedeklenemedi, yalnizca yerel diskte")
        return False
    return True

# v11 kosulari ve veri setleri Drive'da; yerele aliniyor (yoksa indirilecekler)
for sub in ("runs", "LP_PDBBind.csv", "BDB2020+", "mpro", "EGFR"):
    drive_pull(sub)

os.chdir(WORK)
os.makedirs("ensaff", exist_ok=True); open("ensaff/__init__.py", "w").close()
import torch, transformers
print("WORK =", os.getcwd(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("[durum] v11 struct kontrol noktasi:", len(glob.glob("runs/struct_s*/best.pt")),
      "| yapi dosyasi:", len(glob.glob(os.path.join(STRUCT, "*", "ligand.sdf"))))


## Kod

In [ ]:
%%writefile ensaff/features.py
"""Featurization: ligands (RDKit), protein structures (PDB parsing, pocket extraction)."""
from __future__ import annotations

import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem

RDLogger.DisableLog("rdApp.*")

# ----------------------------------------------------------------------------- ligand
ELEMENTS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "P", "B"]
HYBRID = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2, Chem.HybridizationType.SP3]
BOND_TYPES = {Chem.BondType.SINGLE: 1, Chem.BondType.DOUBLE: 2, Chem.BondType.TRIPLE: 3, Chem.BondType.AROMATIC: 4}
N_BOND = 5          # 0 = no bond
MAX_TOPO = 8        # graph distances clipped to 8; 9 = disconnected
N_TOPO = MAX_TOPO + 2
LIG_FEAT_DIM = (len(ELEMENTS) + 1) + 6 + 3 + (len(HYBRID) + 1) + 1 + 1 + 4 + 1  # = 31


def _onehot(v, choices, other=True):
    out = [0.0] * (len(choices) + (1 if other else 0))
    if v in choices:
        out[choices.index(v)] = 1.0
    elif other:
        out[-1] = 1.0
    return out


def atom_features(a: Chem.Atom) -> list[float]:
    f = _onehot(a.GetSymbol(), ELEMENTS)                       # 11
    f += _onehot(min(a.GetDegree(), 5), [0, 1, 2, 3, 4, 5], other=False)  # 6
    f += _onehot(max(-1, min(1, a.GetFormalCharge())), [-1, 0, 1], other=False)  # 3
    f += _onehot(a.GetHybridization(), HYBRID)                 # 4
    f += [float(a.GetIsAromatic())]                            # 1
    f += [float(a.IsInRing())]                                 # 1
    f += _onehot(min(a.GetTotalNumHs(), 3), [0, 1, 2, 3], other=False)  # 4
    f += [float(a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED)]  # 1
    return f


def featurize_mol(mol: Chem.Mol, max_atoms: int = 72, min_atoms: int = 5):
    """Heavy-atom graph. Returns dict(x [N,F], bond [N,N] int8, topo [N,N] int8) or None."""
    if mol is None:
        return None
    mol = Chem.RemoveHs(mol, sanitize=False)
    n = mol.GetNumAtoms()
    if n < min_atoms or n > max_atoms:
        return None
    x = np.asarray([atom_features(a) for a in mol.GetAtoms()], dtype=np.float32)
    bond = np.zeros((n, n), dtype=np.int8)
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        t = BOND_TYPES.get(b.GetBondType(), 1)
        bond[i, j] = bond[j, i] = t
    topo = Chem.GetDistanceMatrix(mol)
    topo = np.where(topo > 1e6, MAX_TOPO + 1, np.minimum(topo, MAX_TOPO)).astype(np.int8)
    return {"x": x, "bond": bond, "topo": topo}


def featurize_smiles(smiles: str, max_atoms: int = 72):
    mol = Chem.MolFromSmiles(str(smiles))
    return featurize_mol(mol, max_atoms=max_atoms)


def ecfp(smiles: str, n_bits: int = 2048, radius: int = 2) -> np.ndarray:
    mol = Chem.MolFromSmiles(str(smiles))
    arr = np.zeros(n_bits, dtype=np.uint8)
    if mol is None:
        return arr
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    on = list(fp.GetOnBits())
    arr[on] = 1
    return arr


def load_ligand_poses(sdf_path: str, max_poses: int = 8):
    """Read all poses from an SDF. Returns (mol_of_first_pose, [coords [N,3] heavy atoms]) or (None, [])."""
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
    mols = [m for m in suppl if m is not None]
    if not mols:  # retry without sanitization
        suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=False)
        mols = []
        for m in suppl:
            if m is None:
                continue
            try:
                m.UpdatePropertyCache(strict=False)
                Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_FINDRADICALS | Chem.SanitizeFlags.SANITIZE_SETAROMATICITY
                                 | Chem.SanitizeFlags.SANITIZE_SETCONJUGATION | Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION,
                                 catchErrors=True)
                mols.append(m)
            except Exception:
                continue
    if not mols:
        return None, []
    ref = Chem.RemoveHs(mols[0], sanitize=False)
    n = ref.GetNumAtoms()
    poses = []
    for m in mols[:max_poses]:
        m = Chem.RemoveHs(m, sanitize=False)
        if m.GetNumAtoms() != n or m.GetNumConformers() == 0:
            continue
        poses.append(m.GetConformer().GetPositions().astype(np.float32))
    return ref, poses


# ----------------------------------------------------------------------------- protein
AA3 = {"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H",
       "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
       "TYR": "Y", "VAL": "V", "MSE": "M", "SEC": "C", "HID": "H", "HIE": "H", "HIP": "H", "CYX": "C", "ASH": "D",
       "GLH": "E", "LYN": "K"}


def parse_pdb(path: str):
    """Minimal PDB parser.

    Returns [{'id', 'seq', 'coords': [[n_atoms,3] per residue], 'bfac': [[n_atoms] per residue]}].
    B-factors are the crystallographic displacement parameters: a per-atom experimental measure of how
    mobile that atom is. They are the one dynamics signal present in essentially every structure, and are
    carried here so the model can use observed flexibility rather than a static picture alone.
    """
    chains: dict[str, dict] = {}
    order: list[str] = []
    last_key = None
    with open(path) as fh:
        for line in fh:
            rec = line[:6]
            if rec.startswith("ENDMDL"):
                break  # first model only
            if not (rec.startswith("ATOM") or (rec.startswith("HETATM") and line[17:20] == "MSE")):
                continue
            resn = line[17:20].strip()
            if resn not in AA3:
                continue
            alt = line[16]
            if alt not in (" ", "A", "1"):
                continue
            elem = line[76:78].strip() or line[12:16].strip()[0]
            if elem.upper() == "H" or line[12:16].strip().startswith("H"):
                continue
            ch = line[21]
            key = (ch, line[22:26], line[26])
            xyz = (float(line[30:38]), float(line[38:46]), float(line[46:54]))
            try:
                b = float(line[60:66])
            except ValueError:
                b = 0.0
            if ch not in chains:
                chains[ch] = {"id": ch, "seq": [], "coords": [], "bfac": []}
                order.append(ch)
            c = chains[ch]
            if key != last_key:
                c["seq"].append(AA3[resn])
                c["coords"].append([])
                c["bfac"].append([])
                last_key = key
            c["coords"][-1].append(xyz)
            c["bfac"][-1].append(b)
    out = []
    for ch in order:
        c = chains[ch]
        out.append({"id": ch, "seq": "".join(c["seq"]),
                    "coords": [np.asarray(r, dtype=np.float32) for r in c["coords"]],
                    "bfac": [np.asarray(r, dtype=np.float32) for r in c["bfac"]]})
    return out


def extract_pocket(chains, poses, cutoff: float = 10.0, max_res: int = 64):
    """Select residues whose heavy atoms come within `cutoff` Å of any ligand atom (any pose).

    Returns list of (chain_idx, res_idx) and distance tensor D [K, P, N] (min residue-atom distance per pose).
    """
    lig_all = np.concatenate(poses, 0)
    cand = []
    for ci, ch in enumerate(chains):
        for ri, rc in enumerate(ch["coords"]):
            d = np.sqrt(((rc[:, None, :] - lig_all[None, :, :]) ** 2).sum(-1)).min()
            if d < cutoff:
                cand.append((d, ci, ri))
    if not cand:
        return [], None
    cand.sort()
    cand = sorted(cand[:max_res], key=lambda t: (t[1], t[2]))
    sel = [(ci, ri) for _, ci, ri in cand]
    D = np.zeros((len(poses), len(sel), poses[0].shape[0]), dtype=np.float32)
    for k, lig in enumerate(poses):
        for p, (ci, ri) in enumerate(sel):
            rc = chains[ci]["coords"][ri]
            D[k, p] = np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
    return sel, D



def extract_pocket_random(chains, poses, max_res: int = 64, rng=None):
    """CONTROL arm: `max_res` residues drawn uniformly at random from the whole protein, with their REAL
    distances to the ligand.

    `extract_pocket` does two things at once: it picks which residues the model sees, and it hands over the
    geometry of those residues. If structure mode beats the sequence baseline, either could be the reason.
    This keeps the geometry machinery untouched and only destroys the selection, so the two can be told
    apart.
    """
    rng = rng or np.random
    lig_all = np.concatenate(poses, 0)
    allres = [(ci, ri) for ci, ch in enumerate(chains) for ri in range(len(ch["coords"]))]
    if not allres:
        return [], None
    k = min(max_res, len(allres))
    idx = rng.choice(len(allres), size=k, replace=False)
    sel = sorted([allres[i] for i in idx])
    D = np.zeros((len(poses), len(sel), poses[0].shape[0]), dtype=np.float32)
    for k_, lig in enumerate(poses):
        for p_, (ci, ri) in enumerate(sel):
            rc = chains[ci]["coords"][ri]
            D[k_, p_] = np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
    return sel, D


def extract_pocket_wide(chains, poses, cutoff: float = 10.0, max_res: int = 128,
                        n_true: int = 48, rng=None):
    """True contact residues HIDDEN among decoy residues drawn from the rest of the protein.

    `extract_pocket` picks residues by distance, so the question "which residues touch the ligand" is
    answered by the featuriser and never reaches the model. v12 measured the consequence: training with
    every distance set to a constant cost 0.000 RMSE, because the residue list already carried all the
    structural signal the model used.

    Here the model gets `n_true` contact residues plus decoys up to `max_res`, merged into sequence order
    so position gives nothing away. The distance matrix is now the ONLY way to tell contacts from decoys,
    which makes "can this architecture read geometry when it has to" a question the data can answer.
    """
    rng = rng or np.random
    true_sel, _ = extract_pocket(chains, poses, cutoff=cutoff, max_res=n_true)
    if not true_sel:
        return [], None
    allres = [(ci, ri) for ci, ch in enumerate(chains) for ri in range(len(ch["coords"]))]
    pool = [t for t in allres if t not in set(true_sel)]
    n_decoy = min(max(0, max_res - len(true_sel)), len(pool))
    decoys = [pool[i] for i in rng.choice(len(pool), size=n_decoy, replace=False)] if n_decoy else []
    sel = sorted(set(true_sel) | set(decoys))
    D = np.zeros((len(poses), len(sel), poses[0].shape[0]), dtype=np.float32)
    for k, lig in enumerate(poses):
        for p_, (ci, ri) in enumerate(sel):
            rc = chains[ci]["coords"][ri]
            D[k, p_] = np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
    return sel, D

def residue_flexibility(chains, pocket):
    """Per-pocket-residue flexibility features from B-factors, standardised WITHIN the structure.

    Absolute B-factors are not comparable across entries (resolution, refinement protocol and overall
    scaling differ; across BDB2020+ the per-structure mean ranges from 0 to 131). What is comparable is
    how mobile a residue is *relative to the rest of its own structure*, so each feature is a z-score
    against that structure's own atoms.

    Returns [K, 2], or None when the structure's B-factor column is constant (no information).
    """
    allb = np.concatenate([b for c in chains for b in c["bfac"] if len(b)]) if chains else np.zeros(1)
    mu, sd = float(allb.mean()), float(allb.std())
    if sd < 1e-6:
        # Constant B-factor column: the deposited file carries no flexibility information at all (every
        # mpro/*/protein.pdb in this project is like this). Returning zeros would hand the model a constant
        # it never saw in training and silently confound the comparison, so the caller is told instead.
        return None
    out = np.zeros((len(pocket), 2), dtype=np.float32)
    for i, (ci, ri) in enumerate(pocket):
        b = chains[ci]["bfac"][ri]
        if len(b):
            out[i, 0] = (float(b.mean()) - mu) / sd
            out[i, 1] = float(b.std()) / sd
    return np.clip(out, -5.0, 5.0)


In [ ]:
%%writefile ensaff/embed.py
"""Per-residue protein embeddings with on-disk caching.

backend="esm"    : HuggingFace ESM-2 (facebook/esm2_t12_35M_UR50D by default). Use on Colab GPU.
backend="onehot" : 21-d amino-acid one-hot (no download; for tests and as an ablation).
"""
from __future__ import annotations

import hashlib
import os

import numpy as np
import torch

AA = "ACDEFGHIKLMNPQRSTVWY"
CHUNK = 1000  # ESM-2 handles <=1022 residues; longer chains are embedded in windows


def _key(seq: str) -> str:
    return hashlib.md5(seq.encode()).hexdigest()


def onehot_embed(seq: str) -> np.ndarray:
    out = np.zeros((len(seq), 21), dtype=np.float16)
    for i, c in enumerate(seq):
        j = AA.find(c)
        out[i, j if j >= 0 else 20] = 1
    return out


class ProteinEmbedder:
    def __init__(self, backend: str = "esm", model_name: str = "facebook/esm2_t12_35M_UR50D",
                 cache_path: str | None = None, device: str | None = None, batch_tokens: int = 8000,
                 save_every: int = 800):
        self.backend = backend
        self.model_name = model_name
        self.cache_path = cache_path
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_tokens = batch_tokens
        self.save_every = save_every
        self.cache: dict[str, np.ndarray] = {}
        self._persisted: set[str] = set()
        self.shard_dir = (cache_path + ".shards") if cache_path else None
        if cache_path:
            os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
            if os.path.exists(cache_path):            # legacy single-file cache
                self.cache = torch.load(cache_path, weights_only=False)
                print(f"[embed] loaded {len(self.cache)} cached chains from {cache_path}")
            if os.path.isdir(self.shard_dir):         # sharded cache (the current format)
                import glob as _glob
                for sh in sorted(_glob.glob(os.path.join(self.shard_dir, "shard_*.pt"))):
                    try:
                        self.cache.update(torch.load(sh, weights_only=False))
                    except Exception as e:
                        print(f"[embed] bozuk parca atlandi {sh}: {e}")
                print(f"[embed] loaded {len(self.cache)} cached chains from {self.shard_dir}")
            self._persisted = set(self.cache)
        self._model = None
        self._tok = None

    @property
    def dim(self) -> int:
        if self.backend == "onehot":
            return 21
        if self.cache:
            return next(iter(self.cache.values())).shape[1]
        self._load()
        return self._model.config.hidden_size

    def _load(self):
        if self._model is None:
            from transformers import AutoTokenizer, EsmModel
            self._tok = AutoTokenizer.from_pretrained(self.model_name)
            self._model = EsmModel.from_pretrained(self.model_name, add_pooling_layer=False).to(self.device).eval()
            if self.device == "cuda":
                self._model = self._model.half()

    @torch.no_grad()
    def _esm_batch(self, seqs: list[str]) -> list[np.ndarray]:
        self._load()
        enc = self._tok(seqs, return_tensors="pt", padding=True).to(self.device)
        h = self._model(**enc).last_hidden_state.float().cpu().numpy()
        return [h[i, 1:len(s) + 1].astype(np.float16) for i, s in enumerate(seqs)]

    def _embed_chain_uncached(self, seqs: list[str]):
        if self.backend == "onehot":
            for s in seqs:
                self.cache[_key(s)] = onehot_embed(s)
            self.save(quiet=False)
            return
        # split long chains into windows, embed by length-sorted batches
        pieces = []
        for s in seqs:
            for st in range(0, len(s), CHUNK):
                pieces.append((s, st, s[st:st + CHUNK]))
        pieces.sort(key=lambda t: len(t[2]))
        npieces = {s: (len(s) + CHUNK - 1) // CHUNK for s in seqs}
        out: dict[tuple, np.ndarray] = {}
        got: dict[str, int] = {s: 0 for s in seqs}
        since_save, i = 0, 0
        from tqdm.auto import tqdm
        bar = tqdm(total=len(pieces), desc="ESM", disable=len(pieces) < 50)
        while i < len(pieces):
            j = i
            while j < len(pieces) and (j - i + 1) * (len(pieces[j][2]) + 2) <= self.batch_tokens:
                j += 1
            j = max(j, i + 1)
            batch = pieces[i:j]
            embs = self._esm_batch([p[2] for p in batch])
            for (s, st, _), e in zip(batch, embs):
                out[(s, st)] = e
                got[s] += 1
                if got[s] == npieces[s]:  # chain complete -> move to cache, free windows
                    self.cache[_key(s)] = np.concatenate([out.pop((s, k)) for k in range(0, len(s), CHUNK)], 0)
                    since_save += 1
            bar.update(j - i)
            if since_save >= self.save_every:  # periodic checkpoint so Ctrl+C never loses much
                bar.clear()
                self.save(quiet=False)
                since_save = 0
                bar.refresh()
            i = j
        bar.close()
        self.save(quiet=False)

    def embed_chains(self, seqs: list[str]):
        todo = sorted({s for s in seqs if _key(s) not in self.cache})
        if todo:
            print(f"[embed] embedding {len(todo)} new chains with {self.backend}")
            self._embed_chain_uncached(todo)
            self.save()

    def get(self, seq: str) -> np.ndarray:
        return self.cache[_key(seq)]

    def save(self, quiet: bool = True):
        """Write only the chains not yet on disk, as a new shard.

        The cache can reach tens of GB in structure mode (full chains at 1280 dims), so rewriting the whole
        cache at every checkpoint would dominate the run. Each save costs only what is new.
        """
        if not self.cache_path:
            return
        new_keys = [k for k in self.cache if k not in self._persisted]
        if not new_keys:
            return
        os.makedirs(self.shard_dir, exist_ok=True)
        idx = len([f for f in os.listdir(self.shard_dir) if f.startswith("shard_")])
        path = os.path.join(self.shard_dir, f"shard_{idx:04d}.pt")
        mb = sum(self.cache[k].nbytes for k in new_keys) / 1e6
        if not quiet:
            print(f"[embed] saving {len(new_keys)} new chains ({mb:.0f} MB) -> {os.path.basename(path)} "
                  f"— DO NOT interrupt...", flush=True)
        tmp = path + ".tmp"
        torch.save({k: self.cache[k] for k in new_keys}, tmp)
        os.replace(tmp, path)             # atomic: an interrupted write never leaves a half shard
        self._persisted.update(new_keys)
        if not quiet:
            print(f"[embed] saved. toplam onbellek: {len(self.cache)} zincir", flush=True)


In [ ]:
%%writefile ensaff/data.py
"""Datasets: sequence mode (sequence + SMILES) and structure mode (pocket + ligand poses)."""
from __future__ import annotations

import hashlib
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from .features import (MAX_TOPO, extract_pocket, extract_pocket_random, extract_pocket_wide, featurize_mol, featurize_smiles, load_ligand_poses,
                       parse_pdb, residue_flexibility)

LP_URL = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv"
BDB_URL = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/BDB2020+.tgz"
MPRO_URL = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/mpro.tgz"
EGFR_URL = "https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/EGFR.tgz"


# ----------------------------------------------------------------------------- tables
def load_lp_pdbbind(path: str = "LP_PDBBind.csv", clean_level: str = "CL1", drop_covalent: bool = True):
    """LP-PDBBind leak-proof splits (Li et al.). Returns dict split -> DataFrame[id, smiles, seq, y]."""
    if not os.path.exists(path):
        import urllib.request
        urllib.request.urlretrieve(LP_URL, path)
    d = pd.read_csv(path).rename(columns={"Unnamed: 0": "id", "value": "y"})
    if clean_level:
        d = d[d[clean_level].astype(bool)]
    if drop_covalent:
        d = d[~d["covalent"].astype(bool)]
    d = d[~((d["new_split"] == "val") & d["remove_for_balancing_val"].astype(bool))]
    d = d.dropna(subset=["smiles", "seq", "y"])
    return {s: d[d["new_split"] == s][["id", "smiles", "seq", "y"]].reset_index(drop=True)
            for s in ["train", "val", "test"]}


def load_bdb2020(root: str = "BDB2020+"):
    """BDB2020+ external test set (structures released after PDBbind 2020)."""
    if not os.path.exists(root):
        import tarfile
        import urllib.request
        urllib.request.urlretrieve(BDB_URL, "BDB2020+.tgz")
        tarfile.open("BDB2020+.tgz").extractall(".")
    d = pd.read_csv(os.path.join(root, "BDB2020+.csv")).rename(columns={"pdbid": "id", "pKa": "y"})
    return d[["id", "smiles", "seq", "y"]].reset_index(drop=True)


def split_chains(seq: str) -> list[str]:
    return [c for c in str(seq).split(":") if c]


# ----------------------------------------------------------------------------- sequence mode
class SeqAffinityDataset(Dataset):
    """Protein = full chain sequence(s) -> per-residue embeddings; ligand = 2D graph from SMILES."""

    def __init__(self, df: pd.DataFrame, embedder, max_atoms: int = 72, name: str = ""):
        self.recs = []
        chains_all = []
        skipped = 0
        for r in df.itertuples():
            lig = featurize_smiles(r.smiles, max_atoms=max_atoms)
            chains = split_chains(r.seq)
            if lig is None or not chains:
                skipped += 1
                continue
            self.recs.append({"id": r.id, "y": float(r.y), "chains": chains, "lig": lig})
            chains_all += chains
        embedder.embed_chains(chains_all)
        self.emb = embedder
        print(f"[data:{name}] {len(self.recs)} complexes ({skipped} skipped)")

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        prot = np.concatenate([self.emb.get(c) for c in r["chains"]], 0).astype(np.float32)
        return {"id": r["id"], "y": r["y"], "prot": prot, **r["lig"], "dist": None}



# ----------------------------------------------------------------------------- distance-channel probes
def _rand_rotation(rng):
    """Uniform random rotation (QR of a Gaussian matrix, reflection removed)."""
    q, r = np.linalg.qr(rng.randn(3, 3))
    q = q * np.sign(np.diag(r))
    if np.linalg.det(q) < 0:
        q[:, 0] = -q[:, 0]
    return q.astype(np.float32)


def _recompute_D(pocket_xyz, lig):
    return np.stack([np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
                     for rc in pocket_xyz]).astype(np.float32)[None]


def _perturb_D(D, r, spec: str, salt: str = "", rng=None):
    """Controlled damage to the residue-atom distance channel. `spec` is name or name:value.

      none            unchanged
      const:v         every distance set to v        -> geometry removed (RBF term becomes a constant bias)
      permute_res     residue axis shuffled          -> right distances, wrong residues
      permute_atom    ligand-atom axis shuffled      -> right distances, wrong atoms
      scale:f         all distances multiplied by f
      offset:v        v angstrom added to every distance
      translate:s     ligand rigidly displaced, s = per-axis sd (needs coordinates)
      rotate          ligand rotated about its own centroid (needs coordinates)

    Every variant is seeded from the complex id, so the same spec gives the same perturbation on every run
    and two models can be compared on identical damage.
    """
    name, _, val = spec.partition(":")
    if rng is None:   # stable per-complex seed, so a probe repeats exactly across runs and models
        rng = np.random.RandomState(int(hashlib.md5(f"{salt}{name}{val}{r['id']}".encode()).hexdigest()[:8], 16))
    if name == "none":
        return D
    if name == "const":
        return np.full_like(D, float(val or 8.0))
    if name == "scale":
        return (D * float(val)).astype(np.float32)
    if name == "offset":
        return (D + float(val)).astype(np.float32)
    if name == "permute_res":
        return D[:, rng.permutation(D.shape[1]), :]
    if name == "permute_atom":
        return D[:, :, rng.permutation(D.shape[2])]
    if name in ("translate", "rotate"):
        if r.get("pose0") is None or r.get("pocket_xyz") is None:
            raise ValueError(f"'{name}' sondasi koordinat ister; veri kumesini keep_coords=True ile kurun")
        lig = r["pose0"]
        if name == "translate":
            lig = lig + rng.randn(1, 3).astype(np.float32) * float(val)
        else:
            c = lig.mean(0, keepdims=True)
            lig = (lig - c) @ _rand_rotation(rng).T + c
        return _recompute_D(r["pocket_xyz"], lig)
    raise ValueError(f"bilinmeyen sonda: {spec}")


# ----------------------------------------------------------------------------- structure mode
class StructAffinityDataset(Dataset):
    """Protein = binding-pocket residues (ESM embeddings of their chain); ligand = graph from SDF;
    states = ligand poses (all poses in the SDF, optionally plus jittered copies)."""

    def __init__(self, df: pd.DataFrame, embedder, protein_pattern: str, ligand_pattern: str,
                 cutoff: float = 10.0, max_res: int = 64, max_atoms: int = 72, max_poses: int = 4,
                 jitter_states: int = 0, jitter_sigma: float = 0.5, name: str = "", eval_states: int = 0,
                 use_bfactor: bool = False, geom: str = "real", pocket_mode: str = "nearest",
                 keep_coords: bool = False, decoy_spec: str = "translate", n_true: int = 48):
        self.recs, chains_all, skipped = [], [], 0
        degenerate = 0
        self.jitter_states, self.jitter_sigma = jitter_states, jitter_sigma
        # eval_states > 0 turns the multi-state aggregation on at INFERENCE too. With a single crystal pose
        # the aggregation is mathematically the identity (K=1), so without this it is only ever exercised as
        # training-time augmentation and its effect on predictions cannot be measured.
        self.eval_states = eval_states
        self.use_bfactor = use_bfactor
        # geom: training-time control over the distance channel; probe: inference-time perturbation.
        # Both go through _perturb_D so a control arm and a probe cannot disagree about what "const" means.
        self.geom, self.pocket_mode, self.probe = geom, pocket_mode, "none"
        # decoy_spec shapes the extra TRAINING states. "rotate" turns the ligand about its own centroid,
        # which leaves every residue-to-centroid distance untouched and changes only which atom faces
        # which residue, so a model that can tell a decoy apart must be reading atom-level geometry
        # rather than a shift in the overall distance distribution.
        self.decoy_spec = decoy_spec
        for r in df.itertuples():
            pp, lp = protein_pattern.format(id=r.id), ligand_pattern.format(id=r.id)
            if not (os.path.exists(pp) and os.path.exists(lp)):
                skipped += 1
                continue
            mol, poses = load_ligand_poses(lp, max_poses=max_poses)
            lig = featurize_mol(mol, max_atoms=max_atoms) if mol is not None else None
            if lig is None or not poses:
                skipped += 1
                continue
            chains = parse_pdb(pp)
            if pocket_mode in ("random", "wide"):
                prng = np.random.RandomState(int(hashlib.md5(f"pock{r.id}".encode()).hexdigest()[:8], 16))
                if pocket_mode == "random":
                    sel, D = extract_pocket_random(chains, poses, max_res=max_res, rng=prng)
                else:
                    sel, D = extract_pocket_wide(chains, poses, cutoff=cutoff, max_res=max_res,
                                                 n_true=n_true, rng=prng)
            else:
                sel, D = extract_pocket(chains, poses, cutoff=cutoff, max_res=max_res)
            if not sel:
                skipped += 1
                continue
            seqs = [c["seq"] for c in chains]
            flex = residue_flexibility(chains, sel) if use_bfactor else None
            if use_bfactor and flex is None:
                degenerate += 1
                flex = np.zeros((len(sel), 2), dtype=np.float32)
            keep_xyz = (jitter_states > 0 or eval_states > 0 or keep_coords
                        or geom in ("translate", "rotate"))
            self.recs.append({"id": r.id, "y": float(r.y), "chain_seqs": seqs, "pocket": sel, "D": D,
                              "lig": lig, "flex": flex,
                              "pose0": poses[0] if keep_xyz else None,
                              "pocket_xyz": [chains[ci]["coords"][ri] for ci, ri in sel] if keep_xyz else None})
            chains_all += seqs
        embedder.embed_chains(chains_all)
        self.emb = embedder
        self.train_mode = False
        msg = f"[data:{name}] {len(self.recs)} complexes ({skipped} skipped)"
        if geom != "real" or pocket_mode != "nearest":
            msg += f"  [KONTROL KOLU geom={geom} pocket={pocket_mode}]"
        if use_bfactor and degenerate:
            msg += (f"  !! UYARI: {degenerate}/{len(self.recs)} yapida B-faktor sutunu SABIT "
                    f"(bilgi yok, sifirlandi) -> bu sette B-faktor karsilastirmasi gecersiz")
        print(msg)
        self.n_degenerate_bfactor = degenerate

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        prot = np.stack([self.emb.get(r["chain_seqs"][ci])[ri] for ci, ri in r["pocket"]]).astype(np.float32)
        if r.get("flex") is not None:          # observed flexibility appended to the residue embedding
            prot = np.concatenate([prot, r["flex"]], axis=1)
        D = r["D"]
        n_real = D.shape[0]          # states 0..n_real-1 are crystal poses; the rest are decoys
        n_extra = self.jitter_states if self.train_mode else self.eval_states
        if n_extra > 0 and r.get("pose0") is not None and self.decoy_spec != "translate":
            # fresh decoys every epoch in training, repeatable ones at evaluation
            rng = np.random if self.train_mode else None
            extra = [_perturb_D(D[:1], r, self.decoy_spec, salt=f"decoy{_i}", rng=rng)[0]
                     for _i in range(n_extra)]
            D = np.concatenate([D, np.stack(extra)], 0)
        elif n_extra > 0 and r.get("pose0") is not None:
            # Perturb the LIGAND COORDINATES and recompute distances. Adding i.i.d. noise straight to the
            # distance matrix (what this did before) is not realisable by any rigid or flexible pose change,
            # so it could not stand in for an alternative bound state. A rigid-body jiggle of the ligand can.
            if self.train_mode:
                rng = np.random
            else:   # stable per-complex seed: Python's hash() is randomised per process
                rng = np.random.RandomState(int(hashlib.md5(str(r["id"]).encode()).hexdigest()[:8], 16))
            extra = []
            for _ in range(n_extra):
                shift = rng.randn(1, 3).astype(np.float32) * self.jitter_sigma
                lig = r["pose0"] + shift                      # rigid translation of the whole ligand
                dd = np.stack([np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
                               for rc in r["pocket_xyz"]])
                extra.append(dd.astype(np.float32))
            D = np.concatenate([D, np.stack(extra)], 0)
        if self.geom != "real":
            D = _perturb_D(D, r, self.geom, salt="geom")
        if self.probe and self.probe != "none":
            D = _perturb_D(D, r, self.probe, salt="probe")
        return {"id": r["id"], "y": r["y"], "prot": prot, **r["lig"], "dist": D, "n_real": n_real}


# ----------------------------------------------------------------------------- collate
def collate(items):
    B = len(items)
    L = max(it["prot"].shape[0] for it in items)
    N = max(it["x"].shape[0] for it in items)
    E = items[0]["prot"].shape[1]
    F = items[0]["x"].shape[1]
    prot = torch.zeros(B, L, E)
    pmask = torch.zeros(B, L, dtype=torch.bool)
    x = torch.zeros(B, N, F)
    lmask = torch.zeros(B, N, dtype=torch.bool)
    bond = torch.zeros(B, N, N, dtype=torch.long)
    topo = torch.full((B, N, N), MAX_TOPO + 1, dtype=torch.long)
    y = torch.tensor([it["y"] for it in items], dtype=torch.float32)
    has_dist = items[0]["dist"] is not None
    if has_dist:
        K = max(it["dist"].shape[0] for it in items)
        dist = torch.zeros(B, K, L, N)
        smask = torch.zeros(B, K, dtype=torch.bool)
    for b, it in enumerate(items):
        l, n = it["prot"].shape[0], it["x"].shape[0]
        prot[b, :l] = torch.from_numpy(it["prot"])
        pmask[b, :l] = True
        x[b, :n] = torch.from_numpy(it["x"])
        lmask[b, :n] = True
        bond[b, :n, :n] = torch.from_numpy(it["bond"].astype(np.int64))
        topo[b, :n, :n] = torch.from_numpy(it["topo"].astype(np.int64))
        if has_dist:
            k = it["dist"].shape[0]
            dist[b, :k, :l, :n] = torch.from_numpy(it["dist"])
            smask[b, :k] = True
    batch = {"ids": [it["id"] for it in items], "prot": prot, "prot_mask": pmask, "lig_x": x, "lig_mask": lmask,
             "bond": bond, "topo": topo, "y": y}
    if has_dist:
        batch["dist"], batch["state_mask"] = dist, smask
        batch["n_real"] = torch.tensor([it.get("n_real", 1) for it in items], dtype=torch.long)
    return batch


# ----------------------------------------------------------------------------- extra external test sets
def _ensure(url: str, tgz: str, root: str):
    if not os.path.exists(root):
        import tarfile
        import urllib.request
        if not os.path.exists(tgz):
            urllib.request.urlretrieve(url, tgz)
        tarfile.open(tgz).extractall(".")
    return os.path.exists(root)


def _seq_from_pdb(path: str) -> str:
    """Chain sequences joined with ':' - the format split_chains() expects."""
    return ":".join(c["seq"] for c in parse_pdb(path) if len(c["seq"]) >= 20)


def load_mpro(root: str = "mpro"):
    """SARS-CoV-2 main protease: 40 ligands against ONE target. Tests ligand ranking within a target,
    which is the lead-optimisation question, as opposed to BDB2020+ which tests new targets."""
    if not _ensure(MPRO_URL, "mpro.tgz", root):
        return None, None, None
    d = pd.read_csv(os.path.join(root, "info.csv"))
    col = "pKa_reliable" if "pKa_reliable" in d.columns else "pKa_full"
    rows = []
    for r in d.itertuples():
        pp = os.path.join(root, str(r.pdbid), "protein.pdb")
        lp = os.path.join(root, str(r.pdbid), "ligand.sdf")
        y = getattr(r, col, None)
        if not (os.path.exists(pp) and os.path.exists(lp)) or pd.isna(y):
            continue
        rows.append({"id": str(r.pdbid), "smiles": r.smiles, "seq": _seq_from_pdb(pp), "y": float(y)})
    df = pd.DataFrame(rows)
    return df, os.path.join(root, "{id}", "protein.pdb"), os.path.join(root, "{id}", "ligand.sdf")


def load_egfr(root: str = "EGFR"):
    """EGFR kinase: 23 ligands against one target (several point mutants)."""
    if not _ensure(EGFR_URL, "EGFR.tgz", root):
        return None, None, None
    d = pd.read_csv(os.path.join(root, "info.csv"))
    rows = []
    for r in d.itertuples():
        pid = str(r.pdbids)
        pp = os.path.join(root, "protein", f"{pid}.pdb")
        lp = os.path.join(root, "ligand", f"{pid}.sdf")
        if not (os.path.exists(pp) and os.path.exists(lp)) or pd.isna(r.value):
            continue
        rows.append({"id": pid, "smiles": r.smiles, "seq": _seq_from_pdb(pp), "y": float(r.value)})
    df = pd.DataFrame(rows)
    return df, os.path.join(root, "protein", "{id}.pdb"), os.path.join(root, "ligand", "{id}.sdf")


def load_external_sets(data_dir: str = ".", exclude_ids=None, clean_level: str = "CL1"):
    """External test sets, as {name: (df, protein_pattern, ligand_pattern)}.

    Complexes that also appear in the LP-PDBBind val/test splits are REMOVED. Without this the EGFR set
    is not external at all: 11 of its 23 entries are the test split re-scored and 2 are in val, which is
    both the early-stopping and the conformal-calibration set. Keeping them would double-count evidence
    and, for the val ones, score the model on data used to select it.
    """
    if exclude_ids is None:
        try:
            tabs = load_lp_pdbbind(os.path.join(data_dir, "LP_PDBBind.csv"), clean_level=clean_level)
            exclude_ids = set(pd.concat([tabs["val"], tabs["test"]])["id"].astype(str).str.lower())
        except Exception:
            exclude_ids = set()
    out = {}
    bdb = load_bdb2020(os.path.join(data_dir, "BDB2020+"))
    bdb_root = os.path.join(data_dir, "BDB2020+", "dataset")
    cand = [("BDB2020+", bdb, os.path.join(bdb_root, "{id}", "protein.pdb"),
             os.path.join(bdb_root, "{id}", "ligand.sdf"))]
    for name, fn, sub in (("mpro", load_mpro, "mpro"), ("EGFR", load_egfr, "EGFR")):
        try:
            df, pp, lp = fn(os.path.join(data_dir, sub))
            if df is not None and len(df):
                cand.append((name, df, pp, lp))
        except Exception as e:
            print(f"[data] {name} yuklenemedi: {e}")
    for name, df, pp, lp in cand:
        n0 = len(df)
        keep = ~df["id"].astype(str).str.lower().isin(exclude_ids)
        df = df[keep].reset_index(drop=True)
        if n0 != len(df):
            print(f"[data] {name}: LP val/test ile ortusen {n0 - len(df)} kompleks cikarildi "
                  f"-> {len(df)} kaldi (dis set olarak gecerli)")
        if len(df) >= 10:
            out[name] = (df, pp, lp)
        else:
            print(f"[data] {name}: tekillestirme sonrasi yalnizca {len(df)} kompleks kaldi -> KULLANILMIYOR")
    return out


In [ ]:
%%writefile ensaff/model.py
"""EnsAff: AlphaFold-style pair network for protein-ligand binding affinity.

Ideas borrowed from AlphaFold 2 and adapted to a bipartite (protein residue x ligand atom) pair representation:
  * pair representation z[i, j] updated by axial attention along both axes (Evoformer-style)
  * "ligand-triangle" multiplicative update: z[i, j] <- sum_l z[i, l] * e[l, j] using known ligand edges
    (bond type + graph distance), the bipartite analogue of the triangle multiplicative update
  * recycling of z and single representations (stop-gradient on earlier cycles)
  * a per-prediction uncertainty head (Gaussian mean + variance), later calibrated with split-conformal

New element: several protein-ligand states (poses / conformations) are scored separately and combined with a
learnable soft-max over states, pK = tau * logsumexp(pK_k / tau) - tau * log K, which mirrors how a binding free
energy aggregates over an ensemble of bound states (tau is initialised at kT in pK units, ~0.43 at 298 K).

Two protein modes:
  * "seq"    : no structure. A ligand-conditioned latent pocket (Perceiver-style cross-attention of M learned
               slots over all residues) compresses the protein; its attention map is a predicted binding site.
  * "struct" : pocket residues from a structure; residue-atom distances of every pose enter the pair init.
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .features import LIG_FEAT_DIM, N_BOND, N_TOPO

NEG = -1e4


def masked_mean(x, mask, dim):
    m = mask.unsqueeze(-1).to(x.dtype)
    return (x * m).sum(dim) / m.sum(dim).clamp(min=1.0)


class Transition(nn.Module):
    def __init__(self, d, mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, mult * d), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(mult * d, d))

    def forward(self, x):
        return self.net(x)


# ----------------------------------------------------------------------------- ligand encoder
class LigandEncoder(nn.Module):
    """Edge-aware message passing on the heavy-atom graph."""

    def __init__(self, d, n_layers=3, dropout=0.0):
        super().__init__()
        self.inp = nn.Linear(LIG_FEAT_DIM, d)
        self.bond = nn.ModuleList([nn.Embedding(N_BOND, d) for _ in range(n_layers)])
        self.norm = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
        self.mlp = nn.ModuleList([Transition(d, 2, dropout) for _ in range(n_layers)])

    def forward(self, x, bond, mask):
        h = self.inp(x)
        adj = (bond > 0).unsqueeze(-1).to(h.dtype)                      # [B,N,N,1]
        for emb, norm, mlp in zip(self.bond, self.norm, self.mlp):
            msg = (adj * F.gelu(h.unsqueeze(1) + emb(bond))).sum(2)      # sum_j over neighbours of i
            h = h + mlp(norm(h + msg))
            h = h * mask.unsqueeze(-1)
        return h


# ----------------------------------------------------------------------------- latent pocket (seq mode)
class LatentPocket(nn.Module):
    def __init__(self, d, n_slots, n_heads, n_layers=2, dropout=0.0):
        super().__init__()
        self.slots = nn.Parameter(torch.randn(n_slots, d) * 0.02)
        self.ctx = nn.Linear(d, d)
        self.lq = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
        self.lkv = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
        self.att = nn.ModuleList([nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
                                  for _ in range(n_layers)])
        self.ff = nn.ModuleList([Transition(d, 2, dropout) for _ in range(n_layers)])

    def forward(self, prot, prot_mask, lig_pool):
        q = self.slots.unsqueeze(0) + self.ctx(lig_pool).unsqueeze(1)
        attn = None
        for lq, lkv, att, ff in zip(self.lq, self.lkv, self.att, self.ff):
            kv = lkv(prot)
            o, attn = att(lq(q), kv, kv, key_padding_mask=~prot_mask, need_weights=True)
            q = q + o
            q = q + ff(q)
        return q, attn                                                   # attn: [B, M, L]


# ----------------------------------------------------------------------------- pair stack
class AxialAttention(nn.Module):
    """Gated self-attention along the last-but-one axis of x [B, R, S, c] (each row attends over S)."""

    def __init__(self, c, n_heads, n_bias=0, dropout=0.0):
        super().__init__()
        self.h, self.dh = n_heads, c // n_heads
        self.ln = nn.LayerNorm(c)
        self.qkv = nn.Linear(c, 3 * c, bias=False)
        self.gate = nn.Linear(c, c)
        self.out = nn.Linear(c, c)
        self.bias = nn.Embedding(n_bias, n_heads) if n_bias else None
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x, key_mask, bias_idx=None):
        B, R, S, c = x.shape
        xn = self.ln(x)
        q, k, v = self.qkv(xn).view(B, R, S, 3, self.h, self.dh).unbind(3)
        logits = torch.einsum("brshd,brthd->brhst", q, k).float() / math.sqrt(self.dh)
        if self.bias is not None and bias_idx is not None:
            logits = logits + self.bias(bias_idx).permute(0, 3, 1, 2).unsqueeze(1).float()   # [B,1,H,S,S]
        logits = logits.masked_fill(~key_mask[:, None, None, None, :], NEG)
        a = logits.softmax(-1).to(v.dtype)
        o = torch.einsum("brhst,brthd->brshd", a, v).reshape(B, R, S, c)
        o = torch.sigmoid(self.gate(xn)) * o
        return self.drop(self.out(o))


class LigandTriangleUpdate(nn.Module):
    """z[i,j] += g * W( sum_l a[i,l] * e[l,j] ) with e built from ligand bond type + topological distance."""

    def __init__(self, c, dropout=0.0):
        super().__init__()
        self.ln = nn.LayerNorm(c)
        self.a = nn.Linear(c, c)
        self.ag = nn.Linear(c, c)
        self.g = nn.Linear(c, c)
        self.lo = nn.LayerNorm(c)
        self.out = nn.Linear(c, c)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, z, edge, lig_mask):
        zn = self.ln(z)
        a = self.a(zn) * torch.sigmoid(self.ag(zn)) * lig_mask[:, None, :, None]
        n = lig_mask.sum(1).clamp(min=1).to(z.dtype)[:, None, None, None]
        upd = torch.einsum("bilc,bljc->bijc", a, edge) / n
        return self.drop(torch.sigmoid(self.g(zn)) * self.out(self.lo(upd)))


class PairBlock(nn.Module):
    def __init__(self, d, c, n_heads, dropout=0.0):
        super().__init__()
        self.op_p, self.op_l = nn.Linear(d, c), nn.Linear(d, c)
        self.ln_p, self.ln_l = nn.LayerNorm(d), nn.LayerNorm(d)
        self.att_prot = AxialAttention(c, n_heads, dropout=dropout)
        self.att_lig = AxialAttention(c, n_heads, n_bias=N_TOPO, dropout=dropout)
        self.tri = LigandTriangleUpdate(c, dropout)
        self.trans_z = Transition(c, 2, dropout)
        self.zp, self.zl = nn.Linear(c, d), nn.Linear(c, d)
        self.lnz = nn.LayerNorm(c)
        self.trans_p, self.trans_l = Transition(d, 4, dropout), Transition(d, 4, dropout)

    def forward(self, z, s_p, s_l, pmask, lmask, topo, edge):
        pair_mask = (pmask[:, :, None] & lmask[:, None, :]).unsqueeze(-1).to(z.dtype)
        z = z + self.op_p(self.ln_p(s_p))[:, :, None] + self.op_l(self.ln_l(s_l))[:, None, :]
        z = z + self.att_prot(z.transpose(1, 2), pmask).transpose(1, 2)   # each ligand atom attends over residues
        z = z + self.att_lig(z, lmask, topo)                              # each residue attends over atoms
        z = z + self.tri(z, edge, lmask)
        z = (z + self.trans_z(z)) * pair_mask
        zn = self.lnz(z)
        s_p = s_p + self.zp(masked_mean(zn, lmask[:, None, :].expand(-1, z.shape[1], -1), 2))
        s_l = s_l + self.zl(masked_mean(zn.transpose(1, 2), pmask[:, None, :].expand(-1, z.shape[2], -1), 2))
        s_p = s_p + self.trans_p(s_p)
        s_l = s_l + self.trans_l(s_l)
        return z, s_p, s_l


def rbf(d, n=16, dmax=20.0):
    centers = torch.linspace(0.0, dmax, n, device=d.device, dtype=d.dtype)
    width = dmax / n
    return torch.exp(-((d.unsqueeze(-1) - centers) / width) ** 2)


# ----------------------------------------------------------------------------- full model
class EnsAff(nn.Module):
    def __init__(self, esm_dim, mode="seq", d=128, c=64, n_blocks=4, n_heads=4, n_slots=48, n_rbf=16,
                 n_recycle=3, dropout=0.1, tau_init=0.43):
        super().__init__()
        assert mode in ("seq", "struct")
        self.mode, self.n_recycle, self.n_rbf = mode, n_recycle, n_rbf
        self.prot_in = nn.Sequential(nn.LayerNorm(esm_dim), nn.Linear(esm_dim, d))
        self.lig_enc = LigandEncoder(d, 3, dropout)
        if mode == "seq":
            self.pocket = LatentPocket(d, n_slots, n_heads, 2, dropout)
        else:
            self.rbf_proj = nn.Linear(n_rbf, c)
        self.zp0, self.zl0 = nn.Linear(d, c), nn.Linear(d, c)
        self.edge_bond, self.edge_topo = nn.Embedding(N_BOND, c), nn.Embedding(N_TOPO, c)
        self.blocks = nn.ModuleList([PairBlock(d, c, n_heads, dropout) for _ in range(n_blocks)])
        self.rec_z, self.rec_p, self.rec_l = (nn.Sequential(nn.LayerNorm(c), nn.Linear(c, c)),
                                              nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d)),
                                              nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d)))
        for m in (self.rec_z, self.rec_p, self.rec_l):
            nn.init.zeros_(m[1].weight)
            nn.init.zeros_(m[1].bias)
        self.pool = nn.Linear(c, 1)
        self.head = nn.Sequential(nn.LayerNorm(c + 2 * d), nn.Linear(c + 2 * d, d), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(d, 2))
        self.tau_raw = nn.Parameter(torch.tensor(math.log(math.expm1(tau_init - 0.05))))
        self.out_shift = nn.Parameter(torch.tensor(6.4))  # mean pK of PDBbind; eases early training

    @property
    def tau(self):
        return F.softplus(self.tau_raw) + 0.05

    def _trunk(self, z, s_p, s_l, pm, lm, topo, edge):
        for blk in self.blocks:
            z, s_p, s_l = blk(z, s_p, s_l, pm, lm, topo, edge)
        return z, s_p, s_l

    def forward(self, batch, n_cycles=None):
        lm = batch["lig_mask"]
        s_l0 = self.lig_enc(batch["lig_x"], batch["bond"], lm)
        prot = self.prot_in(batch["prot"])
        pocket_attn = None
        if self.mode == "seq":
            s_p0, pocket_attn = self.pocket(prot, batch["prot_mask"], masked_mean(s_l0, lm, 1))
            pm = torch.ones(s_p0.shape[:2], dtype=torch.bool, device=s_p0.device)
        else:
            s_p0, pm = prot, batch["prot_mask"]
        B, M, N = s_p0.shape[0], s_p0.shape[1], s_l0.shape[1]
        z0 = self.zp0(s_p0)[:, :, None] + self.zl0(s_l0)[:, None, :]           # [B,M,N,c]
        if self.mode == "struct":
            dist, smask = batch["dist"], batch["state_mask"]
            K = dist.shape[1]
            z0 = z0[:, None] + self.rbf_proj(rbf(dist, self.n_rbf))             # [B,K,M,N,c]
            z0 = z0.flatten(0, 1)
        else:
            K, smask = 1, torch.ones(B, 1, dtype=torch.bool, device=z0.device)
        rep = lambda t: t.repeat_interleave(K, 0) if K > 1 else t                # noqa: E731
        s_p0, s_l0, pm_k, lm_k = rep(s_p0), rep(s_l0), rep(pm), rep(lm)
        topo, edge = rep(batch["topo"]), rep(self.edge_bond(batch["bond"]) + self.edge_topo(batch["topo"]))

        n = n_cycles or (torch.randint(1, self.n_recycle + 1, (1,)).item() if self.training else self.n_recycle)
        z, s_p, s_l = torch.zeros_like(z0), torch.zeros_like(s_p0), torch.zeros_like(s_l0)
        for cyc in range(n):
            last = cyc == n - 1
            with torch.set_grad_enabled(torch.is_grad_enabled() and last):
                z, s_p, s_l = self._trunk(z0 + self.rec_z(z), s_p0 + self.rec_p(s_p), s_l0 + self.rec_l(s_l),
                                          pm_k, lm_k, topo, edge)
            if not last:
                z, s_p, s_l = z.detach(), s_p.detach(), s_l.detach()

        # per-state affinity
        pair_mask = pm_k[:, :, None] & lm_k[:, None, :]
        w = self.pool(z).squeeze(-1).float().masked_fill(~pair_mask, NEG).flatten(1).softmax(-1)
        zpool = (w.unsqueeze(-1).to(z.dtype) * z.flatten(1, 2)).sum(1)
        h = torch.cat([zpool, masked_mean(s_p, pm_k, 1), masked_mean(s_l, lm_k, 1)], -1)
        o = self.head(h).float()
        mu_k = (o[:, 0] + self.out_shift).view(B, K)
        var_k = (F.softplus(o[:, 1]) + 1e-3).view(B, K)

        # ensemble aggregation (soft-max over states, free-energy style)
        tau = self.tau
        logits = (mu_k / tau).masked_fill(~smask, NEG)
        kcount = smask.sum(1).clamp(min=1).float()
        mu = tau * torch.logsumexp(logits, 1) - tau * torch.log(kcount)
        p = logits.softmax(1)
        mbar = (p * mu_k).sum(1, keepdim=True)
        var = (p * (var_k + (mu_k - mbar) ** 2)).sum(1)
        return {"mu": mu, "var": var, "state_mu": mu_k, "state_w": p, "pocket_attn": pocket_attn}



def pose_hinge(state_mu, smask, n_real, margin: float = 0.5):
    """The crystal pose must outscore its decoys by `margin`.

    Without a term like this nothing in the objective ever makes the per-state predictions differ. v12
    measured the consequence directly: K = 1, 3, 5, 9 states gave identical test RMSE to three decimals,
    and a model trained with every distance set to a constant matched the full model exactly. The state
    axis and the distance channel were both present and both carried nothing.

    This gives the distance channel a job the model cannot do without it. Decoys are generated by
    rotating the ligand about its own centroid, so the coarse distance statistics are unchanged and only
    atom-level geometry separates a decoy from the real pose.
    """
    B, K = state_mu.shape
    idx = torch.arange(K, device=state_mu.device)[None, :]
    real = (idx < n_real[:, None]) & smask
    decoy = (idx >= n_real[:, None]) & smask
    if not bool(decoy.any()):
        return state_mu.sum() * 0.0
    best_real = state_mu.masked_fill(~real, NEG).max(1).values
    viol = (margin - (best_real[:, None] - state_mu)).clamp(min=0) * decoy.float()
    return (viol.sum(1) / decoy.float().sum(1).clamp(min=1)).mean()


def pose_accuracy(state_mu, smask, n_real):
    """Fraction of complexes where every decoy scores below every crystal pose. 0.5-ish means the model
    cannot tell a rotated ligand from the real one; near 1.0 means the distance channel is being read."""
    B, K = state_mu.shape
    idx = torch.arange(K, device=state_mu.device)[None, :]
    real = (idx < n_real[:, None]) & smask
    decoy = (idx >= n_real[:, None]) & smask
    has = decoy.any(1)
    if not bool(has.any()):
        return float("nan")
    worst_real = state_mu.masked_fill(~real, 1e9).min(1).values
    best_decoy = state_mu.masked_fill(~decoy, NEG).max(1).values
    return float(((worst_real > best_decoy) & has).sum().item() / max(1, int(has.sum().item())))

def beta_nll(mu, var, y, beta=0.5):
    """beta-NLL (Seitzer et al., 2022): Gaussian NLL weighted by var^beta to avoid variance collapse."""
    nll = 0.5 * (torch.log(var) + (y - mu) ** 2 / var)
    return (nll * var.detach() ** beta).mean()


In [ ]:
%%writefile ensaff/evaluate.py
"""Metrics, split-conformal calibration and uncertainty diagnostics."""
from __future__ import annotations

import math

import numpy as np
from scipy import stats


def regression_metrics(y, mu):
    y, mu = np.asarray(y, float), np.asarray(mu, float)
    err = mu - y
    return {"n": int(len(y)), "rmse": float(np.sqrt((err ** 2).mean())), "mae": float(np.abs(err).mean()),
            "pearson": float(stats.pearsonr(y, mu)[0]) if len(y) > 2 else float("nan"),
            "spearman": float(stats.spearmanr(y, mu)[0]) if len(y) > 2 else float("nan")}


def conformal_quantile(y_cal, mu_cal, sd_cal, alpha=0.1):
    """Split-conformal with normalised residuals |y-mu|/sd -> interval mu +- q*sd has >= 1-alpha coverage.

    The guarantee assumes the calibration residuals are exchangeable with test AND independent of model
    selection. Using the early-stopping split for calibration breaks that independence and the intervals
    under-cover, so train.py holds out a separate calibration half.
    """
    s = np.abs(np.asarray(y_cal) - np.asarray(mu_cal)) / np.asarray(sd_cal)
    n = len(s)
    level = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(s, level, method="higher"))


def uncertainty_metrics(y, mu, sd, q=None, alpha=0.1):
    y, mu, sd = map(lambda a: np.asarray(a, float), (y, mu, sd))
    err = np.abs(mu - y)
    out = {"nll": float(np.mean(0.5 * np.log(2 * np.pi * sd ** 2) + (y - mu) ** 2 / (2 * sd ** 2))),
           "spearman_sd_vs_err": float(stats.spearmanr(sd, err)[0]) if len(y) > 2 else float("nan")}
    order = np.argsort(sd)
    for frac in (1.0, 0.75, 0.5, 0.25):
        k = max(1, int(round(frac * len(y))))
        idx = order[:k]
        out[f"rmse_top{int(frac * 100)}%conf"] = float(np.sqrt(((mu[idx] - y[idx]) ** 2).mean()))
    if q is not None and np.isfinite(q):
        lo, hi = mu - q * sd, mu + q * sd
        out[f"coverage@{int((1 - alpha) * 100)}"] = float(((y >= lo) & (y <= hi)).mean())
        out["mean_interval_width"] = float((hi - lo).mean())
    return out


In [ ]:
%%writefile ensaff/report.py
"""The consolidated report block printed at the end of the Colab pipeline.

Kept as a file here so it can be unit-tested in the sandbox against real metrics.json
output before being embedded into the notebook.
"""
import glob
import json
import os
import platform
from datetime import datetime


def _fmt(v, n=3):
    return "-" if v is None else (f"{v:.{n}f}" if isinstance(v, float) else str(v))


def build_report(build_id: str) -> str:
    L = []
    A = L.append
    A("=" * 68)
    A(f"ENSAFF RAPOR | BUILD {build_id} | {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    A("=" * 68)

    # --- ortam
    try:
        import torch
        gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        A(f"ORTAM: {gpu} | torch {torch.__version__} | python {platform.python_version()}")
    except Exception as e:
        A(f"ORTAM: okunamadi ({e})")
    try:
        import subprocess
        ram = subprocess.run(["free", "-g"], capture_output=True, text=True).stdout.splitlines()[1].split()
        A(f"RAM: toplam {ram[1]} GB")
    except Exception:
        pass
    A("")

    files = sorted(glob.glob("runs/*/metrics.json"))
    if not files:
        A("!! runs/*/metrics.json bulunamadi - pipeline calismadi.")
        return "\n".join(L)

    # --- her kosunun config + metrikleri
    for f in files:
        run = f.split("/")[1]
        m = json.load(open(f))
        A("-" * 68)
        A(f"[{run}]")
        cfg = m.get("config")
        A(f"  config: {json.dumps(cfg, ensure_ascii=False)}" if cfg else "  config: YOK (eski kod calismis olabilir)")
        if "delta" in str(m):             # eslestirilmis karsilastirma raporu
            cfgc = m.get("config", {})
            A(f"  (fark = B - A, ortak alt kumede; A={cfgc.get('a')} B={cfgc.get('b')})")
            for k, v in m.items():
                if not isinstance(v, dict) or "delta" not in v:
                    continue
                A(f"  {k:<12} ortak n={v['n_common']} (A:{v['n_a']} B:{v['n_b']})")
                for mk in ("rmse", "pearson", "spearman"):
                    d = v["delta"][mk]
                    A(f"      {mk:<9} A={v['A'][mk]:+.3f} B={v['B'][mk]:+.3f} "
                      f"fark={d['diff']:+.3f} [{d['ci95'][0]:+.3f}, {d['ci95'][1]:+.3f}] "
                      f"{'ANLAMLI' if d['significant'] else 'fark yok'}")
            continue
        if "rho_model_sd" in str(m):      # applicability raporu: belirsizlik sinyallerinin karsilastirmasi
            A("  (belirsizlik sinyali ~ |hata| Spearman; buyuk ve pozitif = iyi)")
            for k, v in m.items():
                if not isinstance(v, dict) or "rho_model_sd" not in v:
                    continue
                A(f"  {k:<12} n={v['n']:<5} model_sd={v['rho_model_sd']:+.3f}  "
                  f"ligand_uzaklik={v['rho_lig_dist']:+.3f}  protein_uzaklik={v['rho_prot_dist']:+.3f}  "
                  f"ikisi={v['rho_both_dist']:+.3f}")
                if v.get("rmse_nearest_quartile") is not None:
                    A(f"               RMSE en yakin %25={v['rmse_nearest_quartile']:.3f} "
                      f"| en uzak %25={v['rmse_furthest_quartile']:.3f}"
                      + (f" | kapsama(mesafe)={v['coverage_distance']:.3f}" if "coverage_distance" in v else ""))
            continue
        if m.get("demo"):
            A(f"  NOT: {m.get('note', 'demo')}")
        if "conformal_q" in m:
            A(f"  conformal_q: {_fmt(m['conformal_q'])}")
        for key, sub in m.items():
            if not isinstance(sub, dict) or key == "config":
                continue
            inner = [sub] if "rmse" in sub else [v for v in sub.values() if isinstance(v, dict) and "rmse" in v]
            names = [key] if "rmse" in sub else [f"{key}:{k}" for k, v in sub.items() if isinstance(v, dict) and "rmse" in v]
            for nm, r in zip(names, inner):
                A(f"  {nm:<22} n={r.get('n','-'):<6} rmse={_fmt(r.get('rmse'))} "
                  f"r={_fmt(r.get('pearson'))} rho={_fmt(r.get('spearman'))} "
                  f"cov90={_fmt(r.get('coverage@90'))} w={_fmt(r.get('mean_interval_width'))} "
                  f"sd~err={_fmt(r.get('spearman_sd_vs_err'))}")

    # --- cok tohumlu kosularin ortalama +- std ozeti (seq_s0, seq_s1, ... / baseline_s0, ...)
    import re
    from collections import defaultdict

    def _splits_of(m):
        """metrics.json -> {etiket: metrik_dict}; hem duz hem (rf/xgb) ic ice yapiyi destekler."""
        out = {}
        for k, v in m.items():
            if not isinstance(v, dict) or k == "config":
                continue
            if "rmse" in v:
                out[k] = v
            else:
                for k2, v2 in v.items():
                    if isinstance(v2, dict) and "rmse" in v2:
                        out[f"{k}:{k2}"] = v2
        return out

    groups = defaultdict(list)
    for f in files:
        run = f.split("/")[1]
        base = re.sub(r"_s\d+$", "", run)
        if base != run:
            groups[base].append(f)
    for base, fs in sorted(groups.items()):
        if len(fs) < 2:
            continue
        A("-" * 68)
        A(f"[{base}] {len(fs)} tohum - ortalama +- std")
        per_label = defaultdict(lambda: defaultdict(list))
        for f in fs:
            for label, r in _splits_of(json.load(open(f))).items():
                for k in ("rmse", "pearson", "spearman", "coverage@90", "spearman_sd_vs_err"):
                    if r.get(k) is not None:
                        per_label[label][k].append(r[k])
        for label in sorted(per_label, key=lambda x: ("BDB" in x, "test" not in x, x)):
            parts = []
            for k in ("rmse", "pearson", "spearman", "coverage@90", "spearman_sd_vs_err"):
                vals = per_label[label][k]
                if vals:
                    mu = sum(vals) / len(vals)
                    sd = (sum((x - mu) ** 2 for x in vals) / max(1, len(vals) - 1)) ** 0.5
                    parts.append(f"{k}={mu:.3f}+-{sd:.3f}")
            A(f"  {label:<16} " + "  ".join(parts))

    # --- geometri sondasi: metinler oldugu gibi aktarilir
    for pr in sorted(glob.glob("runs/*/probe_report.txt")):
        A("-" * 68)
        A(f"[{pr.split('/')[1]} geometri sondasi]")
        try:
            A("\n".join("  " + ln for ln in open(pr).read().splitlines()))
        except Exception as e:
            A(f"  okunamadi: {e}")

    # --- yorumlamayi gecersiz kilan durumlar; sonuclarin yaninda degil, tek yerde toplanir
    warn = []
    for f in files:
        run, m = f.split("/")[1], json.load(open(f))
        if m.get("conformal_degenerate"):
            warn.append(f"{run}: konformal q={_fmt(m.get('conformal_q'), 2)} dejenere -> cov90 ve "
                        f"aralik genisligi yorumlanamaz")
        for sp, c in (m.get("bfactor_degenerate") or {}).items():
            if c:
                warn.append(f"{run}/{sp}: {c} yapida B-faktor sutunu SABIT -> bu sette B-faktor "
                            f"karsilastirmasi gecersiz")
        for label, r in _splits_of(m).items():
            n = r.get("n")
            if isinstance(n, int) and n < 30:
                warn.append(f"{run}/{label}: n={n} -> iddia tasiyamaz, yalnizca betimleyici")
    if os.path.exists("runs/_notes.txt"):
        warn += [ln.strip() for ln in open("runs/_notes.txt").read().splitlines() if ln.strip()]
    if warn:
        A("-" * 68)
        A("!! YORUMLAMA UYARILARI")
        for w in sorted(set(warn)):
            A(f"  - {w}")

    # --- egitim egrisi ozeti
    A("-" * 68)
    for h in sorted(glob.glob("runs/*/history.csv")):
        try:
            import pandas as pd
            d = pd.read_csv(h)
            best = int(d["val_rmse"].idxmin())
            A(f"[{h.split('/')[1]} egitim] epoch={len(d)} en_iyi_epoch={best} "
              f"en_iyi_val_rmse={d['val_rmse'].min():.3f} son_val_rmse={d['val_rmse'].iloc[-1]:.3f} "
              f"tau={d['tau'].iloc[-1]:.3f} sn/epoch={d['sec'].mean():.0f}")
            A("  val_rmse: " + " ".join(f"{v:.3f}" for v in d["val_rmse"]))
            if "lr" in d.columns:
                A("  lr:       " + " ".join(f"{v:.1e}" for v in d["lr"]))
        except Exception as e:
            A(f"[{h}] okunamadi: {e}")
    A("=" * 68)
    return "\n".join(L)


def emit(build_id: str, work: str = None):
    """Print the report between copy markers, save it, and try to download it."""
    rep = build_report(build_id)
    print("\n\n>>>>>>>>>> BURADAN KOPYALA >>>>>>>>>>")
    print(rep)
    print("<<<<<<<<<< BURAYA KADAR <<<<<<<<<<\n")
    os.makedirs("results", exist_ok=True)
    safe = "".join(c if c.isalnum() or c in "-_." else "-" for c in str(build_id))
    path = f"results/EnsAff_rapor_{safe}_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
    with open(path, "w") as fh:
        fh.write(rep)
    print(f"[kaydedildi] {os.path.join(work or os.getcwd(), path)}")
    return path


In [ ]:
%%writefile ensaff/contactcount.py
r"""Contact-count baseline: the POSITIVE CONTROL for the perturbation probes.

The probes in `geom_probe` report that damaging the residue-atom distance channel barely moves the
pair-representation model's predictions. On its own that is ambiguous: it could mean the model ignores the
channel, or it could mean the probes are too weak to move any model. This module settles it.

The features here are counts of (pocket residue class, ligand atom class) pairs falling in each distance
shell, in the spirit of RF-Score (Ballester & Mitchell, 2010). Two properties make it the control this study
needs:

  * It is built from EXACTLY the same inputs the neural model receives (the same pocket residues, the same
    residue-atom distance matrix, the same ligand). Any difference in behaviour is the model, not the input.
  * Distance is the whole feature set. Collapse every distance to a constant and the shells merge into one,
    leaving only composition. A model that uses geometry MUST lose accuracy under that damage.

Two of the probes damage this model in different ways, and both are reported. `permute_res` keeps the
multiset of distances and only breaks which residue each distance belongs to, so the number of counted
contacts is unchanged and the damage is matched exactly to what the same probe does to the neural model.
`const:8` is stronger: with no distances there is no distance cutoff either, so every residue-atom pair in
the pocket is counted once and the shell axis disappears. That is the honest meaning of removing geometry
(a cutoff is itself geometric information), but it changes the contact total as well as its resolution, so
`permute_res` is the cleaner like-for-like comparison and `const:8` the upper bound on the damage.

So if `const:8` destroys this model while leaving the neural model untouched, the information is present in
the channel, the probe can detect its removal, and the neural model is not using it. If it destroys both,
the probe is too blunt and the neural result means nothing.

    python -m ensaff.contactcount --probes none,translate:2.0,permute_res,const:8 --out runs/cc

CPU only; no GPU and no high RAM needed.
"""
from __future__ import annotations

import argparse
import json
import os

import numpy as np
import pandas as pd

from .data import StructAffinityDataset, load_external_sets, load_lp_pdbbind
from .evaluate import regression_metrics
from .features import ELEMENTS

# Residues grouped by the property that drives the contact, not by identity: 20 classes against 5 distance
# shells and 5 ligand classes would give 500 sparse features on 7393 training complexes.
RES_CLASS = {"A": 0, "V": 0, "L": 0, "I": 0, "M": 0, "P": 0, "G": 0,          # hydrophobic/small
             "F": 1, "W": 1, "Y": 1,                                          # aromatic
             "S": 2, "T": 2, "N": 2, "Q": 2, "C": 2,                          # polar
             "D": 3, "E": 3,                                                  # acidic
             "K": 4, "R": 4, "H": 4}                                          # basic
N_RES_CLASS = 5
LIG_CLASS = {"C": 0, "N": 1, "O": 2, "S": 3, "P": 3, "F": 4, "Cl": 4, "Br": 4, "I": 4, "B": 4}
N_LIG_CLASS = 5
SHELL_EDGES = (4.0, 6.0, 8.0, 10.0, 12.0)      # last edge doubles as the cutoff
N_SHELL = len(SHELL_EDGES)
N_FEAT = N_RES_CLASS * N_LIG_CLASS * N_SHELL


def _lig_classes(x: np.ndarray) -> np.ndarray:
    """Element class per ligand heavy atom, read back from the one-hot block of the atom features."""
    idx = x[:, :len(ELEMENTS) + 1].argmax(1)
    out = np.full(len(idx), N_LIG_CLASS - 1, dtype=np.int64)
    for i, e in enumerate(ELEMENTS):
        out[idx == i] = LIG_CLASS.get(e, N_LIG_CLASS - 1)
    return out


def featurize_item(item, res_letters) -> np.ndarray:
    """item: a StructAffinityDataset sample (its `dist` already carries any geom/probe damage)."""
    D = np.asarray(item["dist"])[0]                       # [residues, ligand atoms], first state
    rc = np.array([RES_CLASS.get(c, 0) for c in res_letters], dtype=np.int64)
    lc = _lig_classes(np.asarray(item["x"]))
    shell = np.searchsorted(np.asarray(SHELL_EDGES), D, side="left")   # 0..N_SHELL, N_SHELL = beyond cutoff
    feat = np.zeros((N_RES_CLASS, N_LIG_CLASS, N_SHELL), dtype=np.float32)
    keep = shell < N_SHELL
    if keep.any():
        ri, ai = np.nonzero(keep)
        np.add.at(feat, (rc[ri], lc[ai], shell[ri, ai]), 1.0)
    return feat.reshape(-1)


def featurize_dataset(ds: StructAffinityDataset):
    X, y, ids = [], [], []
    for i in range(len(ds)):
        rec = ds.recs[i]
        letters = [rec["chain_seqs"][ci][ri] for ci, ri in rec["pocket"]]
        X.append(featurize_item(ds[i], letters))
        y.append(rec["y"])
        ids.append(rec["id"])
    return np.stack(X) if X else np.zeros((0, N_FEAT), np.float32), np.asarray(y, float), ids


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--out", default="runs/cc")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--protein_pattern", default="pdbbind/{id}/{id}_protein.pdb")
    p.add_argument("--ligand_pattern", default="pdbbind/{id}/{id}_ligand.sdf")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--cutoff", type=float, default=10.0)
    p.add_argument("--max_res", type=int, default=64)
    p.add_argument("--max_atoms", type=int, default=72)
    p.add_argument("--max_poses", type=int, default=4)
    p.add_argument("--probes", default="none,translate:1.0,translate:4.0,rotate,permute_res,const:8")
    p.add_argument("--splits", default="test,BDB2020+")
    p.add_argument("--seeds", default="0,1,2,3,4")
    a = p.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)

    class _NoEmb:              # the contact-count model needs no protein embeddings
        dim = 1
        def embed_chains(self, seqs): pass
        def get(self, seq): return np.zeros((len(seq), 1), dtype=np.float32)

    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=a.clean_level)
    exts = load_external_sets(a.data_dir)
    mk = lambda df, n, pp, lp: StructAffinityDataset(  # noqa: E731
        df, _NoEmb(), pp, lp, a.cutoff, a.max_res, a.max_atoms, a.max_poses, 0, 0.5, n,
        eval_states=0, keep_coords=True)
    dsets = {"train": mk(tabs["train"], "cc-train", a.protein_pattern, a.ligand_pattern),
             "val": mk(tabs["val"], "cc-val", a.protein_pattern, a.ligand_pattern),
             "test": mk(tabs["test"], "cc-test", a.protein_pattern, a.ligand_pattern)}
    for name, (df, pp, lp) in exts.items():
        dsets[name] = mk(df, f"cc-{name}", pp, lp)

    Xtr, ytr, _ = featurize_dataset(dsets["train"])
    print(f"[cc] egitim ozellik matrisi {Xtr.shape}, ortalama temas {Xtr.sum(1).mean():.1f}")

    from sklearn.ensemble import RandomForestRegressor
    try:
        from xgboost import XGBRegressor
        have_xgb = True
    except Exception as e:
        print("[cc] xgboost yok, yalnizca random forest:", e)
        have_xgb = False

    want = [s.strip() for s in a.splits.split(",") if s.strip()]
    probes = [s.strip() for s in a.probes.split(",") if s.strip()]
    if probes[0] != "none":
        probes = ["none"] + [x for x in probes if x != "none"]
    seeds = [int(s) for s in a.seeds.split(",")]

    res = {"config": {"features": f"{N_RES_CLASS}x{N_LIG_CLASS}x{N_SHELL} temas sayimi = {N_FEAT}",
                      "shells": list(SHELL_EDGES), "n_train": int(len(ytr)), "seeds": seeds,
                      "probes": probes, "note": "pozitif kontrol: geometri butun ozellik kumesi"}}
    rows = []
    for sd in seeds:
        models = {"rf": RandomForestRegressor(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=sd)}
        if have_xgb:
            models["xgb"] = XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                                         colsample_bytree=0.8, n_jobs=-1, random_state=sd)
        for mn, m in models.items():
            m.fit(Xtr, ytr)
            for probe in probes:
                for split in want:
                    ds = dsets.get(split)
                    if ds is None or not len(ds):
                        continue
                    ds.probe = probe
                    Xs, ys, _ids = featurize_dataset(ds)
                    ds.probe = "none"
                    mt = regression_metrics(ys, m.predict(Xs))
                    res.setdefault(f"{mn}:{probe}", {})[split] = mt
                    rows.append((mn, probe, split, sd, mt))
                    print(f"  seed {sd} {mn:4s} {probe:14s} {split:10s} n={mt['n']:<5} "
                          f"rmse={mt['rmse']:.3f} r={mt['pearson']:.3f}", flush=True)

    lines = ["(POZITIF KONTROL: ozellikler tamamen mesafeden tureyen temas sayimlari)",
             "(const:8 ya da permute_res bu modeli bozmazsa sonda yontemi gecersizdir)",
             "(not: mesafe silinince mesafe esigi de uygulanamaz, bu yuzden const:8 altinda cepteki her",
             " kalinti-atom cifti bir kez sayilir; permute_res ise temas toplamini aynen korur ve sinir",
             " agina uygulanan hasarla birebir eslesir)", ""]
    for mn in sorted({r[0] for r in rows}):
        for split in want:
            got0 = [r[4]["rmse"] for r in rows if r[0] == mn and r[1] == "none" and r[2] == split]
            if not got0:
                continue
            base = float(np.mean(got0))
            lines.append(f"  {mn} / {split}  (bozulmamis rmse = {base:.3f} pK, {len(got0)} tohum)")
            for probe in probes:
                g = [r[4]["rmse"] for r in rows if r[0] == mn and r[1] == probe and r[2] == split]
                if not g:
                    continue
                m_, s_ = float(np.mean(g)), float(np.std(g, ddof=1)) if len(g) > 1 else 0.0
                lines.append(f"    {probe:16s} rmse={m_:7.3f}+-{s_:<5.3f}  bedel={m_ - base:+.3f} pK")
            lines.append("")
    txt = "\n".join(lines)
    print("\n" + txt)
    with open(os.path.join(a.out, "probe_report.txt"), "w") as fh:
        fh.write(txt)
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    pd.DataFrame([{"model": r[0], "probe": r[1], "split": r[2], "seed": r[3], **r[4]} for r in rows]) \
        .to_csv(os.path.join(a.out, "probe_rows.csv"), index=False)
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/train.py
"""Train / evaluate EnsAff.

Sequence mode (no structures needed):
    python -m ensaff.train --mode seq --esm esm --out runs/seq_esm35m
Structure mode (PDBbind-style folders, e.g. after registering at PDBbind):
    python -m ensaff.train --mode struct --esm esm \
        --protein_pattern "/content/pdbbind/{id}/{id}_protein.pdb" \
        --ligand_pattern  "/content/pdbbind/{id}/{id}_ligand.sdf" --out runs/struct
"""
from __future__ import annotations

import argparse
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from .data import (SeqAffinityDataset, StructAffinityDataset, collate, load_external_sets, load_lp_pdbbind)
from .embed import ProteinEmbedder
from .evaluate import conformal_quantile, regression_metrics, uncertainty_metrics
from .model import EnsAff, beta_nll, pose_accuracy, pose_hinge


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)


def to_device(batch, dev):
    return {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


@torch.no_grad()
def predict(model, loader, dev, amp=True):
    model.eval()
    ids, ys, mus, sds, attn = [], [], [], [], []
    for batch in loader:
        batch = to_device(batch, dev)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp and dev.type == "cuda"):
            out = model(batch)
        ids += batch["ids"]
        ys.append(batch["y"].cpu())
        mus.append(out["mu"].float().cpu())
        sds.append(out["var"].float().sqrt().cpu())
    return ids, torch.cat(ys).numpy(), torch.cat(mus).numpy(), torch.cat(sds).numpy()


def build_datasets(a, emb):
    """Returns (train, val, test, {name: external_dataset}).

    External sets probe two different kinds of generalisation: BDB2020+ holds unseen targets, while mpro and
    EGFR hold many ligands against a SINGLE target, which is the lead-optimisation question.
    """
    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=a.clean_level)
    exts = load_external_sets(a.data_dir)
    if a.limit:
        tabs = {k: v.sample(min(len(v), a.limit if k == "train" else max(50, a.limit // 4)), random_state=0)
                for k, v in tabs.items()}
    out = {}
    if a.mode == "seq":
        mk = lambda df, n: SeqAffinityDataset(df, emb, a.max_atoms, n)  # noqa: E731
        for name, (df, _pp, _lp) in exts.items():
            out[name] = mk(df, name)
    else:
        # a control arm has to damage train, val, test and the external sets identically, otherwise the
        # comparison measures a train/test mismatch instead of the contribution of the channel
        mk = lambda df, n: StructAffinityDataset(df, emb, a.protein_pattern, a.ligand_pattern, a.cutoff,  # noqa: E731
                                                 a.max_res, a.max_atoms, a.max_poses, a.jitter_states, 0.5, n,
                                                 eval_states=a.eval_states, use_bfactor=a.use_bfactor,
                                                 geom=a.geom, pocket_mode=a.pocket_mode,
                                                 decoy_spec=a.pose_decoy, n_true=a.n_true)
        for name, (df, pp, lp) in exts.items():
            out[name] = StructAffinityDataset(df, emb, pp, lp, a.cutoff, a.max_res, a.max_atoms,
                                              a.max_poses, 0, 0.5, name, eval_states=a.eval_states,
                                              use_bfactor=a.use_bfactor, geom=a.geom,
                                              pocket_mode=a.pocket_mode, decoy_spec=a.pose_decoy,
                                              n_true=a.n_true)
    return mk(tabs["train"], "train"), mk(tabs["val"], "val"), mk(tabs["test"], "test"), out



def _chance_level(smask, n_real):
    """Poz ayirt etme olcutunun null altindaki basari olasiligi, kompleks basina.

    Olcut "her kristal poz her celdiriciden yuksek puan alsin" seklindedir. K durum
    degisilebilir ise, n_real gercegin hepsinin ilk n_real sirayi kapma olasiligi
    n_real! * d! / K! = 1 / C(K, n_real) olur.

    Bu fonksiyon, daha once ekrana basilan 1/(1+jitter_states) ifadesinin yerine gecer.
    O ifade egitimdeki celdirici sayisini varsayiyordu; degerlendirme ise
    max(4, jitter_states) celdirici kullanir ve birden fazla kristal pozu olan
    kompleksleri hic hesaba katmiyordu. Ikisi de sans duzeyini OLDUGUNDAN YUKSEK
    gosteriyordu, yani sonucu oldugundan zayif gosteriyordu.
    """
    from math import comb
    K = smask.sum(1).detach().cpu().numpy()
    nr = n_real.detach().cpu().numpy()
    out = [1.0 / comb(int(k), int(r)) for k, r in zip(K, nr) if k > r >= 1]
    return float(np.mean(out)) if out else float("nan")

def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--mode", default="seq", choices=["seq", "struct"])
    p.add_argument("--esm", default="esm", choices=["esm", "onehot"])
    p.add_argument("--esm_model", default="facebook/esm2_t12_35M_UR50D")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--cache_dir", default="", help="where to keep the ESM embedding cache (default: data_dir). "
                   "On Colab use a LOCAL path like /content/emb_cache — writing the multi-GB cache to Drive is slow.")
    p.add_argument("--out", default="runs/exp")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--protein_pattern", default="pdbbind/{id}/{id}_protein.pdb")
    p.add_argument("--ligand_pattern", default="pdbbind/{id}/{id}_ligand.sdf")
    p.add_argument("--cutoff", type=float, default=10.0)
    p.add_argument("--max_res", type=int, default=64)
    p.add_argument("--max_atoms", type=int, default=72)
    p.add_argument("--max_poses", type=int, default=4)
    p.add_argument("--jitter_states", type=int, default=0, help="struct mode: extra jittered states in training")
    p.add_argument("--posedisc", type=float, default=0.0,
                   help="weight of the pose-discrimination hinge: the crystal pose must outscore its "
                        "decoys. Without it nothing in the objective makes the states differ (v12: K=1..9 "
                        "identical to 3 d.p.). Needs --jitter_states > 0.")
    p.add_argument("--pose_margin", type=float, default=0.5, help="required pK gap, crystal over decoy")
    p.add_argument("--pose_decoy", default="rotate",
                   help="how training decoys are made: rotate (about the ligand centroid, so the coarse "
                        "distance statistics are unchanged) | translate | translate:s")
    p.add_argument("--n_true", type=int, default=48,
                   help="pocket_mode=wide: how many genuine contact residues are hidden among the decoys")
    p.add_argument("--geom", default="real",
                   help="KONTROL: mesafe kanalina kasitli hasar. real | const:8 (geometri yok) | "
                        "permute_res (dogru mesafeler, yanlis kalintilar) | permute_atom | scale:f")
    p.add_argument("--pocket_mode", default="nearest", choices=["nearest", "random", "wide"],
                   help="KONTROL: random = cep secimi bozulur, geometri gercek kalir; "
                        "wide = gercek cep sahte kalintilar arasina saklanir (kisayol kapanir)")
    p.add_argument("--use_bfactor", action="store_true",
                   help="D1: append observed per-residue flexibility (B-factor z-scores) to the embedding")
    p.add_argument("--eval_states", type=int, default=0,
                   help="struct mode: extra states at EVALUATION too. K=1 makes the aggregation an identity, "
                        "so this is what actually exercises it at inference (ablation of the central idea)")
    p.add_argument("--d", type=int, default=128)
    p.add_argument("--c", type=int, default=64)
    p.add_argument("--blocks", type=int, default=4)
    p.add_argument("--heads", type=int, default=4)
    p.add_argument("--slots", type=int, default=48)
    p.add_argument("--recycle", type=int, default=3)
    p.add_argument("--dropout", type=float, default=0.15)
    p.add_argument("--epochs", type=int, default=60)
    p.add_argument("--bs", type=int, default=16)
    p.add_argument("--lr", type=float, default=1.5e-4)   # 3e-4 overfit by epoch 1; lower peak LR trains longer
    p.add_argument("--wd", type=float, default=5e-2)     # stronger weight decay
    p.add_argument("--beta", type=float, default=0.5)
    p.add_argument("--mse_w", type=float, default=0.5)
    p.add_argument("--patience", type=int, default=12)
    p.add_argument("--lr_patience", type=int, default=2, help="epochs without val improvement before LR is halved")
    p.add_argument("--alpha", type=float, default=0.1, help="conformal miscoverage (0.1 -> 90%% intervals)")
    p.add_argument("--limit", type=int, default=0, help="debug: subsample train to this many complexes")
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args(argv)

    set_seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cache = os.path.join(a.cache_dir or a.data_dir,
                         f"emb_{a.esm}_{a.esm_model.split('/')[-1] if a.esm == 'esm' else 'oh'}.pt")
    emb = ProteinEmbedder(a.esm, a.esm_model, cache_path=cache)
    tr, va, te, exts = build_datasets(a, emb)
    if a.mode == "struct":
        tr.train_mode = True

    kw = dict(collate_fn=collate, num_workers=a.workers, pin_memory=dev.type == "cuda")
    dl_tr = DataLoader(tr, batch_size=a.bs, shuffle=True, drop_last=True, **kw)
    dl = {n: DataLoader(ds, batch_size=a.bs, shuffle=False, **kw)
          for n, ds in ([("val", va), ("test", te)] + list(exts.items())) if len(ds)}
    print(f"[data] degerlendirme setleri: " + ", ".join(f"{n}({len(ds)})" for n, ds in dl.items()))

    in_dim = emb.dim + (2 if (a.use_bfactor and a.mode == 'struct') else 0)
    model = EnsAff(in_dim, a.mode, a.d, a.c, a.blocks, a.heads, a.slots, n_recycle=a.recycle,
                   dropout=a.dropout).to(dev)
    print(f"[model] {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters, device={dev}")
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    # Warmup on steps, then halve the LR whenever validation stops improving. A cosine schedule sized to
    # --epochs never anneals when early stopping fires first (the LR stayed near its peak and validation
    # oscillated), so decay is driven by validation instead of by a fixed horizon.
    warm = min(500, (a.epochs * max(1, len(dl_tr))) // 10)
    warmup = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, warm)))
    plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5,
                                                         patience=a.lr_patience, min_lr=a.lr * 0.01)
    step_count = 0

    best, bad, hist = float("inf"), 0, []
    for ep in range(a.epochs):
        model.train()
        t0, losses, pose_acc = time.time(), [], []
        for batch in dl_tr:
            batch = to_device(batch, dev)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                out = model(batch)
            y = batch["y"]
            loss = beta_nll(out["mu"], out["var"], y, a.beta) + a.mse_w * ((out["mu"] - y) ** 2).mean()
            if a.posedisc > 0 and "state_mask" in batch:
                loss = loss + a.posedisc * pose_hinge(out["state_mu"].float(), batch["state_mask"],
                                                      batch["n_real"], a.pose_margin)
                pose_acc.append(pose_accuracy(out["state_mu"].float(), batch["state_mask"], batch["n_real"]))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step_count += 1
            if step_count <= warm:
                warmup.step()
            losses.append(loss.item())
        _, yv, mv, sv = predict(model, dl["val"], dev)
        m = regression_metrics(yv, mv)
        cur_lr = opt.param_groups[0]["lr"]
        plateau.step(m["rmse"])
        hist.append({"epoch": ep, "loss": float(np.mean(losses)),
                     "pose_acc": float(np.nanmean(pose_acc)) if pose_acc else float("nan"),
                     **{f"val_{k}": v for k, v in m.items()},
                     "tau": model.tau.item(), "lr": cur_lr, "sec": time.time() - t0})
        print(f"ep {ep:3d} loss {np.mean(losses):.3f} val rmse {m['rmse']:.3f} r {m['pearson']:.3f} "
              f"lr {cur_lr:.2e} ({time.time() - t0:.0f}s)")
        if m["rmse"] < best - 1e-4:
            best, bad = m["rmse"], 0
            torch.save({"model": model.state_dict(), "args": vars(a), "esm_dim": emb.dim, "in_dim": in_dim}, f"{a.out}/best.pt")
        else:
            bad += 1
            if bad >= a.patience:
                print("[early stop]")
                break
    pd.DataFrame(hist).to_csv(f"{a.out}/history.csv", index=False)

    # ---- final evaluation with the best checkpoint + split-conformal calibration on val
    model.load_state_dict(torch.load(f"{a.out}/best.pt", weights_only=False)["model"])
    # Conformal calibration on a half of val that was NOT used for early stopping. Calibrating on the
    # selection set makes its residuals optimistically small, so q comes out too tight and the nominal
    # 90% intervals under-cover on test (seen directly: val coverage 0.901 vs test 0.872).
    ids_v, yv, mv, sv = predict(model, dl["val"], dev)
    rng = np.random.RandomState(a.seed)
    half = rng.rand(len(yv)) < 0.5
    if half.sum() < 50 or (~half).sum() < 50:
        half = np.ones(len(yv), dtype=bool)
        print("[conformal] val cok kucuk -> ayri kalibrasyon yarisi kullanilamadi")
    q = conformal_quantile(yv[half], mv[half], sv[half], a.alpha)
    pose_val = float("nan")
    if a.posedisc > 0 and a.mode == "struct":
        ds = dl["val"].dataset
        keep = ds.eval_states
        ds.eval_states = max(4, a.jitter_states)       # decoys only for this pass; metrics above used none
        accs, chances = [], []
        with torch.no_grad():
            for b in dl["val"]:
                b = to_device(b, dev)
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                    o = model(b)
                accs.append(pose_accuracy(o["state_mu"].float(), b["state_mask"], b["n_real"]))
                chances.append(_chance_level(b["state_mask"], b["n_real"]))
        ds.eval_states = keep
        pose_val = float(np.nanmean(accs))
        chance = float(np.nanmean(chances))
        print(f"[poz] val ayirt etme dogrulugu = {pose_val:.3f} "
              f"(sans duzeyi {chance:.3f}, 1.0 = tam ayirt etme)")
        print(f"[poz] sans duzeyi degerlendirmede FIILEN kullanilan durum sayisindan hesaplanir; "
              f"ulasilan deger sansin {pose_val / max(chance, 1e-9):.2f} kati")
    best_epoch = int(pd.read_csv(f"{a.out}/history.csv")["val_rmse"].idxmin()) if hist else -1
    results = {"config": {"mode": a.mode, "esm": a.esm, "esm_model": a.esm_model if a.esm == "esm" else "onehot",
                          "clean_level": a.clean_level, "epochs_run": len(hist), "best_epoch": best_epoch,
                          "eval_states": a.eval_states, "lr": a.lr, "lr_patience": a.lr_patience, "wd": a.wd, "dropout": a.dropout, "recycle": a.recycle, "bs": a.bs,
                          "esm_dim": emb.dim, "in_dim": in_dim, "use_bfactor": bool(a.use_bfactor), "geom": a.geom, "pocket_mode": a.pocket_mode, "posedisc": a.posedisc,
                          "pose_decoy": a.pose_decoy, "jitter_states": a.jitter_states,
                          "max_res": a.max_res, "n_true": a.n_true, "bs_": a.bs, "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
                          "seed": a.seed}, "conformal_q": q, "conformal_n_cal": int(half.sum()),
               "conformal_note": "calibrated on a val half disjoint from early stopping",
               "pose_val_accuracy": pose_val,
               "bfactor_degenerate": ({k: int(getattr(v.dataset, "n_degenerate_bfactor", 0)) for k, v in dl.items()}
                                      if a.use_bfactor else {})}
    for name, loader in dl.items():
        ids, y, mu, sd = predict(model, loader, dev)
        results[name] = {**regression_metrics(y, mu), **uncertainty_metrics(y, mu, sd, q, a.alpha)}
        pd.DataFrame({"id": ids, "y": y, "pred": mu, "sd": sd, "lo": mu - q * sd, "hi": mu + q * sd}) \
            .to_csv(f"{a.out}/pred_{name}.csv", index=False)
    with open(f"{a.out}/metrics.json", "w") as fh:
        json.dump(results, fh, indent=2)
    print(json.dumps(results, indent=2))
    return results


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/baseline.py
"""Baseline: ECFP4 + mean protein embedding -> RandomForest (+ XGBoost if installed). Same splits as EnsAff.

    python -m ensaff.baseline --esm esm --out runs/baseline
"""
from __future__ import annotations

import argparse
import json
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from .data import load_external_sets, load_lp_pdbbind, split_chains
from .embed import ProteinEmbedder
from .evaluate import regression_metrics
from .features import ecfp


def featurize(df, emb):
    chains = [split_chains(s) for s in df.seq]
    emb.embed_chains([c for cs in chains for c in cs])
    P = np.stack([np.concatenate([emb.get(c) for c in cs], 0).astype(np.float32).mean(0) for cs in chains])
    L = np.stack([ecfp(s) for s in df.smiles]).astype(np.float32)
    return np.concatenate([L, P], 1), df.y.values.astype(float)


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--esm", default="esm", choices=["esm", "onehot"])
    p.add_argument("--esm_model", default="facebook/esm2_t12_35M_UR50D")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--cache_dir", default="", help="where to keep the ESM embedding cache (default: data_dir)")
    p.add_argument("--out", default="runs/baseline")
    p.add_argument("--limit", type=int, default=0)
    p.add_argument("--seed", type=int, default=0, help="random_state for RF/XGB; vary it for error bars")
    a = p.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    cache = os.path.join(a.cache_dir or a.data_dir,
                         f"emb_{a.esm}_{a.esm_model.split('/')[-1] if a.esm == 'esm' else 'oh'}.pt")
    emb = ProteinEmbedder(a.esm, a.esm_model, cache_path=cache)
    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"))
    for _n, (_df, _pp, _lp) in load_external_sets(a.data_dir).items():
        tabs[_n] = _df
    if a.limit:
        tabs["train"] = tabs["train"].sample(min(a.limit, len(tabs["train"])), random_state=0)
    X, y = featurize(tabs["train"], emb)
    models = {"rf": RandomForestRegressor(n_estimators=500, max_features=0.3, n_jobs=-1, random_state=a.seed)}
    try:
        from xgboost import XGBRegressor
        models["xgb"] = XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=8, subsample=0.8,
                                     colsample_bytree=0.3, n_jobs=-1, random_state=a.seed)
    except ImportError:
        pass
    res = {"config": {"esm": a.esm, "esm_model": a.esm_model if a.esm == "esm" else "onehot",
                      "features": "ECFP4(2048) + mean ESM", "n_train": int(len(tabs["train"])), "seed": a.seed}}
    best_name = "xgb" if "xgb" in models else "rf"   # predictions saved for the stronger baseline
    feats = {}
    for split in [k for k in tabs if k != "train"]:
        if len(tabs[split]):
            feats[split] = featurize(tabs[split], emb)
    for name, m in models.items():
        m.fit(X, y)
        res[name] = {}
        for split, (Xs, ys) in feats.items():
            pred = m.predict(Xs)
            res[name][split] = regression_metrics(ys, pred)
            if name == best_name:
                # the stronger baseline's predictions are written out so ensaff.compare can run paired,
                # common-subset comparisons against it (sd is NaN: tree baselines give no uncertainty)
                pd.DataFrame({"id": tabs[split]["id"].values, "y": ys, "pred": pred, "sd": np.nan}) \
                    .to_csv(f"{a.out}/pred_{split}.csv", index=False)
    with open(f"{a.out}/metrics.json", "w") as fh:
        json.dump(res, fh, indent=2)
    print(json.dumps(res, indent=2))
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/ensemble.py
r"""Deep-ensemble aggregation over the per-seed sequence-mode runs.

Averaging several independently seeded models and using their disagreement as uncertainty is the standard
deep-ensemble recipe (Lakshminarayanan et al., 2017); it is used here as an established baseline for
uncertainty, not as a contribution of this work. For a Gaussian mixture with equal weights the predictive
mean and variance are

    mu  = (1/M) * sum_i mu_i
    var = (1/M) * sum_i sigma_i^2   +   (1/M) * sum_i (mu_i - mu)^2
          \_____ aleatoric _____/       \_____ ensemble spread _____/

The conformal quantile is refitted on the validation split so coverage is calibrated for the ensemble.

    python -m ensaff.ensemble --runs 'runs/seq_s*' --out runs/seq_ens
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd

from .evaluate import conformal_quantile, regression_metrics, uncertainty_metrics

BASE_SPLITS = ("val", "test")   # external sets are discovered from the files on disk
Q_DEGENERATE = 5.0   # above this the quantile, not the model, is setting the interval width


def _load(run_dirs, split):
    """Returns (ids, y, [mu per model], [sd per model]) aligned on id, or None if unavailable."""
    frames = []
    for d in run_dirs:
        p = os.path.join(d, f"pred_{split}.csv")
        if not os.path.exists(p):
            return None
        frames.append(pd.read_csv(p).set_index("id").sort_index())
    ids = frames[0].index
    for f in frames[1:]:
        if not f.index.equals(ids):          # align defensively; seeds must score the same complexes
            common = ids.intersection(f.index)
            if len(common) == 0:
                return None
            ids = common
    frames = [f.loc[ids] for f in frames]
    y = frames[0]["y"].to_numpy(float)
    mu = np.stack([f["pred"].to_numpy(float) for f in frames])
    sd = np.stack([f["sd"].to_numpy(float) for f in frames])
    return list(ids), y, mu, sd


def combine(mu, sd):
    """Mixture mean and variance. Tree baselines report no per-prediction sd (NaN), in which case the
    spread between members is the only uncertainty available - this keeps them ensemble-able too, so the
    comparison can be ensemble against ensemble rather than ensemble against a single model."""
    m = mu.mean(0)
    spread = ((mu - m) ** 2).mean(0)
    aleatoric = np.nan_to_num((sd ** 2), nan=0.0).mean(0)
    return m, np.sqrt(aleatoric + spread)


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--runs", default="runs/seq_s*", help="glob of per-seed run directories")
    p.add_argument("--out", default="runs/seq_ens")
    p.add_argument("--alpha", type=float, default=0.1)
    p.add_argument("--cal_seed", type=int, default=0, help="which half of val calibrates (fixed across runs)")
    a = p.parse_args(argv)
    run_dirs = sorted(d for d in glob.glob(a.runs) if os.path.isdir(d))
    if len(run_dirs) < 2:
        print(f"[ensemble] en az 2 kosu gerekli, bulunan: {run_dirs} -> atlaniyor")
        return None
    os.makedirs(a.out, exist_ok=True)
    print(f"[ensemble] {len(run_dirs)} kosu birlestiriliyor: {run_dirs}")

    cal = _load(run_dirs, "val")
    if cal is None:
        print("[ensemble] val tahminleri yok -> atlaniyor")
        return None
    _, yv, muv, sdv = cal
    mv, sv = combine(muv, sdv)
    # Every member was early-stopped on this same val split. Calibrating on all of it makes the residuals
    # dependent on model selection, so q comes out too tight and the nominal 90% intervals under-cover on
    # test (v11: val 0.902 against test 0.874). Calibrate on one fixed half instead, as train.py does.
    rng = np.random.RandomState(a.cal_seed)
    half = rng.rand(len(yv)) < 0.5
    if half.sum() < 50 or (~half).sum() < 50:
        half = np.ones(len(yv), dtype=bool)
        print("[ensemble] val cok kucuk -> ayri kalibrasyon yarisi kullanilamadi")
    q = conformal_quantile(yv[half], mv[half], sv[half], a.alpha) if np.all(sv[half] > 0) else float("nan")
    degenerate = bool(np.isfinite(q) and q > Q_DEGENERATE)
    if degenerate:
        print(f"  !! UYARI: konformal q={q:.2f} dejenere. Uyeler birbirine neredeyse hic karsi cikmiyor "
              f"(tahmin sd'si ~0), kapsama tamamen bu buyuk carpanla saglaniyor; bu toplulugun "
              f"belirsizlik sayilari (cov90, genislik) yorumlanamaz.")

    # whatever splits the member runs actually scored (val, test, BDB2020+, mpro, EGFR, ...)
    import re as _re
    splits = sorted({_re.sub(r"^pred_|\.csv$", "", os.path.basename(f))
                     for f in glob.glob(os.path.join(run_dirs[0], "pred_*.csv"))},
                    key=lambda x: (x not in BASE_SPLITS, x))
    res = {"config": {"members": len(run_dirs), "runs": run_dirs, "splits": splits,
                      "method": "deep ensemble (mean + mixture variance), conformal refit on val"},
           "conformal_q": q, "conformal_n_cal": int(half.sum()), "conformal_degenerate": degenerate,
           "conformal_note": "calibrated on a val half disjoint from early stopping"}
    for split in splits:
        got = _load(run_dirs, split)
        if got is None:
            continue
        ids, y, mu, sd = got
        m, s = combine(mu, sd)
        res[split] = {**regression_metrics(y, m), **uncertainty_metrics(y, m, s, q, a.alpha)}
        pd.DataFrame({"id": ids, "y": y, "pred": m, "sd": s, "lo": m - q * s, "hi": m + q * s}) \
            .to_csv(os.path.join(a.out, f"pred_{split}.csv"), index=False)
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    for k in splits:
        if k in res:
            r = res[k]
            print(f"  {k:<12} rmse={r['rmse']:.3f} r={r['pearson']:.3f} rho={r['spearman']:.3f} "
                  f"cov90={r.get(f'coverage@{int((1-a.alpha)*100)}'):.3f} sd~err={r['spearman_sd_vs_err']:.3f}")
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/applicability.py
r"""Applicability domain: how far is each evaluation complex from the training data, and does that predict
the error better than the model's own uncertainty?

Motivation. On the leak-proof test split the learned variance head gives a weak positive signal, but on the
external BDB2020+ set it goes NEGATIVE: the model is confident exactly where it is wrong. Volkov et al.
(2022) argued that such models largely memorise, so proximity to the training set should govern accuracy.
This module measures that directly with two cheap distances:

  * ligand  - 1 - max Tanimoto (ECFP4) to any training ligand
  * protein - 1 - max 3-mer Jaccard to any training sequence

Both are computed in bulk (RDKit BulkTanimoto; sparse matrix product for the k-mer Jaccard) so they scale to
the full test split. The module reports, per split, the rank correlation of each signal with |error|, and
recalibrates conformal intervals using whichever signal is chosen.

    python -m ensaff.applicability --runs 'runs/seq_s*' --out runs/applicability
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
from scipy import sparse, stats

from .data import load_external_sets, load_lp_pdbbind
from .evaluate import conformal_quantile, regression_metrics, uncertainty_metrics

K = 3


def _fps(smiles):
    from rdkit import Chem, RDLogger
    from rdkit.Chem import AllChem
    RDLogger.DisableLog("rdApp.*")
    out = []
    for s in smiles:
        m = Chem.MolFromSmiles(str(s))
        out.append(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m is not None else None)
    return out


def max_tanimoto(query_smiles, ref_fps):
    from rdkit import DataStructs
    out = np.full(len(query_smiles), np.nan)
    for i, f in enumerate(_fps(query_smiles)):
        if f is not None and ref_fps:
            out[i] = max(DataStructs.BulkTanimotoSimilarity(f, ref_fps))
    return out


def _kmer_matrix(seqs, vocab):
    rows, cols = [], []
    for i, s in enumerate(seqs):
        s = str(s).replace(":", "")
        seen = {s[j:j + K] for j in range(len(s) - K + 1)}
        for km in seen:
            c = vocab.get(km)
            if c is not None:
                rows.append(i)
                cols.append(c)
    m = sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(seqs), len(vocab)))
    return m


def max_kmer_jaccard(query_seqs, ref_seqs, block: int = 512):
    """Max 3-mer Jaccard of each query against any reference, via sparse intersection counts."""
    vocab = {}
    for s in list(ref_seqs) + list(query_seqs):
        s = str(s).replace(":", "")
        for j in range(len(s) - K + 1):
            vocab.setdefault(s[j:j + K], len(vocab))
    R = _kmer_matrix(ref_seqs, vocab)
    Q = _kmer_matrix(query_seqs, vocab)
    rn = np.asarray(R.sum(1)).ravel()
    qn = np.asarray(Q.sum(1)).ravel()
    out = np.zeros(Q.shape[0], dtype=np.float32)
    for i in range(0, Q.shape[0], block):
        q = Q[i:i + block]
        inter = np.asarray((q @ R.T).todense(), dtype=np.float32)
        union = qn[i:i + block, None] + rn[None, :] - inter
        out[i:i + block] = np.nanmax(np.where(union > 0, inter / np.maximum(union, 1e-9), 0.0), axis=1)
    return out


def signals_for(pred_df, meta_df, ref_fps, ref_seqs):
    """Returns the per-row distances for one evaluation split, aligned to pred_df['id']."""
    meta = meta_df.set_index("id")
    smi = meta["smiles"].reindex(pred_df["id"]).values
    seq = meta["seq"].reindex(pred_df["id"]).values
    lig = 1.0 - max_tanimoto(smi, ref_fps)
    prot = 1.0 - max_kmer_jaccard(seq, ref_seqs)
    return lig, prot


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--runs", default="runs/seq_ens", help="run dir (or glob) holding pred_*.csv")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--out", default="runs/applicability")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--alpha", type=float, default=0.1)
    a = p.parse_args(argv)
    run_dirs = sorted(d for d in glob.glob(a.runs) if os.path.isdir(d))
    if not run_dirs:
        print(f"[applicability] kosu bulunamadi: {a.runs}")
        return None
    run = run_dirs[0]
    os.makedirs(a.out, exist_ok=True)

    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=a.clean_level)
    ref_fps = [f for f in _fps(tabs["train"].smiles) if f is not None]
    ref_seqs = list(pd.unique(tabs["train"].seq))
    meta = {"val": tabs["val"], "test": tabs["test"]}
    for name, (df, _pp, _lp) in load_external_sets(a.data_dir).items():
        meta[name] = df
    print(f"[applicability] referans: {len(ref_fps)} ligand, {len(ref_seqs)} protein | kosu: {run}")

    res = {"config": {"run": run, "k": K, "ref_ligands": len(ref_fps), "ref_proteins": len(ref_seqs)}}
    cal = None
    for f in sorted(glob.glob(os.path.join(run, "pred_*.csv"))):
        split = os.path.basename(f)[len("pred_"):-len(".csv")]
        if split not in meta:
            continue
        p_df = pd.read_csv(f)
        lig, prot = signals_for(p_df, meta[split], ref_fps, ref_seqs)
        err = np.abs(p_df["pred"] - p_df["y"]).values
        sd = p_df["sd"].values
        ok = ~(np.isnan(lig) | np.isnan(prot))
        rho = lambda x: float(stats.spearmanr(x[ok], err[ok])[0])  # noqa: E731
        both = lig * prot
        entry = {"n": int(ok.sum()),
                 "mean_lig_dist": float(np.nanmean(lig)), "mean_prot_dist": float(np.nanmean(prot)),
                 "rho_model_sd": rho(sd), "rho_lig_dist": rho(lig), "rho_prot_dist": rho(prot),
                 "rho_both_dist": rho(both)}
        # error of the closest vs furthest quartile, by the combined distance
        q = np.nanquantile(both[ok], [0.25, 0.75])
        near, far = err[ok][both[ok] <= q[0]], err[ok][both[ok] >= q[1]]
        entry["rmse_nearest_quartile"] = float(np.sqrt((near ** 2).mean())) if len(near) else None
        entry["rmse_furthest_quartile"] = float(np.sqrt((far ** 2).mean())) if len(far) else None
        # conformal intervals driven by the distance signal instead of the model's sd
        score = 0.25 + both                      # strictly positive width multiplier
        if split == "val":
            cal = conformal_quantile(p_df["y"].values[ok], p_df["pred"].values[ok], score[ok], a.alpha)
            res["conformal_q_distance"] = cal
        if cal is not None:
            lo, hi = p_df["pred"].values - cal * score, p_df["pred"].values + cal * score
            y = p_df["y"].values
            entry["coverage_distance"] = float(((y >= lo) & (y <= hi))[ok].mean())
            entry["width_distance"] = float((hi - lo)[ok].mean())
        res[split] = entry
        print(f"  {split:<10} n={entry['n']:<5} sd:{entry['rho_model_sd']:+.3f}  "
              f"ligand:{entry['rho_lig_dist']:+.3f}  protein:{entry['rho_prot_dist']:+.3f}  "
              f"ikisi:{entry['rho_both_dist']:+.3f}")
        pd.DataFrame({"id": p_df["id"], "lig_dist": lig, "prot_dist": prot, "abs_err": err, "sd": sd}) \
            .to_csv(os.path.join(a.out, f"signals_{split}.csv"), index=False)
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/compare.py
r"""Paired comparison of two runs on the complexes they BOTH scored.

Structure mode drops complexes whose files are missing (n=4222 of 4250 in the first real run), so reading its
RMSE next to the sequence-mode or baseline RMSE compares different sets of molecules. This restricts every
comparison to the intersection of ids and reports a paired bootstrap interval for the difference, which also
accounts for the fact that the two models see the same test items.

    python -m ensaff.compare --a runs/struct_ens --b runs/seq_ens --out runs/compare
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import pandas as pd
from scipy import stats


def _metrics(y, p):
    return {"rmse": float(np.sqrt(((p - y) ** 2).mean())),
            "pearson": float(stats.pearsonr(y, p)[0]),
            "spearman": float(stats.spearmanr(y, p)[0])}


def paired_bootstrap(y, pa, pb, n_boot=2000, seed=0):
    """95% interval for (B - A) on each metric, resampling complexes (both models share the resample)."""
    rng = np.random.RandomState(seed)
    n = len(y)
    keys = ("rmse", "pearson", "spearman")
    diffs = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y[idx])) < 3:
            continue
        ma, mb = _metrics(y[idx], pa[idx]), _metrics(y[idx], pb[idx])
        for k in keys:
            diffs[k].append(mb[k] - ma[k])
    out = {}
    for k in keys:
        d = np.array(diffs[k])
        lo, hi = np.percentile(d, [2.5, 97.5])
        out[k] = {"diff": float(d.mean()), "ci95": [float(lo), float(hi)],
                  "significant": bool(lo > 0 or hi < 0)}
    return out


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--a", required=True, help="reference run dir (e.g. runs/seq_ens)")
    p.add_argument("--b", required=True, help="run dir to compare against it (e.g. runs/struct_ens)")
    p.add_argument("--out", default="runs/compare")
    p.add_argument("--n_boot", type=int, default=2000)
    a = p.parse_args(argv)
    os.makedirs(a.out, exist_ok=True)
    res = {"config": {"a": a.a, "b": a.b, "n_boot": a.n_boot}}
    print(f"[compare] A={a.a}  B={a.b}  (fark = B - A, ortak alt kumede)")
    for fa in sorted(glob.glob(os.path.join(a.a, "pred_*.csv"))):
        split = os.path.basename(fa)[len("pred_"):-len(".csv")]
        fb = os.path.join(a.b, f"pred_{split}.csv")
        if not os.path.exists(fb):
            continue
        da = pd.read_csv(fa).set_index("id").sort_index()
        db = pd.read_csv(fb).set_index("id").sort_index()
        common = da.index.intersection(db.index)
        if len(common) < 20:
            continue
        da, db = da.loc[common], db.loc[common]
        y = da["y"].to_numpy(float)
        pa, pb = da["pred"].to_numpy(float), db["pred"].to_numpy(float)
        entry = {"n_common": int(len(common)), "n_a": int(len(pd.read_csv(fa))), "n_b": int(len(pd.read_csv(fb))),
                 "A": _metrics(y, pa), "B": _metrics(y, pb), "delta": paired_bootstrap(y, pa, pb, a.n_boot)}
        res[split] = entry
        print(f"  {split:<10} ortak n={entry['n_common']} (A:{entry['n_a']} B:{entry['n_b']})")
        for k in ("rmse", "pearson", "spearman"):
            d = entry["delta"][k]
            mark = "ANLAMLI" if d["significant"] else "fark yok"
            print(f"      {k:<9} A={entry['A'][k]:+.3f}  B={entry['B'][k]:+.3f}  "
                  f"fark={d['diff']:+.3f} [{d['ci95'][0]:+.3f}, {d['ci95'][1]:+.3f}]  {mark}")
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/ablate_states.py
r"""Does aggregating over several bound states at INFERENCE change anything?

This is the direct ablation of the idea the project is named after. With a single crystal pose K=1 and the
aggregation mu = tau*logsumexp(mu_k/tau) - tau*log K reduces exactly to mu_1, so every result so far says
nothing about it: in the runs to date the extra states existed only during training, as augmentation.

Here a trained checkpoint is re-evaluated with K = 1, 3, 5, 9 states. Each extra state is a rigid-body
translation of the ligand with distances recomputed from the moved coordinates, seeded per complex from a
stable hash so the comparison repeats exactly. (An earlier version added noise directly to the distance
matrix, which no pose change can produce; its K-ablation therefore tested noise robustness, not state
aggregation.) If the aggregation carries real signal, accuracy should move with K.

    python -m ensaff.ablate_states --ckpt runs/struct_s0/best.pt --out runs/ablate_s0
"""
from __future__ import annotations

import argparse
import json
import os

import torch
from torch.utils.data import DataLoader

from .data import StructAffinityDataset, collate, load_external_sets, load_lp_pdbbind
from .embed import ProteinEmbedder
from .evaluate import regression_metrics, uncertainty_metrics
from .model import EnsAff
from .train import predict


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", required=True)
    p.add_argument("--out", default="runs/ablate_states")
    p.add_argument("--states", default="0,2,4,8", help="extra jittered states to add at evaluation")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--cache_dir", default="")
    p.add_argument("--splits", default="test,BDB2020+")
    a = p.parse_args(argv)
    ck = torch.load(a.ckpt, weights_only=False)
    args = ck["args"]
    if args.get("mode") != "struct":
        print("[ablate] yalnizca yapi modu icin anlamli (dizi modunda K hep 1)")
        return None
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cache = os.path.join(a.cache_dir or args.get("cache_dir") or a.data_dir,
                         f"emb_{args['esm']}_{args['esm_model'].split('/')[-1] if args['esm'] == 'esm' else 'oh'}.pt")
    emb = ProteinEmbedder(args["esm"], args["esm_model"], cache_path=cache)
    model = EnsAff(ck.get("in_dim", ck["esm_dim"]), "struct", args["d"], args["c"], args["blocks"], args["heads"], args["slots"],
                   n_recycle=args["recycle"], dropout=args["dropout"]).to(dev)
    model.load_state_dict(ck["model"])

    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=args["clean_level"])
    exts = load_external_sets(a.data_dir)
    want = [s.strip() for s in a.splits.split(",") if s.strip()]
    res = {"config": {"ckpt": a.ckpt, "states": a.states, "seed": args.get("seed")}}
    for n_extra in [int(x) for x in a.states.split(",")]:
        K = n_extra + 1
        res[f"K={K}"] = {}
        for split in want:
            if split == "test":
                df, pp, lp = tabs["test"], args["protein_pattern"], args["ligand_pattern"]
            elif split in exts:
                df, pp, lp = exts[split]
            else:
                continue
            ds = StructAffinityDataset(df, emb, pp, lp, args["cutoff"], args["max_res"], args["max_atoms"],
                                       args["max_poses"], 0, 0.5, f"{split}-K{K}", eval_states=n_extra,
                                       use_bfactor=args.get("use_bfactor", False))
            if not len(ds):
                continue
            dl = DataLoader(ds, batch_size=args["bs"], shuffle=False, collate_fn=collate)
            _ids, y, mu, sd = predict(model, dl, dev)
            res[f"K={K}"][split] = {**regression_metrics(y, mu), **uncertainty_metrics(y, mu, sd)}
            r = res[f"K={K}"][split]
            print(f"  K={K:<2} {split:<10} n={r['n']:<5} rmse={r['rmse']:.3f} r={r['pearson']:.3f} "
                  f"rho={r['spearman']:.3f} sd~err={r['spearman_sd_vs_err']:+.3f}", flush=True)
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/geom_probe.py
r"""Does the structure model actually read the pocket geometry?

Every dynamics result so far is null: the K-state aggregation is identical to three decimals for K=1,3,5,9,
and the B-factor flexibility features do not replicate over five seeds. Before blaming the dynamics idea it
is worth asking whether the model is sensitive to *static* geometry at all, because an 0.8 angstrom rigid
displacement of the ligand already changes the RBF distance encoding by about 65% of its own norm while the
prediction moves by ~0.001 pK. A model that ignores the geometry it is given cannot be helped by giving it
better geometry.

This re-evaluates trained checkpoints under controlled damage to the residue-atom distance channel and
reports, next to the usual accuracy, two numbers that answer the question directly:

    d|mu|   mean absolute change in the predicted pK against the unperturbed run
    r_base  Pearson correlation between perturbed and unperturbed predictions

r_base near 1.000 with d|mu| near 0 means the channel is being ignored. The probes run in increasing order
of damage, ending with ones that no real structure could produce (const, permute), so the ceiling of the
effect is bracketed rather than guessed.

    python -m ensaff.geom_probe --ckpts 'runs/struct_s*/best.pt' --out runs/probe_struct

GPU, high-RAM, inference only - no training.
"""
from __future__ import annotations

import argparse
import glob
import json
import os

import numpy as np
import torch
from scipy import stats
from torch.utils.data import DataLoader

from .data import StructAffinityDataset, collate, load_external_sets, load_lp_pdbbind
from .embed import ProteinEmbedder
from .evaluate import regression_metrics
from .model import EnsAff
from .train import predict

DEFAULT_PROBES = ("none,translate:0.5,translate:1.0,translate:2.0,translate:4.0,translate:8.0,"
                  "rotate,scale:1.5,permute_res,permute_atom,const:8")


def _ds_key(args):
    """Dataset-relevant settings. Checkpoints that agree on these can share one built dataset."""
    return (args["clean_level"], args["cutoff"], args["max_res"], args["max_atoms"], args["max_poses"],
            bool(args.get("use_bfactor", False)), args.get("pocket_mode", "nearest"),
            args.get("n_true", 48), args.get("geom", "real"),
            args["protein_pattern"], args["ligand_pattern"])


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--ckpts", required=True, help="glob or comma-separated list of best.pt files")
    p.add_argument("--out", default="runs/probe")
    p.add_argument("--probes", default=DEFAULT_PROBES)
    p.add_argument("--splits", default="test,BDB2020+")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--cache_dir", default="")
    a = p.parse_args(argv)

    cks = sorted(glob.glob(a.ckpts)) if any(c in a.ckpts for c in "*?[") else \
        [c.strip() for c in a.ckpts.split(",") if c.strip()]
    cks = [c for c in cks if os.path.exists(c)]
    if not cks:
        print(f"[probe] kontrol noktasi bulunamadi: {a.ckpts}")
        return None
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    probes = [s.strip() for s in a.probes.split(",") if s.strip()]
    if probes[0] != "none":
        probes = ["none"] + [x for x in probes if x != "none"]   # d|mu| and r_base need the baseline first
    want = [s.strip() for s in a.splits.split(",") if s.strip()]

    loaded = []
    for c in cks:
        ck = torch.load(c, weights_only=False)
        if ck["args"].get("mode") != "struct":
            print(f"[probe] {c}: yapi modu degil, atlaniyor")
            continue
        loaded.append((c, ck))
    if not loaded:
        return None
    keys = {_ds_key(ck["args"]) for _, ck in loaded}
    assert len(keys) == 1, "kontrol noktalari ayni veri kumesi ayarlarini paylasmali"
    args = loaded[0][1]["args"]

    cache = os.path.join(a.cache_dir or args.get("cache_dir") or a.data_dir,
                         f"emb_{args['esm']}_{args['esm_model'].split('/')[-1] if args['esm'] == 'esm' else 'oh'}.pt")
    emb = ProteinEmbedder(args["esm"], args["esm_model"], cache_path=cache)
    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=args["clean_level"])
    exts = load_external_sets(a.data_dir)

    # built once, reused for every checkpoint and every probe: the parsing is the expensive part
    dsets = {}
    for split in want:
        if split == "test":
            df, pp, lp = tabs["test"], args["protein_pattern"], args["ligand_pattern"]
        elif split in exts:
            df, pp, lp = exts[split]
        else:
            continue
        ds = StructAffinityDataset(df, emb, pp, lp, args["cutoff"], args["max_res"], args["max_atoms"],
                                   args["max_poses"], 0, 0.5, f"probe-{split}", eval_states=0,
                                   use_bfactor=args.get("use_bfactor", False), keep_coords=True,
                                   # the probe must feed the model exactly what it was trained on,
                                   # otherwise it measures a featurisation mismatch instead of geometry use
                                   pocket_mode=args.get("pocket_mode", "nearest"),
                                   n_true=args.get("n_true", 48), geom=args.get("geom", "real"))
        if len(ds):
            dsets[split] = (ds, DataLoader(ds, batch_size=args["bs"], shuffle=False, collate_fn=collate))

    res = {"config": {"ckpts": cks, "probes": probes, "splits": list(dsets)}}
    rows = []
    for cpath, ck in loaded:
        tag = os.path.basename(os.path.dirname(cpath))
        model = EnsAff(ck.get("in_dim", ck["esm_dim"]), "struct", args["d"], args["c"], args["blocks"],
                       args["heads"], args["slots"], n_recycle=args["recycle"],
                       dropout=args["dropout"]).to(dev)
        model.load_state_dict(ck["model"])
        base = {}
        for probe in probes:
            for split, (ds, dl) in dsets.items():
                ds.probe = probe
                _ids, y, mu, _sd = predict(model, dl, dev)
                m = regression_metrics(y, mu)
                if probe == "none":
                    base[split] = mu.copy()
                    m["dmu"], m["r_base"] = 0.0, 1.0
                else:
                    b = base[split]
                    m["dmu"] = float(np.abs(mu - b).mean())
                    m["r_base"] = float(stats.pearsonr(mu, b)[0]) if np.std(mu) > 0 else float("nan")
                res.setdefault(tag, {}).setdefault(probe, {})[split] = m
                rows.append((tag, probe, split, m))
                print(f"  {tag:<12} {probe:<14} {split:<10} n={m['n']:<5} rmse={m['rmse']:.3f} "
                      f"r={m['pearson']:.3f} d|mu|={m['dmu']:.4f} r_base={m['r_base']:.4f}", flush=True)
        for ds, _dl in dsets.values():
            ds.probe = "none"
        del model
        if dev.type == "cuda":
            torch.cuda.empty_cache()

    # mean +- std over checkpoints, written as text so the report can inline it verbatim
    lines = ["(d|mu| = bozulmamis tahmine gore ortalama |degisim|, pK;  r_base = bozulmus vs bozulmamis Pearson)",
             "(r_base ~ 1.000 ve d|mu| ~ 0  =>  model bu kanali kullanmiyor)", ""]
    for split in dsets:
        lines.append(f"  {split}  ({len(loaded)} kontrol noktasi, ortalama +- std)")
        lines.append(f"    {'sonda':<16} {'rmse':>14} {'pearson':>14} {'d|mu| (pK)':>16} {'r_base':>14}")
        for probe in probes:
            got = [m for t, pr, sp, m in rows if pr == probe and sp == split]
            if not got:
                continue
            f = lambda k: (np.mean([g[k] for g in got]), np.std([g[k] for g in got], ddof=1) if len(got) > 1 else 0.0)  # noqa: E731
            (rm, rs), (pm, ps), (dm, dsd), (bm, bs) = f("rmse"), f("pearson"), f("dmu"), f("r_base")
            lines.append(f"    {probe:<16} {rm:7.3f}+-{rs:<5.3f} {pm:7.3f}+-{ps:<5.3f} "
                         f"{dm:9.4f}+-{dsd:<5.4f} {bm:7.4f}+-{bs:<5.4f}")
        lines.append("")
    # verdict: the point of the probe is whether erasing geometry costs anything
    for split in dsets:
        g0 = [m for t, pr, sp, m in rows if pr == "none" and sp == split]
        gc = [m for t, pr, sp, m in rows if pr == "const:8" and sp == split]
        if not (g0 and gc):
            continue
        d = np.mean([m["rmse"] for m in gc]) - np.mean([m["rmse"] for m in g0])
        rb = np.mean([m["r_base"] for m in gc])
        verdict = ("GEOMETRIYI KULLANIYOR" if d > 0.05 and rb < 0.99 else
                   "sinirda" if d > 0.02 or rb < 0.995 else "GEOMETRIYI KULLANMIYOR")
        lines.append(f"  KARAR [{split}]: geometriyi silmenin bedeli {d:+.3f} pK, r_base={rb:.4f} -> {verdict}")
    lines.append("  (olcut: bedel > 0.05 pK VE r_base < 0.99 ise model geometriyi okuyor demektir)")
    txt = "\n".join(lines)
    print("\n" + txt)
    with open(os.path.join(a.out, "probe_report.txt"), "w") as fh:
        fh.write(txt)
    with open(os.path.join(a.out, "metrics.json"), "w") as fh:
        json.dump(res, fh, indent=2)
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/voxel.py
r"""IKINCI MIMARI: voksel tabanli 3B evrisimli ag.


NEDEN BU MODUL VAR
------------------
v12, cift-gosterim modelinde uc boyutlu geometrinin olculebilir hicbir katki
saglamadigini gosterdi. Buna verilebilecek en guclu hakem itirazi sudur:
"katki sifir cikti cunku mimarin geometriyi okumak icin kurulmamis; daha iyi
bir uzamsal onyargiya sahip bir model farkli davranirdi."

Bu modul o itirazi kapatmak icin yazildi. Burada egitilen model, cift-gosterim
modeliyle SADECE girdi kompleksini paylasir:

  * girdi bir mesafe matrisi degil, bir uzamsal isgal izgarasi (3B voksel),
  * dil modeli gomulmesi yok; kanallar atom/kalinti sinifi,
  * dikkat yok; yalnizca 3B evrisim, yani tum tasarim varsayimi uzamsaldir,
  * ayri egitim dongusu, ayri parametreler.

Eger uzamsal evrisimin tum amaci konumu okumak olan bir mimari de ayni taniyi
veriyorsa, bulgu "bu model" olmaktan cikip "bu model sinifi" olur.

SONDA BATARYASI KOORDINAT UZAYINDA
----------------------------------
Sondalar mesafe matrisine degil koordinatlara uygulanir. Adlar, anlami ayni
kalan yerlerde ayni tutuldu; farkli oldugu tek yerde yeni bir ad verildi:

  translate:s   ligandin katı yer degistirmesi            (ayni anlam)
  rotate        ligandin kendi agirlik merkezi etrafinda  (ayni anlam)
  scale:s       tum koordinatlarin merkeze gore olceklenmesi
  permute_res   kalinti SINIF etiketlerinin karistirilmasi, konumlar yerinde
                (cift-gosterim modelindeki "mesafeler yanlis kalintiya bagli"
                 hasarinin birebir karsiligi)
  permute_atom  ligand atom sinif etiketlerinin karistirilmasi
  scramble      her atomun kutu icinde rastgele bir konuma tasinmasi, bilesim
                korunur. Bu, "const:8"in voksel karsiligidir: ne varsa durur,
                nerede oldugu silinir. Mesafe kavrami olmadigi icin ad da
                farklidir.

Hasar, her komplekste karmadan tureyen sabit bir tohumla uretilir, boylece bir
sonda calistirmadan calistirmaya ve modelden modele birebir tekrarlanir.

KOLLAR
------
  vx        gercek koordinatlar
  vx_scr    scramble ile SIFIRDAN egitim (geometrinin ogrenilebilir katkisi)
  vx_rand   rastgele kalinti cebi, gercek koordinatlar (cep secimi)

KULLANIM
--------
  python -m ensaff.voxel --arm vx --seeds 0,1,2,3,4 --out runs/vx
  python -m ensaff.voxel --probe-only runs/vx --probes none,translate:2.0,scramble

GPU gerekir (CPU'da da calisir ama cok yavastir). Yuksek RAM GEREKMEZ:
koordinatlar onbellege alinir, izgara her ornekte anlik kurulur.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from .contactcount import LIG_CLASS, N_LIG_CLASS, N_RES_CLASS, RES_CLASS
from .evaluate import regression_metrics
from .features import ELEMENTS, extract_pocket, extract_pocket_random, featurize_mol, load_ligand_poses, parse_pdb

GRID = 24          # kenar uzunlugu, voksel sayisi
SPACING = 1.0      # angstrom / voksel
N_CH = N_RES_CLASS + N_LIG_CLASS


# --------------------------------------------------------------------------- koordinat uzayinda hasar
def _rand_rotation(rng):
    q, _ = np.linalg.qr(rng.randn(3, 3))
    if np.linalg.det(q) < 0:
        q[:, 0] = -q[:, 0]
    return q.astype(np.float32)


def _seeded(cid: str, spec: str, salt: str):
    h = hashlib.md5(f"{salt}{spec}{cid}".encode()).hexdigest()[:8]
    return np.random.RandomState(int(h, 16))


def perturb_coords(rec: dict, spec: str, salt: str = "", rng=None):
    """Kontrollu hasar. `rec`i degistirmez; (prot_xyz, prot_cls, lig_xyz, lig_cls) dondurur."""
    px, pc = rec["prot_xyz"], rec["prot_cls"]
    lx, lc = rec["lig_xyz"], rec["lig_cls"]
    name, _, val = spec.partition(":")
    if name in ("", "none"):
        return px, pc, lx, lc
    r = rng if rng is not None else _seeded(rec["id"], spec, salt)
    if name == "translate":
        return px, pc, lx + r.randn(1, 3).astype(np.float32) * float(val or 1.0), lc
    if name == "rotate":
        c = lx.mean(0, keepdims=True)
        return px, pc, ((lx - c) @ _rand_rotation(r).T + c).astype(np.float32), lc
    if name == "scale":
        s = float(val)
        c = np.concatenate([px, lx], 0).mean(0, keepdims=True)
        return ((px - c) * s + c).astype(np.float32), pc, ((lx - c) * s + c).astype(np.float32), lc
    if name == "permute_res":
        # konumlar yerinde, kalinti sinifi etiketleri kaydirilir
        return px, pc[r.permutation(len(pc))], lx, lc
    if name == "permute_atom":
        return px, pc, lx, lc[r.permutation(len(lc))]
    if name == "scramble":
        # "const:8"in karsiligi: bilesim durur, konum silinir.
        # Ornekleme araligi kutudan bir voksel ICERIDE tutulur. Tam kenarda ornekleyen
        # bir atomun trilineer agirliginin yarisi izgaranin disina duser; bu, karistirilmis
        # kolun toplam kutlesini gercek kola gore dusururdu ve geometriyle ilgisi olmayan
        # bir bilesim farki yaratirdi. Icerideki aralik iki kolu ayni toplam kutlede tutar.
        c = np.asarray(rec["centre"], dtype=np.float32).reshape(1, 3)
        lim = GRID * SPACING / 2.0 - SPACING
        npx = (r.uniform(-lim, lim, size=px.shape).astype(np.float32) + c)
        nlx = (r.uniform(-lim, lim, size=lx.shape).astype(np.float32) + c)
        return npx, pc, nlx, lc
    raise ValueError(f"bilinmeyen sonda: {spec}")


# --------------------------------------------------------------------------- izgara
def splat(prot_xyz, prot_cls, lig_xyz, lig_cls, centre) -> np.ndarray:
    """Trilineer dagitim. SABIT bir merkeze oturtulmus [N_CH, G, G, G] izgara.

    `centre` bozulmamis ligandin agirlik merkezidir ve kayit kurulurken bir kez
    hesaplanir. Izgarayi o ana ligandin merkezine oturtmak cazip gorunur ama
    yanlistir: o zaman translate sondasi cerceveyi de birlikte tasir, sonda
    kismen kendini goturur ve kutunun disina tasan atomlar yuzunden "geometri
    degisti" ile "bilesim degisti" birbirine karisir. Sabit cerceve ikisini
    ayirir.

    Sert (ikili) isgal yerine trilineer dagitim kullanilir: bir voksel genisliginden
    kucuk yer degistirmeler boylece girdide GORUNUR kalir. Modele geometriyi okumak
    icin mumkun olan en iyi sans verilmis olur, ki bulgu "izgara cok kabaydi"
    diye aciklanamasin.
    """
    g = np.zeros((N_CH, GRID, GRID, GRID), dtype=np.float32)
    centre = np.asarray(centre, dtype=np.float32).reshape(3)
    half = GRID / 2.0
    for xyz, cls, off in ((prot_xyz, prot_cls, 0), (lig_xyz, lig_cls, N_RES_CLASS)):
        if len(xyz) == 0:
            continue
        p = (xyz - centre) / SPACING + half - 0.5          # voksel koordinatlari
        i0 = np.floor(p).astype(np.int64)
        frac = (p - i0).astype(np.float32)
        for dx in (0, 1):
            for dy in (0, 1):
                for dz in (0, 1):
                    w = (np.where(dx, frac[:, 0], 1 - frac[:, 0]) *
                         np.where(dy, frac[:, 1], 1 - frac[:, 1]) *
                         np.where(dz, frac[:, 2], 1 - frac[:, 2]))
                    idx = i0 + np.array([dx, dy, dz])
                    ok = ((idx >= 0).all(1) & (idx < GRID).all(1) & (w > 0))
                    if not ok.any():
                        continue
                    np.add.at(g, (cls[ok] + off, idx[ok, 0], idx[ok, 1], idx[ok, 2]), w[ok])
    return g


# --------------------------------------------------------------------------- veri kumesi
def _lig_cls_from_x(x: np.ndarray) -> np.ndarray:
    idx = x[:, :len(ELEMENTS) + 1].argmax(1)
    out = np.full(len(idx), N_LIG_CLASS - 1, dtype=np.int64)
    for i, e in enumerate(ELEMENTS):
        out[idx == i] = LIG_CLASS.get(e, N_LIG_CLASS - 1)
    return out


class VoxelDataset(Dataset):
    """Cep agir atom koordinatlari + ligand koordinatlari; izgara her ornekte kurulur.

    Onbellekte tutulan sey koordinatlardir (kompleks basina birkac KB), izgara degil.
    Bu yuzden YUKSEK RAM GEREKMEZ ve sondalar koordinat uzayinda uygulanabilir.
    """

    def __init__(self, df, protein_pattern: str, ligand_pattern: str, cutoff: float = 10.0,
                 max_res: int = 64, max_atoms: int = 72, name: str = "", arm: str = "vx",
                 cache: str | None = None):
        self.arm, self.probe = arm, "none"
        if cache and os.path.exists(cache):
            self.recs = list(np.load(cache, allow_pickle=True)["recs"])
            print(f"[voksel:{name}] {len(self.recs)} kompleks (onbellekten)")
            return
        self.recs, skipped, clipped = [], 0, 0
        for r in df.itertuples():
            pp, lp = protein_pattern.format(id=r.id), ligand_pattern.format(id=r.id)
            if not (os.path.exists(pp) and os.path.exists(lp)):
                skipped += 1
                continue
            mol, poses = load_ligand_poses(lp, max_poses=1)
            lig = featurize_mol(mol, max_atoms=max_atoms) if mol is not None else None
            if lig is None or not poses:
                skipped += 1
                continue
            chains = parse_pdb(pp)
            if arm == "vx_rand":
                prng = _seeded(str(r.id), "", "pock")
                sel, _ = extract_pocket_random(chains, poses, max_res=max_res, rng=prng)
            else:
                sel, _ = extract_pocket(chains, poses, cutoff=cutoff, max_res=max_res)
            if not sel:
                skipped += 1
                continue
            xyz, cls = [], []
            for ci, ri in sel:
                rc = chains[ci]["coords"][ri]
                k = RES_CLASS.get(chains[ci]["seq"][ri], 0)
                xyz.append(np.asarray(rc, dtype=np.float32))
                cls.append(np.full(len(rc), k, dtype=np.int64))
            lx = np.asarray(poses[0], dtype=np.float32)[: lig["x"].shape[0]]
            # Kutuya sigmayan protein atomlari burada atilir. Boylece gercek kol ile
            # karistirilmis kol tam olarak ayni atom cokluguyla calisir ve aralarindaki
            # tek fark konum olur.
            px_all, pc_all = np.concatenate(xyz, 0), np.concatenate(cls, 0)
            centre = lx.mean(0).astype(np.float32)
            keep = (np.abs(px_all - centre) <= GRID * SPACING / 2.0 - SPACING).all(1)
            px_all, pc_all = px_all[keep], pc_all[keep]
            if len(px_all) == 0:
                skipped += 1
                continue
            if not (np.abs(lx - centre) <= GRID * SPACING / 2.0 - SPACING).all():
                clipped += 1        # ligand kutuya sigmiyor; sayilir ve bildirilir
            self.recs.append({"id": str(r.id), "y": float(r.y), "centre": centre,
                              "prot_xyz": px_all,
                              "prot_cls": pc_all,
                              "lig_xyz": lx,
                              "lig_cls": _lig_cls_from_x(np.asarray(lig["x"]))[: len(lx)]})
        msg = f"[voksel:{name}] {len(self.recs)} kompleks ({skipped} atlandi), kol={arm}"
        if clipped:
            msg += (f"  !! {clipped} komplekste ligand {GRID} A kutuya tam sigmiyor "
                    f"(bu komplekslerde kenar atomlari izgaraya girmez)")
        print(msg)
        if cache:
            os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
            np.savez_compressed(cache, recs=np.array(self.recs, dtype=object))

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        px, pc, lx, lc = r["prot_xyz"], r["prot_cls"], r["lig_xyz"], r["lig_cls"]
        if self.arm == "vx_scr":                       # SIFIRDAN egitimde kalici hasar
            px, pc, lx, lc = perturb_coords(r, "scramble", salt="arm")
        if self.probe and self.probe != "none":        # cikarim aninda sonda
            rec = {"id": r["id"], "centre": r["centre"], "prot_xyz": px,
                   "prot_cls": pc, "lig_xyz": lx, "lig_cls": lc}
            px, pc, lx, lc = perturb_coords(rec, self.probe, salt="probe")
        return {"id": r["id"], "y": np.float32(r["y"]),
                "g": splat(px, pc, lx, lc, r["centre"])}


def vox_collate(items):
    return {"ids": [it["id"] for it in items],
            "g": torch.from_numpy(np.stack([it["g"] for it in items])),
            "y": torch.tensor([it["y"] for it in items], dtype=torch.float32)}


# --------------------------------------------------------------------------- model
class VoxelCNN(nn.Module):
    """Duz 3B evrisimli ag. Dikkat yok, mesafe kodlamasi yok, dil modeli yok.

    Cikis cift-gosterim modeliyle ayni: ortalama ve log varyans, beta-NLL ile
    egitilir, boylece iki mimarinin kayiplari ve belirsizlikleri karsilastirilabilir.
    """

    def __init__(self, width: int = 32, dropout: float = 0.15):
        super().__init__()
        w = width
        def blk(i, o, pool=True):
            layers = [nn.Conv3d(i, o, 3, padding=1, bias=False), nn.BatchNorm3d(o), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool3d(2))
            return nn.Sequential(*layers)
        self.net = nn.Sequential(blk(N_CH, w), blk(w, 2 * w), blk(2 * w, 4 * w),
                                 blk(4 * w, 4 * w, pool=False))
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(8 * w, 4 * w), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout), nn.Linear(4 * w, 2))

    def forward(self, g):
        h = self.net(g)
        h = torch.cat([h.mean((2, 3, 4)), h.amax((2, 3, 4))], 1)
        o = self.head(self.drop(h))
        return o[:, 0], o[:, 1].clamp(-6.0, 4.0)


def beta_nll(mu, logvar, y, beta: float = 0.5):
    var = logvar.exp()
    nll = 0.5 * (logvar + (y - mu) ** 2 / var)
    return (nll * var.detach() ** beta).mean()


# --------------------------------------------------------------------------- egitim
def run_epoch(model, loader, dev, opt=None, beta: float = 0.5):
    train = opt is not None
    model.train(train)
    tot, n, P, Y, I = 0.0, 0, [], [], []
    for b in loader:
        g, y = b["g"].to(dev, non_blocking=True), b["y"].to(dev, non_blocking=True)
        with torch.set_grad_enabled(train):
            mu, lv = model(g)
            loss = beta_nll(mu, lv, y, beta)
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        tot += float(loss.detach()) * len(y)
        n += len(y)
        P.append(mu.detach().float().cpu().numpy())
        Y.append(y.detach().float().cpu().numpy())
        I += b["ids"]
    return tot / max(n, 1), np.concatenate(P), np.concatenate(Y), I


def train_one(seed: int, tr, va, te, out: str, dev, epochs: int, bs: int, lr: float,
              width: int, workers: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = VoxelCNN(width=width).to(dev)
    npar = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    kw = dict(num_workers=workers, collate_fn=vox_collate, pin_memory=(dev.type == "cuda"),
              persistent_workers=workers > 0)
    ltr = DataLoader(tr, batch_size=bs, shuffle=True, drop_last=True, **kw)
    lva = DataLoader(va, batch_size=bs, **kw)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        t0 = time.time()
        trl, *_ = run_epoch(model, ltr, dev, opt)
        val, vp, vy, _ = run_epoch(model, lva, dev)
        vr = float(np.sqrt(((vp - vy) ** 2).mean()))
        sched.step(vr)
        if vr < best - 1e-4:
            best, bad = vr, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        print(f"  [tohum {seed}] devir {ep + 1}/{epochs} egitim {trl:.4f} dogrulama RMSE {vr:.4f} "
              f"({time.time() - t0:.0f} sn)", flush=True)
        if bad >= 6:
            print(f"  [tohum {seed}] erken durdu")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    d = os.path.join(out, f"seed{seed}")
    os.makedirs(d, exist_ok=True)
    torch.save({"state": model.state_dict(), "width": width, "seed": seed, "n_param": npar},
               os.path.join(d, "model.pt"))
    res = {"seed": seed, "n_param": npar, "val_rmse": best}
    for nm, ds in (("val", va), ("test", te)):
        if ds is None:
            continue
        _, p, y, ids = run_epoch(model, DataLoader(ds, batch_size=bs, **kw), dev)
        res[nm] = regression_metrics(y, p)
        np.savez(os.path.join(d, f"pred_{nm}.npz"), ids=np.array(ids), y=y, p=p)
    json.dump(res, open(os.path.join(d, "metrics.json"), "w"), indent=2)
    print(f"  [tohum {seed}] test RMSE {res.get('test', {}).get('rmse', float('nan')):.4f} "
          f"({npar / 1e6:.3f} M parametre)")
    return res


# --------------------------------------------------------------------------- sonda bataryasi
DEFAULT_PROBES = ("none", "translate:0.5", "translate:1.0", "translate:2.0", "translate:4.0",
                  "translate:8.0", "rotate", "scale:1.5", "permute_res", "permute_atom", "scramble")
COST_THRESH, RBASE_THRESH = 0.05, 0.99


def probe_battery(out: str, ds, dev, probes=DEFAULT_PROBES, bs: int = 32, workers: int = 2,
                  tag: str = "test"):
    """Egitilmis her kontrol noktasini hasarli girdilerle yeniden degerlendirir."""
    seeds = sorted(d for d in os.listdir(out) if d.startswith("seed"))
    if not seeds:
        raise SystemExit(f"{out} altinda kontrol noktasi yok")
    rows, base = {}, {}
    for sd in seeds:
        ck = torch.load(os.path.join(out, sd, "model.pt"), map_location=dev, weights_only=False)
        model = VoxelCNN(width=ck["width"]).to(dev)
        model.load_state_dict(ck["state"])
        for spec in probes:
            ds.probe = spec
            _, p, y, _ = run_epoch(model, DataLoader(ds, batch_size=bs, num_workers=workers,
                                                     collate_fn=vox_collate), dev)
            m = regression_metrics(y, p)
            if spec == "none":
                base[sd] = p
                dmu, rb = 0.0, 1.0
            else:
                b = base[sd]
                dmu = float(np.abs(p - b).mean())
                rb = float(np.corrcoef(p, b)[0, 1]) if p.std() > 1e-9 and b.std() > 1e-9 else 0.0
            rows.setdefault(spec, []).append((m["rmse"], m["pearson"], dmu, rb))
            print(f"  [{sd}] {spec:16s} RMSE {m['rmse']:.4f}  d|mu| {dmu:.4f}  r_temel {rb:.4f}",
                  flush=True)
    ds.probe = "none"
    lines = [f"VOKSEL SONDA RAPORU ({tag}, n = {len(ds)}, {len(seeds)} model)", "=" * 62,
             f"{'sonda':18s}{'RMSE':>16s}{'Pearson':>16s}{'d|mu|':>10s}{'r_temel':>10s}"]
    agg = {}
    for spec in probes:
        a = np.array(rows[spec])
        agg[spec] = {"rmse": float(a[:, 0].mean()), "rmse_sd": float(a[:, 0].std()),
                     "pearson": float(a[:, 1].mean()), "dmu": float(a[:, 2].mean()),
                     "rbase": float(a[:, 3].mean())}
        lines.append(f"{spec:18s}{a[:, 0].mean():>9.4f} +/-{a[:, 0].std():<5.3f}"
                     f"{a[:, 1].mean():>9.4f} +/-{a[:, 1].std():<5.3f}"
                     f"{a[:, 2].mean():>10.4f}{a[:, 3].mean():>10.4f}")
    cost = agg["scramble"]["rmse"] - agg["none"]["rmse"]
    rb = agg["scramble"]["rbase"]
    uses = (cost > COST_THRESH) and (rb < RBASE_THRESH)
    lines += ["", f"geometriyi silmenin bedeli (scramble - none) = {cost:+.4f} pK",
              f"tohumlar arasi RMSE standart sapmasi            =  {agg['none']['rmse_sd']:.4f} pK",
              f"bozulmus tahminlerin korelasyonu                =  {rb:.4f}",
              f"KARAR: {'GEOMETRIYI KULLANIYOR' if uses else 'GEOMETRIYI KULLANMIYOR'} "
              f"(olcut: bedel > {COST_THRESH} pK VE r_temel < {RBASE_THRESH})"]
    txt = "\n".join(lines)
    os.makedirs(out, exist_ok=True)
    open(os.path.join(out, f"probe_report_{tag}.txt"), "w").write(txt + "\n")
    json.dump(agg, open(os.path.join(out, f"probes_{tag}.json"), "w"), indent=2)
    print("\n" + txt)
    return agg


# --------------------------------------------------------------------------- CLI
def build_sets(a, arm: str):
    from .data import load_lp_pdbbind
    sp = load_lp_pdbbind(a.splits, clean_level=a.clean_level)
    mk = lambda df, nm: VoxelDataset(df, a.protein_pattern, a.ligand_pattern, cutoff=a.cutoff,
                                     max_res=a.max_res, name=nm, arm=arm,
                                     cache=(os.path.join(a.cache, f"{arm}_{nm}.npz") if a.cache else None))
    return mk(sp["train"], "train"), mk(sp["val"], "val"), mk(sp["test"], "test")


def main(argv=None):
    p = argparse.ArgumentParser(description="Voksel tabanli 3B-CNN kolu (ikinci mimari)")
    p.add_argument("--arm", default="vx", choices=["vx", "vx_scr", "vx_rand"])
    p.add_argument("--seeds", default="0,1,2")
    p.add_argument("--out", default="runs/vx")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--bs", type=int, default=24)
    p.add_argument("--lr", type=float, default=1.5e-3)
    p.add_argument("--width", type=int, default=32)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--splits", default="LP_PDBBind.csv")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--protein_pattern", default="pdb/{id}/{id}_protein.pdb")
    p.add_argument("--ligand_pattern", default="pdb/{id}/{id}_ligand.sdf")
    p.add_argument("--cutoff", type=float, default=10.0)
    p.add_argument("--max_res", type=int, default=64)
    p.add_argument("--cache", default="cache_vox")
    p.add_argument("--probes", default=",".join(DEFAULT_PROBES))
    p.add_argument("--probe-only", dest="probe_only", default=None,
                   help="egitimi atla, bu dizindeki kontrol noktalarina sonda uygula")
    a = p.parse_args(argv)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[voksel] cihaz = {dev}, izgara {GRID}^3 @ {SPACING} A, {N_CH} kanal")
    if dev.type != "cuda":
        print("[voksel] !! UYARI: GPU bulunamadi. CPU'da bu kol saatler surer.")

    if a.probe_only:
        _, _, te = build_sets(a, a.arm)
        probe_battery(a.probe_only, te, dev, probes=tuple(a.probes.split(",")),
                      workers=a.workers, tag="test")
        return

    tr, va, te = build_sets(a, a.arm)
    os.makedirs(a.out, exist_ok=True)
    res = [train_one(int(s), tr, va, te, a.out, dev, a.epochs, a.bs, a.lr, a.width, a.workers)
           for s in a.seeds.split(",")]
    r = np.array([x["test"]["rmse"] for x in res if "test" in x])
    print(f"\n[voksel:{a.arm}] test RMSE tohum basina {r.mean():.4f} +/- {r.std():.4f} ({len(r)} tohum)")
    json.dump({"arm": a.arm, "per_seed_rmse": r.tolist(), "mean": float(r.mean()),
               "sd": float(r.std())}, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    if a.arm == "vx":
        probe_battery(a.out, te, dev, probes=tuple(a.probes.split(",")), workers=a.workers, tag="test")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/conformers.py
r"""D3: build a real conformational ensemble for a complex from OTHER crystal structures of the same protein.

The project's question is whether observed protein dynamics helps affinity prediction. Until now the "states"
fed to the aggregation layer were synthetic. The dataset itself, however, contains genuine conformational
variation: 1581 proteins appear in two or more PDB entries (46% of complexes; 56% of the test split). Those
entries are the same protein captured in different crystal forms with different ligands - an experimentally
observed ensemble.

For a complex, sibling entries of the same protein sequence are superposed onto its pocket (Kabsch on the
shared pocket residues) and each gives one alternative pocket geometry. The ligand pose stays fixed, so the
K states differ only in where the protein atoms sit.

LEAKAGE CONSTRAINT (enforced and tested): a sibling contributes GEOMETRY ONLY. Its affinity label is never
read. Siblings are additionally restricted to the same split, so no val/test structure can influence a
training example and vice versa.
"""
from __future__ import annotations

import numpy as np


def kabsch(P: np.ndarray, Q: np.ndarray):
    """Rotation+translation minimising RMSD of P onto Q (both [n,3], paired). Returns (R, t)."""
    pc, qc = P.mean(0), Q.mean(0)
    H = (P - pc).T @ (Q - qc)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, qc - R @ pc


def sibling_index(df, split_col: str = None):
    """{sequence: [ids]} - which complexes share a protein sequence. Only ids inside `df` are linked, so
    callers pass one split at a time and siblings can never cross a split boundary."""
    idx = {}
    for r in df.itertuples():
        idx.setdefault(str(r.seq), []).append(str(r.id))
    return {k: v for k, v in idx.items() if len(v) > 1}


def pocket_states_from_siblings(ref_chains, ref_pocket, ref_pose, sibling_chains_list, max_states: int = 4):
    """Distance matrices [K, n_pocket, n_lig] from the reference plus superposed sibling conformations.

    Each sibling is aligned to the reference on the shared pocket residues (matched by sequence position
    within the chain), then the SAME ligand pose is measured against the sibling's pocket atoms.
    """
    def dist(pocket_xyz, lig):
        return np.stack([np.sqrt(((rc[:, None, :] - lig[None, :, :]) ** 2).sum(-1)).min(0)
                         for rc in pocket_xyz])

    ref_xyz = [ref_chains[ci]["coords"][ri] for ci, ri in ref_pocket]
    out = [dist(ref_xyz, ref_pose)]
    used = 0
    for sib in sibling_chains_list:
        if used >= max_states:
            break
        # align on CA-like first atoms of pocket residues that exist in both
        P, Q, sib_xyz, ok = [], [], [], True
        for ci, ri in ref_pocket:
            ch = ref_chains[ci]
            scand = next((c for c in sib if c["seq"] == ch["seq"]), None)
            if scand is None or ri >= len(scand["coords"]) or not len(scand["coords"][ri]):
                ok = False
                break
            sib_xyz.append(scand["coords"][ri])
            P.append(scand["coords"][ri][0])
            Q.append(ch["coords"][ri][0])
        if not ok or len(P) < 8:
            continue
        R, t = kabsch(np.asarray(P), np.asarray(Q))
        out.append(dist([(R @ x.T).T + t for x in sib_xyz], ref_pose))
        used += 1
    return np.stack(out).astype(np.float32), used


def ensemble_rmsf(ref_chains, ref_pocket, sibling_chains_list, max_sib: int = 8):
    """Per-pocket-residue conformational spread measured ACROSS sibling crystal structures.

    This is the ensemble analogue of a B-factor: instead of one structure's refined mobility parameter, it
    is how far a residue actually moves between independent determinations of the same protein. B-factors
    (a cruder, single-structure proxy) already produced a measurable gain, and the spread here is strongly
    non-uniform - across mpro the mean pocket deviation is 0.34 A while the maximum reaches 4.2 A - so the
    signal lives in which residues move, not in the average.

    Returns [n_pocket, 2]: standardised per-residue RMSF, and the number of siblings it was measured over
    (scaled), or None when fewer than two siblings could be aligned.
    """
    per_res = [[] for _ in ref_pocket]
    used = 0
    for sib in sibling_chains_list:
        if used >= max_sib:
            break
        P, Q, sib_xyz, ok = [], [], [], True
        for ci, ri in ref_pocket:
            ch = ref_chains[ci]
            scand = next((c for c in sib if c["seq"] == ch["seq"]), None)
            if scand is None or ri >= len(scand["coords"]) or not len(scand["coords"][ri]):
                ok = False
                break
            sib_xyz.append(scand["coords"][ri])
            P.append(scand["coords"][ri][0])
            Q.append(ch["coords"][ri][0])
        if not ok or len(P) < 8:
            continue
        R, t = kabsch(np.asarray(P), np.asarray(Q))
        for k, (ci, ri) in enumerate(ref_pocket):
            moved = (R @ sib_xyz[k].T).T + t
            ref_at = ref_chains[ci]["coords"][ri]
            n = min(len(moved), len(ref_at))
            per_res[k].append(float(np.sqrt(((moved[:n] - ref_at[:n]) ** 2).sum(1)).mean()))
        used += 1
    if used < 2:
        return None
    rmsf = np.array([np.mean(v) if v else 0.0 for v in per_res], dtype=np.float32)
    mu, sd = float(rmsf.mean()), float(rmsf.std())
    z = (rmsf - mu) / (sd if sd > 1e-6 else 1.0)
    return np.stack([np.clip(z, -5, 5), np.full(len(rmsf), used / 8.0, dtype=np.float32)], 1)


In [ ]:
%%writefile ensaff/ccdyn.py
r"""Temas sayimi uzerinde: mesafe cozunurlugu, GERCEK konformasyon toplulugu ve hareketlilik.


NEDEN BU MODUL VAR
------------------
v14 temas-sayim modelinin geometriyi kullandigini ve sinir aglari kadar iyi
tahmin ettigini gosterdi. Bu, makalede iki boslugu acik birakti.

1) Sinir aglari icin "geometriyi kaldirip sifirdan egit" kolunu kosabildik ama
   temas sayimi icin kosmadik; sinirliliklara "mesafe olmadan hic ozelligi
   kalmaz" diye yazdim. Bu tam dogru degil. Kabuk SAYISINI dusurmek mesafe
   cozunurlugunu kademeli olarak siler: bes kabuk tam cozunurluk, tek kabuk
   yalnizca bir kesme yaricapi, sifir kabuk ise kesme bile yok (cepteki her
   cift bir kez sayilir), yani saf bilesim. Bu, o model icin birebir eslesen
   "sabit mesafe" koludur ve olculebilir.

2) Projenin ozgun sorusu protein hareketliliginin baglanma gucune katkisiydi.
   O soru simdiye kadar GEOMETRIYE KOR bir model uzerinde soruldu, bu yuzden
   cevabi yorumlanamazdi: geometrinin kendisini okumayan bir modele geometrinin
   DEGISIMINI sormak bos bir testtir. Temas-sayim modeli geometriyi kullandigi
   KANITLANMIS bir modeldir, dolayisiyla adil test burada yapilabilir.

Konformasyonlar sentetik degildir. `conformers.py` ayni proteinin baska PDB
kayitlarini cebe gore ust uste bindirir; ligand pozu sabit kalir, yalnizca
protein atomlarinin yeri degisir. Kardesler ayni bolme icinden secilir ve
yalnizca GEOMETRI katar, etiketleri hic okunmaz.

KOLLAR
------
  shells      kabuk sayisi 0,1,2,3,5 ile yeniden egitim (mesafe cozunurlugu)
  dyn         K konformasyon uzerinden toplama (ortalama; ortalama+sapma)
  rmsf        kardesler arasi kalinti hareketliligi ile agirliklandirilmis sayimlar

KULLANIM
--------
  python -m ensaff.ccdyn --arms shells,dyn,rmsf --out runs/ccdyn

CPU yeterlidir, GPU KULLANILMAZ, yuksek RAM gerekmez.
"""
from __future__ import annotations

import argparse
import json
import os

import numpy as np
import pandas as pd

from .conformers import ensemble_rmsf, pocket_states_from_siblings, sibling_index
from .contactcount import (LIG_CLASS, N_LIG_CLASS, N_RES_CLASS, RES_CLASS, _lig_classes)
from .data import load_external_sets, load_lp_pdbbind
from .evaluate import regression_metrics
from .features import extract_pocket, featurize_mol, load_ligand_poses, parse_pdb

DEFAULT_EDGES = (4.0, 6.0, 8.0, 10.0, 12.0)


def edges_for(n_shell: int):
    """Kabuk sayisindan kenarlar. 0 -> kesme yok (saf bilesim)."""
    if n_shell <= 0:
        return ()
    if n_shell == 1:
        return (12.0,)
    if n_shell == 2:
        return (6.0, 12.0)
    if n_shell == 3:
        return (4.0, 8.0, 12.0)
    if n_shell == 5:
        return DEFAULT_EDGES
    raise ValueError(f"desteklenmeyen kabuk sayisi: {n_shell}")


def counts(D, rc, lc, edges, weight=None):
    """[res_class, lig_class, shell] sayimlari, duzlestirilmis.

    edges bos ise mesafe hic kullanilmaz: cepteki her kalinti-atom cifti bir kez
    sayilir, yani ozellik vektoru iki bilesimin dis carpimidir. Geometrinin
    TAMAMEN kaldirildigi durum budur.
    `weight` verilirse her kalintinin katkisi o agirlikla carpilir (RMSF kolu).
    """
    ns = max(len(edges), 1)
    feat = np.zeros((N_RES_CLASS, N_LIG_CLASS, ns), dtype=np.float32)
    if not edges:
        ri, ai = np.meshgrid(np.arange(D.shape[0]), np.arange(D.shape[1]), indexing="ij")
        ri, ai = ri.ravel(), ai.ravel()
        w = np.ones(len(ri), dtype=np.float32) if weight is None else weight[ri]
        np.add.at(feat, (rc[ri], lc[ai], 0), w)
        return feat.reshape(-1)
    shell = np.searchsorted(np.asarray(edges), D, side="left")
    keep = shell < len(edges)
    if keep.any():
        ri, ai = np.nonzero(keep)
        w = np.ones(len(ri), dtype=np.float32) if weight is None else weight[ri]
        np.add.at(feat, (rc[ri], lc[ai], shell[ri, ai]), w)
    return feat.reshape(-1)


# --------------------------------------------------------------------------- kayit kurulumu
def build_records(df, pp_pat, lp_pat, name, cutoff=10.0, max_res=64, max_atoms=72,
                  max_poses=4, max_states=4, want_siblings=True, cache=None):
    """Kompleks basina: referans D, kalinti/atom siniflari, kardes konformasyonlari, RMSF.

    Kardesler YALNIZCA `df` icinden secilir, yani ayni bolmeden. Bir kardesin
    etiketi hicbir yerde okunmaz; katkisi sadece protein atomlarinin konumudur.
    """
    if cache and os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        print(f"[ccdyn:{name}] {len(z['recs'])} kompleks (onbellekten)")
        return list(z["recs"])
    sib = sibling_index(df) if want_siblings else {}
    chains_cache, recs, skipped = {}, [], 0

    CACHE_MAX = 1500        # ayristirilmis yapilari sinirli tut: yuksek RAM gerekmesin
    def chains_of(cid):
        if cid not in chains_cache:
            if len(chains_cache) >= CACHE_MAX:
                chains_cache.pop(next(iter(chains_cache)))
            p = pp_pat.format(id=cid)
            chains_cache[cid] = parse_pdb(p) if os.path.exists(p) else None
        return chains_cache[cid]

    for r in df.itertuples():
        cid = str(r.id)
        pp, lp = pp_pat.format(id=cid), lp_pat.format(id=cid)
        if not (os.path.exists(pp) and os.path.exists(lp)):
            skipped += 1
            continue
        mol, poses = load_ligand_poses(lp, max_poses=max_poses)
        lig = featurize_mol(mol, max_atoms=max_atoms) if mol is not None else None
        if lig is None or not poses:
            skipped += 1
            continue
        chains = chains_of(cid)
        if chains is None:
            skipped += 1
            continue
        sel, D = extract_pocket(chains, poses, cutoff=cutoff, max_res=max_res)
        if not sel or D is None:
            skipped += 1
            continue
        letters = [chains[ci]["seq"][ri] for ci, ri in sel]
        rc = np.array([RES_CLASS.get(c, 0) for c in letters], dtype=np.int64)
        lc = _lig_classes(np.asarray(lig["x"]))
        sib_ids = [i for i in sib.get(str(r.seq), []) if i != cid][: max_states * 2]
        sib_chains = [c for c in (chains_of(i) for i in sib_ids) if c]
        Dk, used, rmsf = D[:1], 0, None
        if sib_chains:
            Dk, used = pocket_states_from_siblings(chains, sel, poses[0], sib_chains,
                                                   max_states=max_states)
            rmsf = ensemble_rmsf(chains, sel, sib_chains)
        recs.append({"id": cid, "y": float(r.y), "rc": rc, "lc": lc,
                     "D": D[0].astype(np.float32),          # referans poz
                     "Dk": Dk.astype(np.float32),           # [K, P, N] gercek konformasyonlar
                     "n_sib": int(used),
                     "rmsf": None if rmsf is None else rmsf[:, 0].astype(np.float32)})
    ks = np.array([r["Dk"].shape[0] for r in recs])
    print(f"[ccdyn:{name}] {len(recs)} kompleks ({skipped} atlandi)  "
          f"K dagilimi: {dict(zip(*np.unique(ks, return_counts=True)))}  "
          f"RMSF olculebilen: {sum(r['rmsf'] is not None for r in recs)}")
    if cache:
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        np.savez_compressed(cache, recs=np.array(recs, dtype=object))
    return recs


# --------------------------------------------------------------------------- ozellik matrisleri
def featurize(recs, n_shell=5, agg="ref", use_rmsf=False, dup=False):
    """agg: ref | mean | meanstd ; use_rmsf: RMSF agirlikli blok ekle ;
    dup: ozellikleri ikiye katla (YENI BILGI YOK).

    `dup`, 250 ozellikli kollarin SEYRELME kontroludur: 125'ten 250'ye cikmanin
    kendisi ne kadar zarar veriyor? Bu olculmeden, 250 ozellikli bir kolun
    kotulesmesi bilgiye mi boyuta mi bagli bilinemez.
    """
    edges = edges_for(n_shell)
    X, y, ids = [], [], []
    for r in recs:
        if agg == "ref":
            f = counts(r["D"], r["rc"], r["lc"], edges)
        else:
            per = np.stack([counts(Dk, r["rc"], r["lc"], edges) for Dk in r["Dk"]])
            f = per.mean(0) if agg == "mean" else np.concatenate([per.mean(0), per.std(0)])
        if use_rmsf:
            w = r["rmsf"] if r["rmsf"] is not None else np.zeros(len(r["rc"]), dtype=np.float32)
            f = np.concatenate([f, counts(r["D"], r["rc"], r["lc"], edges, weight=w)])
        if dup:
            f = np.concatenate([f, f])
        X.append(f)
        y.append(r["y"])
        ids.append(r["id"])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32), np.array(ids)


# --------------------------------------------------------------------------- model ve karsilastirma
def fit_predict(Xtr, ytr, sets, seeds=(0, 1, 2, 3, 4), n_estimators=500):
    """Rastgele orman, tohum basina; her bolme icin topluluk ve tohum tahminleri."""
    from sklearn.ensemble import RandomForestRegressor
    out = {k: [] for k in sets}
    for sd in seeds:
        m = RandomForestRegressor(n_estimators=n_estimators, n_jobs=-1, random_state=sd,
                                  min_samples_leaf=2)
        m.fit(Xtr, ytr)
        for k, (X, _y, _i) in sets.items():
            out[k].append(m.predict(X))
    return {k: np.stack(v) for k, v in out.items()}       # [tohum, n]


def paired_bootstrap(y, pa, pb, n_boot=2000, seed=0):
    """(B - A) icin %95 aralik; iki model ayni yeniden ornekleme uzerinde."""
    rng = np.random.RandomState(seed)
    n, d = len(y), []
    for _ in range(n_boot):
        i = rng.randint(0, n, n)
        d.append(float(np.sqrt(((pb[i] - y[i]) ** 2).mean()) - np.sqrt(((pa[i] - y[i]) ** 2).mean())))
    d = np.asarray(d)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": float(d.mean()), "ci95": [float(lo), float(hi)],
            "significant": bool(lo > 0 or hi < 0)}


def load_network_preds(path):
    """Sinir agi tahminlerini oku: pred_*.csv (cift gosterim) veya pred_*.npz (voksel)."""
    if path.endswith(".csv"):
        d = pd.read_csv(path)
        col = "mu" if "mu" in d.columns else ("pred" if "pred" in d.columns else d.columns[-1])
        idc = "id" if "id" in d.columns else d.columns[0]
        yc = "y" if "y" in d.columns else None
        return (d[idc].astype(str).values, d[yc].values if yc else None, d[col].values)
    z = np.load(path, allow_pickle=True)
    return z["ids"].astype(str), z["y"], z["p"]


def compare_to(ids, y, p, path, label, log, min_common: int = 50):
    """Temas sayimini bir sinir agiyla ortak kimlikler uzerinde eslesmis olarak karsilastirir."""
    if not os.path.exists(path):
        log(f"  {label:34s} tahmin dosyasi yok ({path})")
        return None
    nid, ny, npred = load_network_preds(path)
    common = np.intersect1d(ids, nid)
    if len(common) < min_common:
        log(f"  {label:34s} ortak kimlik az ({len(common)})")
        return None
    a = {i: v for i, v in zip(nid, npred)}
    b = {i: v for i, v in zip(ids, p)}
    yy = {i: v for i, v in zip(ids, y)}
    cy = np.array([yy[i] for i in common])
    net = np.array([a[i] for i in common])
    cc = np.array([b[i] for i in common])
    d = paired_bootstrap(cy, net, cc)          # B - A = temas sayimi - sinir agi
    log(f"  {label:34s} n={len(common):5d}  sinir agi {np.sqrt(((net-cy)**2).mean()):.4f}  "
        f"temas {np.sqrt(((cc-cy)**2).mean()):.4f}  fark {d['diff']:+.4f} "
        f"[{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] "
        f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}")
    return d


# --------------------------------------------------------------------------- CLI
def main(argv=None):
    p = argparse.ArgumentParser(description="Temas sayimi: cozunurluk, konformasyon, hareketlilik")
    p.add_argument("--arms", default="shells,dyn,rmsf")
    p.add_argument("--shells", default="0,1,2,3,5")
    p.add_argument("--seeds", default="0,1,2,3,4")
    p.add_argument("--out", default="runs/ccdyn")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--protein_pattern", default="pdbstruct/{id}/protein.pdb")
    p.add_argument("--ligand_pattern", default="pdbstruct/{id}/ligand.sdf")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--cutoff", type=float, default=10.0)
    p.add_argument("--max_res", type=int, default=64)
    p.add_argument("--max_states", type=int, default=4)
    p.add_argument("--cache", default="cache_ccdyn")
    p.add_argument("--net_pair", default="runs/struct_ens/pred_test.csv")
    p.add_argument("--net_voxel", default="runs/vx/seed0/pred_test.npz")
    a = p.parse_args(argv)

    os.makedirs(a.out, exist_ok=True)
    L = []
    def log(s=""):
        print(s, flush=True)
        L.append(s)

    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=a.clean_level)
    mk = lambda df, n: build_records(df, a.protein_pattern, a.ligand_pattern, n, cutoff=a.cutoff,
                                     max_res=a.max_res, max_states=a.max_states,
                                     cache=(os.path.join(a.cache, f"{n}.npz") if a.cache else None))
    tr, te = mk(tabs["train"], "train"), mk(tabs["test"], "test")
    seeds = tuple(int(x) for x in a.seeds.split(","))
    arms = set(a.arms.split(","))
    res = {}

    log("=" * 78)
    log("TEMAS SAYIMI UZERINDE COZUNURLUK, KONFORMASYON VE HAREKETLILIK")
    log("=" * 78)
    ks = np.array([r["Dk"].shape[0] for r in te])
    log(f"test bolmesinde konformasyon sayisi K: ortalama {ks.mean():.2f}, "
        f"K>1 olan kompleks {int((ks > 1).sum())}/{len(ks)} "
        f"({100 * (ks > 1).mean():.0f}%)")
    log(f"RMSF olculebilen test kompleksi: {sum(r['rmsf'] is not None for r in te)}/{len(te)}")

    def run(tag, **kw):
        Xtr, ytr, _ = featurize(tr, **kw)
        Xte, yte, ite = featurize(te, **kw)
        P = fit_predict(Xtr, ytr, {"test": (Xte, yte, ite)}, seeds=seeds)["test"]
        ens = P.mean(0)
        per = np.array([float(np.sqrt(((q - yte) ** 2).mean())) for q in P])
        m = regression_metrics(yte, ens)
        res[tag] = {"kw": kw, "ensemble_rmse": m["rmse"], "pearson": m["pearson"],
                    "per_seed_mean": float(per.mean()), "per_seed_sd": float(per.std(ddof=1)),
                    "n_feat": int(Xtr.shape[1]), "n": int(len(yte))}
        np.savez(os.path.join(a.out, f"pred_{tag}.npz"), ids=ite, y=yte, p=ens, per_seed=P)
        log(f"  {tag:22s} ozellik {Xtr.shape[1]:4d}  topluluk RMSE {m['rmse']:.4f}"
            f"  Pearson {m['pearson']:.4f}  tohum basina {per.mean():.4f} +/- {per.std(ddof=1):.4f}")
        return ite, yte, ens

    if "shells" in arms:
        log("\n[A] MESAFE COZUNURLUGU (sifirdan egitim, kabuk sayisi azaltilarak)")
        log("  0 kabuk = kesme bile yok, saf bilesim; geometrinin TAMAMEN kaldirildigi kol")
        base = None
        for ns in [int(x) for x in a.shells.split(",")]:
            ite, yte, ens = run(f"shell{ns}", n_shell=ns, agg="ref")
            if ns == 5:
                base = (ite, yte, ens)
        if base is not None:
            log("\n  5 kabuga karsi eslesmis farklar (pozitif = az kabuk daha kotu):")
            for ns in [int(x) for x in a.shells.split(",")]:
                if ns == 5:
                    continue
                z = np.load(os.path.join(a.out, f"pred_shell{ns}.npz"), allow_pickle=True)
                d = paired_bootstrap(base[1], base[2], z["p"])
                res[f"shell{ns}_vs_5"] = d
                log(f"    {ns} kabuk: {d['diff']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]  "
                    f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}")

    if "dyn" in arms:
        log("\n[B] GERCEK KONFORMASYON TOPLULUGU (kardes kristal yapilar)")
        ite, yte, ref = run("dyn_ref", n_shell=5, agg="ref")
        for agg in ("mean", "meanstd"):
            _, _, q = run(f"dyn_{agg}", n_shell=5, agg=agg)
            d = paired_bootstrap(yte, ref, q)
            res[f"dyn_{agg}_vs_ref"] = d
            log(f"    {agg:8s} referansa karsi: {d['diff']:+.4f} "
                f"[{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]  "
                f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}")

    if "rmsf" in arms:
        log("\n[C] HAREKETLILIK OZELLIKLERI (kardesler arasi RMSF ile agirlikli sayimlar)")
        ite, yte, ref = run("rmsf_off", n_shell=5, agg="ref")
        _, _, q = run("rmsf_on", n_shell=5, agg="ref", use_rmsf=True)
        d = paired_bootstrap(yte, ref, q)
        res["rmsf_vs_off"] = d
        log(f"    RMSF eklendiginde: {d['diff']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]  "
            f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}")
        # SEYRELME KONTROLU: 250 ozellige cikmanin kendisi ne kadar zarar veriyor
        _, _, qd = run("dup_control", n_shell=5, agg="ref", dup=True)
        dd = paired_bootstrap(yte, ref, qd)
        res["dup_vs_off"] = dd
        log(f"    yalnizca ozellik ikiye katlanirsa (yeni bilgi yok): {dd['diff']:+.4f} "
            f"[{dd['ci95'][0]:+.4f}, {dd['ci95'][1]:+.4f}]  "
            f"{'ANLAMLI' if dd['significant'] else 'anlamli degil'}")
        log(f"    -> RMSF'in seyrelmeyi asan katkisi: "
            f"{d['diff'] - dd['diff']:+.4f} pK")

    log("\n[D] SINIR AGLARINA KARSI ESLESMIS KARSILASTIRMA (5 kabuk, referans poz)")
    base_npz = os.path.join(a.out, "pred_shell5.npz")
    if not os.path.exists(base_npz):
        base_npz = os.path.join(a.out, "pred_dyn_ref.npz")
    if os.path.exists(base_npz):
        z = np.load(base_npz, allow_pickle=True)
        for path, lab in ((a.net_pair, "cift gosterim (pair)"), (a.net_voxel, "voksel 3B-CNN")):
            d = compare_to(z["ids"].astype(str), z["y"], z["p"], path, lab, log)
            if d:
                res[f"vs_{lab.split()[0]}"] = d
    else:
        log("  temel kol kosulmadi, karsilastirma atlandi")

    json.dump(res, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    open(os.path.join(a.out, "report.txt"), "w").write("\n".join(L) + "\n")
    log(f"\nyazildi: {a.out}/report.txt")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/equiv.py
r"""UCUNCU MIMARI: E(n)-degismez graf sinir agi.

Terim hassasiyeti: mesaj gecisi esdegiskendir, cikti bir SKALER oldugu icin
ag butun kompleksin otelenmesi ve dondurulmesi altinda DEGISMEZDIR. Makalede
de bu ayrim boyle yazilmalidir.


NEDEN BU MODUL VAR
------------------
Tartisma bolumunde su hipotezi one surduk: cift-gosterim ve voksel modellerinde
mesafe kanali TOPLAMSAL giriyor ve havuzlamada mesafeye bagli bir esik yok, bu
yuzden hicbir sey bir temasi temas olarak islemeye zorlamiyor; temas-sayim modeli
ise bastan sona mesafe esiginden ibaret ve geometriyi kullanan tek model o.

Bu modul o hipotezin dogrudan testidir. Esdegisken graf aginda geometri
toplamsal degildir: iki dugum arasindaki mesaj, aralarindaki uzakligin bir
FONKSIYONU olarak uretilir (Satorras ve ark.), yani mesaj mesafeyle kapilanir.
Ustelik kenarlar bir yaricap esigiyle kurulur. Boylece temas-sayim modelinin
geometriyi kullanmasini sagladigini dusundugumuz iki ozellik de burada var:
mesafeyle kapilanan mesaj ve mesafeye bagli kesme.

Bu mimari de ayni tanıyı veriyorsa hipotez zayiflar ve bulgu guclenir; vermiyorsa
"geometriyi hangi mimari sinifi kullanir" sorusunun cevabi bulunmus olur. Her iki
sonuc da ise yarar.

DUGUMLER
--------
Cep kalintilari (kalinti basina bir dugum, agir atomlarin agirlik merkezinde) ve
ligand agir atomlari. Dugum ozellikleri yalnizca SINIF kimligidir (bes kalinti
sinifi, bes ligand atom sinifi, bir de protein/ligand bayragi); konum bilgisi
ozelliklerde degil, yalnizca koordinatlardadir. Boylece "geometriyi kaldir"
mudahalesi gercekten yalnizca geometriyi kaldirir.

KOLLAR
------
  eq        gercek koordinatlar
  eq_const  gercek kenarlar, butun mesafeler sabit -> olcu silinir, topoloji kalir
            (cift-gosterim modelindeki const:8'in birebir karsiligi)
  eq_scr    konumlar kutu icinde rastgele -> hem olcu hem topoloji silinir
  eq_rand   rastgele kalinti cebi, gercek koordinatlar

ESDEGISKENLIK
-------------
Model tum kompleksin otelenmesine ve dondurulmesine gore DEGISMEZDIR; bu bir
tasarim ozelligi degil, dogrulanan bir ozelliktir (tools/test_equiv.py). Sondalar
bu yuzden yalnizca BAGIL geometriyi bozanlardir: ligandi kendi icinde oteleme,
kendi agirlik merkezi etrafinda dondurme, olcekleme ve etiket karistirma.

GPU gerekir. Yuksek RAM GEREKMEZ.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from .contactcount import LIG_CLASS, N_LIG_CLASS, N_RES_CLASS, RES_CLASS, _lig_classes
from .evaluate import regression_metrics
from .features import extract_pocket, extract_pocket_random, featurize_mol, load_ligand_poses, parse_pdb

RADIUS = 10.0          # kenar esigi, angstrom
BOX = 24.0             # scramble kolunda konumlarin cizildigi kutu
N_CLASS = N_RES_CLASS + N_LIG_CLASS


def _rand_rotation(rng):
    q, _ = np.linalg.qr(rng.randn(3, 3))
    if np.linalg.det(q) < 0:
        q[:, 0] = -q[:, 0]
    return q.astype(np.float32)


def _seeded(cid, spec, salt):
    return np.random.RandomState(
        int(hashlib.md5(f"{salt}{spec}{cid}".encode()).hexdigest()[:8], 16))


def perturb(rec, spec, salt=""):
    """Koordinat uzayinda kontrollu hasar. (xyz, cls, is_lig) dondurur."""
    xyz, cls, lig = rec["xyz"].copy(), rec["cls"].copy(), rec["is_lig"]
    name, _, val = spec.partition(":")
    if name in ("", "none"):
        return xyz, cls, lig
    r = _seeded(rec["id"], spec, salt)
    L = lig.astype(bool)
    if name == "translate":
        xyz[L] += r.randn(1, 3).astype(np.float32) * float(val or 1.0)
    elif name == "rotate":
        c = xyz[L].mean(0, keepdims=True)
        xyz[L] = (xyz[L] - c) @ _rand_rotation(r).T + c
    elif name == "scale":
        c = xyz.mean(0, keepdims=True)
        xyz = ((xyz - c) * float(val) + c).astype(np.float32)
    elif name == "permute_res":
        idx = np.where(~L)[0]
        cls[idx] = cls[idx][r.permutation(len(idx))]
    elif name == "permute_atom":
        idx = np.where(L)[0]
        cls[idx] = cls[idx][r.permutation(len(idx))]
    elif name == "scramble":
        c = rec["centre"].reshape(1, 3)
        xyz = (r.uniform(-BOX / 2, BOX / 2, size=xyz.shape).astype(np.float32) + c)
    else:
        raise ValueError(f"bilinmeyen sonda: {spec}")
    return xyz.astype(np.float32), cls, lig


# --------------------------------------------------------------------------- veri kumesi
class EquivDataset(Dataset):
    """Cep kalintisi agirlik merkezleri + ligand agir atomlari, sinif etiketleriyle."""

    def __init__(self, df, pp_pat, lp_pat, name="", arm="eq", cutoff=10.0, max_res=64,
                 max_atoms=72, cache=None):
        self.arm, self.probe = arm, "none"
        if cache and os.path.exists(cache):
            self.recs = list(np.load(cache, allow_pickle=True)["recs"])
            print(f"[esdegisken:{name}] {len(self.recs)} kompleks (onbellekten), kol={arm}")
            return
        self.recs, skipped = [], 0
        for r in df.itertuples():
            cid = str(r.id)
            pp, lp = pp_pat.format(id=cid), lp_pat.format(id=cid)
            if not (os.path.exists(pp) and os.path.exists(lp)):
                skipped += 1
                continue
            mol, poses = load_ligand_poses(lp, max_poses=1)
            lig = featurize_mol(mol, max_atoms=max_atoms) if mol is not None else None
            if lig is None or not poses:
                skipped += 1
                continue
            chains = parse_pdb(pp)
            if arm == "eq_rand":
                sel, _ = extract_pocket_random(chains, poses, max_res=max_res,
                                               rng=_seeded(cid, "", "pock"))
            else:
                sel, _ = extract_pocket(chains, poses, cutoff=cutoff, max_res=max_res)
            if not sel:
                skipped += 1
                continue
            # kalinti dugumu = agir atomlarinin agirlik merkezi
            pxyz = np.stack([np.asarray(chains[ci]["coords"][ri], np.float32).mean(0)
                             for ci, ri in sel])
            pcls = np.array([RES_CLASS.get(chains[ci]["seq"][ri], 0) for ci, ri in sel],
                            dtype=np.int64)
            lxyz = np.asarray(poses[0], np.float32)[: lig["x"].shape[0]]
            lcls = _lig_cls_safe(np.asarray(lig["x"]))[: len(lxyz)] + N_RES_CLASS
            xyz = np.concatenate([pxyz, lxyz], 0).astype(np.float32)
            cls = np.concatenate([pcls, lcls], 0)
            is_lig = np.concatenate([np.zeros(len(pxyz), np.float32),
                                     np.ones(len(lxyz), np.float32)])
            self.recs.append({"id": cid, "y": float(r.y), "xyz": xyz, "cls": cls,
                              "is_lig": is_lig,
                              "centre": lxyz.mean(0).astype(np.float32)})
        print(f"[esdegisken:{name}] {len(self.recs)} kompleks ({skipped} atlandi), kol={arm}")
        if cache:
            os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
            np.savez_compressed(cache, recs=np.array(self.recs, dtype=object))

    def __len__(self):
        return len(self.recs)

    def __getitem__(self, i):
        r = self.recs[i]
        xyz, cls, lig = r["xyz"], r["cls"], r["is_lig"]
        const = False
        if self.arm == "eq_scr":
            xyz, cls, lig = perturb(r, "scramble", salt="arm")
        elif self.arm == "eq_const":
            const = True          # kenarlar gercek konumlardan, mesafeler sabit
        if self.probe and self.probe != "none":
            rec = {"id": r["id"], "xyz": xyz, "cls": cls, "is_lig": lig,
                   "centre": r["centre"]}
            xyz, cls, lig = perturb(rec, self.probe, salt="probe")
        return {"id": r["id"], "y": np.float32(r["y"]), "xyz": xyz, "cls": cls,
                "is_lig": lig, "const": const}


def _lig_cls_safe(x):
    return _lig_classes(x)


def collate(items):
    B = len(items)
    N = max(len(it["xyz"]) for it in items)
    xyz = torch.zeros(B, N, 3)
    cls = torch.zeros(B, N, dtype=torch.long)
    lig = torch.zeros(B, N)
    mask = torch.zeros(B, N, dtype=torch.bool)
    for b, it in enumerate(items):
        n = len(it["xyz"])
        xyz[b, :n] = torch.from_numpy(np.asarray(it["xyz"]))
        cls[b, :n] = torch.from_numpy(np.asarray(it["cls"]))
        lig[b, :n] = torch.from_numpy(np.asarray(it["is_lig"]))
        mask[b, :n] = True
    return {"ids": [it["id"] for it in items], "xyz": xyz, "cls": cls, "is_lig": lig,
            "mask": mask, "const": torch.tensor([bool(it["const"]) for it in items]),
            "y": torch.tensor([it["y"] for it in items], dtype=torch.float32)}


# --------------------------------------------------------------------------- model
class EGNNLayer(nn.Module):
    """E(n)-esdegisken mesaj gecisi.

    Geometri YALNIZCA ||x_i - x_j||^2 uzerinden girer ve mesajin ICINDE uretilir,
    yani mesaj mesafeyle kapilanir. Toplamsal bir mesafe kanali yoktur. Kenarlar
    bir yaricap esigiyle sinirlanir, yani mesafeye bagli kesme de vardir.
    """

    def __init__(self, dim, edge_dim=None):
        super().__init__()
        e = edge_dim or dim
        self.phi_e = nn.Sequential(nn.Linear(2 * dim + 1, e), nn.SiLU(), nn.Linear(e, e), nn.SiLU())
        self.phi_h = nn.Sequential(nn.Linear(dim + e, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.gate = nn.Sequential(nn.Linear(e, 1), nn.Sigmoid())

    def forward(self, h, d2, adj):
        B, N, D = h.shape
        hi = h[:, :, None, :].expand(B, N, N, D)
        hj = h[:, None, :, :].expand(B, N, N, D)
        m = self.phi_e(torch.cat([hi, hj, d2[..., None]], -1))
        m = m * self.gate(m) * adj[..., None]
        agg = m.sum(2) / adj.sum(2).clamp(min=1)[..., None]
        return h + self.phi_h(torch.cat([h, agg], -1))


class EquivNet(nn.Module):
    def __init__(self, dim=128, layers=4, dropout=0.15, radius=RADIUS):
        super().__init__()
        self.radius = radius
        self.emb = nn.Embedding(N_CLASS, dim)
        self.flag = nn.Linear(1, dim)
        self.layers = nn.ModuleList([EGNNLayer(dim) for _ in range(layers)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * dim, dim), nn.SiLU(),
                                  nn.Dropout(dropout), nn.Linear(dim, 2))

    def forward(self, b):
        xyz, mask = b["xyz"], b["mask"]
        h = self.emb(b["cls"]) + self.flag(b["is_lig"][..., None])
        d2 = ((xyz[:, :, None, :] - xyz[:, None, :, :]) ** 2).sum(-1)
        pair = mask[:, :, None] & mask[:, None, :]
        adj = ((d2 <= self.radius ** 2) & pair).float()
        adj = adj * (1 - torch.eye(adj.shape[1], device=adj.device))[None]
        if b["const"].any():                 # olcu silinir, topoloji korunur
            c = b["const"].view(-1, 1, 1).float()
            d2 = d2 * (1 - c) + c * (self.radius ** 2) / 4.0
        d2 = d2 / (self.radius ** 2)         # olcek normalizasyonu
        for L in self.layers:
            h = L(h, d2, adj)
        h = self.norm(h) * mask[..., None]
        n = mask.sum(1, keepdim=True).clamp(min=1)
        pooled = torch.cat([h.sum(1) / n, h.masked_fill(~mask[..., None], -1e9).amax(1)], -1)
        o = self.head(pooled)
        return o[:, 0], o[:, 1].clamp(-6.0, 4.0)


def beta_nll(mu, logvar, y, beta=0.5):
    var = logvar.exp()
    return ((0.5 * (logvar + (y - mu) ** 2 / var)) * var.detach() ** beta).mean()


# --------------------------------------------------------------------------- egitim
def run_epoch(model, loader, dev, opt=None):
    train = opt is not None
    model.train(train)
    tot, n, P, Y, I = 0.0, 0, [], [], []
    for b in loader:
        b = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}
        with torch.set_grad_enabled(train):
            mu, lv = model(b)
            loss = beta_nll(mu, lv, b["y"])
        if train:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
        tot += float(loss.detach()) * len(b["y"])
        n += len(b["y"])
        P.append(mu.detach().float().cpu().numpy())
        Y.append(b["y"].detach().float().cpu().numpy())
        I += b["ids"]
    return tot / max(n, 1), np.concatenate(P), np.concatenate(Y), I


def train_one(seed, tr, va, te, out, dev, epochs=30, bs=16, lr=3e-4, dim=128, workers=2):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = EquivNet(dim=dim).to(dev)
    npar = sum(p.numel() for p in model.parameters())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    kw = dict(num_workers=workers, collate_fn=collate, pin_memory=(dev.type == "cuda"))
    ltr = DataLoader(tr, batch_size=bs, shuffle=True, drop_last=True, **kw)
    lva = DataLoader(va, batch_size=bs, **kw)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(epochs):
        t0 = time.time()
        trl, *_ = run_epoch(model, ltr, dev, opt)
        _, vp, vy, _ = run_epoch(model, lva, dev)
        vr = float(np.sqrt(((vp - vy) ** 2).mean()))
        sched.step(vr)
        if vr < best - 1e-4:
            best, bad = vr, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        print(f"  [tohum {seed}] devir {ep + 1}/{epochs} egitim {trl:.4f} "
              f"dogrulama RMSE {vr:.4f} ({time.time() - t0:.0f} sn)", flush=True)
        if bad >= 6:
            print(f"  [tohum {seed}] erken durdu")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    d = os.path.join(out, f"seed{seed}")
    os.makedirs(d, exist_ok=True)
    torch.save({"state": model.state_dict(), "dim": dim, "seed": seed, "n_param": npar},
               os.path.join(d, "model.pt"))
    res = {"seed": seed, "n_param": npar, "val_rmse": best}
    for nm, ds in (("val", va), ("test", te)):
        if ds is None:
            continue
        _, p, y, ids = run_epoch(model, DataLoader(ds, batch_size=bs, **kw), dev)
        res[nm] = regression_metrics(y, p)
        np.savez(os.path.join(d, f"pred_{nm}.npz"), ids=np.array(ids), y=y, p=p)
    json.dump(res, open(os.path.join(d, "metrics.json"), "w"), indent=2)
    print(f"  [tohum {seed}] test RMSE {res.get('test', {}).get('rmse', float('nan')):.4f} "
          f"({npar / 1e6:.3f} M parametre)")
    return res


DEFAULT_PROBES = ("none", "translate:0.5", "translate:1.0", "translate:2.0", "translate:4.0",
                  "translate:8.0", "rotate", "scale:1.5", "permute_res", "permute_atom",
                  "scramble")


def probe_battery(out, ds, dev, probes=DEFAULT_PROBES, bs=32, workers=2, tag="test"):
    seeds = sorted(d for d in os.listdir(out) if d.startswith("seed"))
    if not seeds:
        raise SystemExit(f"{out} altinda kontrol noktasi yok")
    rows, base = {}, {}
    for sd in seeds:
        ck = torch.load(os.path.join(out, sd, "model.pt"), map_location=dev, weights_only=False)
        model = EquivNet(dim=ck["dim"]).to(dev)
        model.load_state_dict(ck["state"])
        for spec in probes:
            ds.probe = spec
            _, p, y, _ = run_epoch(model, DataLoader(ds, batch_size=bs, num_workers=workers,
                                                     collate_fn=collate), dev)
            m = regression_metrics(y, p)
            if spec == "none":
                base[sd], dmu, rb = p, 0.0, 1.0
            else:
                b = base[sd]
                dmu = float(np.abs(p - b).mean())
                rb = float(np.corrcoef(p, b)[0, 1]) if p.std() > 1e-9 and b.std() > 1e-9 else 0.0
            rows.setdefault(spec, []).append((m["rmse"], m["pearson"], dmu, rb))
            print(f"  [{sd}] {spec:16s} RMSE {m['rmse']:.4f}  d|mu| {dmu:.4f}  r_temel {rb:.4f}",
                  flush=True)
    ds.probe = "none"
    lines = [f"ESDEGISKEN SONDA RAPORU ({tag}, n = {len(ds)}, {len(seeds)} model)", "=" * 62,
             f"{'sonda':18s}{'RMSE':>16s}{'Pearson':>16s}{'d|mu|':>10s}{'r_temel':>10s}"]
    agg = {}
    for spec in probes:
        a = np.array(rows[spec])
        agg[spec] = {"rmse": float(a[:, 0].mean()), "rmse_sd": float(a[:, 0].std()),
                     "pearson": float(a[:, 1].mean()), "dmu": float(a[:, 2].mean()),
                     "rbase": float(a[:, 3].mean())}
        lines.append(f"{spec:18s}{a[:, 0].mean():>9.4f} +/-{a[:, 0].std():<5.3f}"
                     f"{a[:, 1].mean():>9.4f} +/-{a[:, 1].std():<5.3f}"
                     f"{a[:, 2].mean():>10.4f}{a[:, 3].mean():>10.4f}")
    cost = agg["scramble"]["rmse"] - agg["none"]["rmse"]
    rb = agg["scramble"]["rbase"]
    uses = (cost > 0.05) and (rb < 0.99)
    lines += ["", f"geometriyi silmenin bedeli (scramble - none) = {cost:+.4f} pK",
              f"bozulmus tahminlerin korelasyonu                =  {rb:.4f}",
              f"KARAR: {'GEOMETRIYI KULLANIYOR' if uses else 'GEOMETRIYI KULLANMIYOR'}"]
    txt = "\n".join(lines)
    open(os.path.join(out, f"probe_report_{tag}.txt"), "w").write(txt + "\n")
    json.dump(agg, open(os.path.join(out, f"probes_{tag}.json"), "w"), indent=2)
    print("\n" + txt)
    return agg


# --------------------------------------------------------------------------- CLI
def build_sets(a, arm):
    from .data import load_lp_pdbbind
    sp = load_lp_pdbbind(a.splits, clean_level=a.clean_level)
    mk = lambda df, nm: EquivDataset(df, a.protein_pattern, a.ligand_pattern, nm, arm=arm,
                                     cutoff=a.cutoff, max_res=a.max_res,
                                     cache=(os.path.join(a.cache, f"{arm}_{nm}.npz")
                                            if a.cache else None))
    return mk(sp["train"], "train"), mk(sp["val"], "val"), mk(sp["test"], "test")


def main(argv=None):
    p = argparse.ArgumentParser(description="E(n)-esdegisken graf agi (ucuncu mimari)")
    p.add_argument("--arm", default="eq", choices=["eq", "eq_const", "eq_scr", "eq_rand"])
    p.add_argument("--seeds", default="0,1,2")
    p.add_argument("--out", default="runs/eq")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--bs", type=int, default=16)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--splits", default="LP_PDBBind.csv")
    p.add_argument("--clean_level", default="CL1")
    p.add_argument("--protein_pattern", default="pdb/{id}/{id}_protein.pdb")
    p.add_argument("--ligand_pattern", default="pdb/{id}/{id}_ligand.sdf")
    p.add_argument("--cutoff", type=float, default=10.0)
    p.add_argument("--max_res", type=int, default=64)
    p.add_argument("--cache", default="cache_eq")
    p.add_argument("--probes", default=",".join(DEFAULT_PROBES))
    p.add_argument("--probe-only", dest="probe_only", default=None)
    a = p.parse_args(argv)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[esdegisken] cihaz = {dev}, yaricap {RADIUS} A, {N_CLASS} dugum sinifi")
    if dev.type != "cuda":
        print("[esdegisken] !! UYARI: GPU bulunamadi, bu kol CPU'da cok yavastir.")

    if a.probe_only:
        _, _, te = build_sets(a, a.arm)
        probe_battery(a.probe_only, te, dev, probes=tuple(a.probes.split(",")),
                      workers=a.workers, tag="test")
        return

    tr, va, te = build_sets(a, a.arm)
    os.makedirs(a.out, exist_ok=True)
    res = [train_one(int(s), tr, va, te, a.out, dev, a.epochs, a.bs, a.lr, a.dim, a.workers)
           for s in a.seeds.split(",")]
    r = np.array([x["test"]["rmse"] for x in res if "test" in x])
    print(f"\n[esdegisken:{a.arm}] test RMSE tohum basina {r.mean():.4f} +/- "
          f"{r.std(ddof=1) if len(r) > 1 else 0.0:.4f} ({len(r)} tohum)")
    json.dump({"arm": a.arm, "per_seed_rmse": r.tolist(), "mean": float(r.mean()),
               "sd": float(r.std(ddof=1)) if len(r) > 1 else 0.0},
              open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    if a.arm == "eq":
        probe_battery(a.out, te, dev, probes=tuple(a.probes.split(",")), workers=a.workers,
                      tag="test")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/demo_struct.py
"""Self-contained STRUCTURE-MODE demo on the auto-downloading BDB2020+ structures.

This exists so a bare "Run all" can exercise the full structure + multi-state pipeline end to end
without PDBbind registration. BDB2020+ has only 115 complexes and the split here is a plain random
split (NOT leak-proof), so this is a PROOF-OF-RUN, not a benchmark. The real structure-mode result
needs PDBbind training structures (see the notebook's structure-mode section).

    python -m ensaff.demo_struct --esm esm --esm_model facebook/esm2_t33_650M_UR50D --cache_dir /content/emb_cache
"""
from __future__ import annotations

import argparse
import json
import math
import os

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import StructAffinityDataset, collate, load_bdb2020
from .embed import ProteinEmbedder
from .evaluate import conformal_quantile, regression_metrics, uncertainty_metrics
from .model import EnsAff, beta_nll
from .train import predict, set_seed, to_device


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--esm", default="esm", choices=["esm", "onehot"])
    p.add_argument("--esm_model", default="facebook/esm2_t12_35M_UR50D")
    p.add_argument("--data_dir", default=".")
    p.add_argument("--cache_dir", default="")
    p.add_argument("--out", default="runs/struct_demo")
    p.add_argument("--epochs", type=int, default=40)
    p.add_argument("--bs", type=int, default=8)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--max_poses", type=int, default=4)
    p.add_argument("--jitter_states", type=int, default=2)
    p.add_argument("--alpha", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args(argv)
    set_seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("[demo_struct] PROOF-OF-RUN on BDB2020+ (115 complexes, random split) — NOT a leak-proof benchmark.")

    df = load_bdb2020(os.path.join(a.data_dir, "BDB2020+")).sample(frac=1.0, random_state=a.seed).reset_index(drop=True)
    n = len(df)
    tr_df, va_df, te_df = df[: int(0.8 * n)], df[int(0.8 * n): int(0.9 * n)], df[int(0.9 * n):]
    cache = os.path.join(a.cache_dir or a.data_dir,
                         f"emb_{a.esm}_{a.esm_model.split('/')[-1] if a.esm == 'esm' else 'oh'}.pt")
    emb = ProteinEmbedder(a.esm, a.esm_model, cache_path=cache)
    root = os.path.join(a.data_dir, "BDB2020+", "dataset", "{id}")
    mk = lambda d, j, nm: StructAffinityDataset(d, emb, os.path.join(root, "protein.pdb"),  # noqa: E731
                                                os.path.join(root, "ligand.sdf"), jitter_states=j, name=nm)
    tr, va, te = mk(tr_df, a.jitter_states, "train"), mk(va_df, 0, "val"), mk(te_df, 0, "test")
    tr.train_mode = True
    kw = dict(collate_fn=collate, num_workers=2, pin_memory=dev.type == "cuda")
    dl_tr = DataLoader(tr, batch_size=a.bs, shuffle=True, drop_last=len(tr) > a.bs, **kw)
    dl_va, dl_te = DataLoader(va, batch_size=a.bs, collate_fn=collate), DataLoader(te, batch_size=a.bs, collate_fn=collate)

    model = EnsAff(emb.dim, "struct", d=128, c=64, n_blocks=4, n_heads=4).to(dev)
    print(f"[model] {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params, device={dev}")
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-2)
    best, best_state = float("inf"), None
    for ep in range(a.epochs):
        model.train()
        for batch in dl_tr:
            batch = to_device(batch, dev)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                out = model(batch)
            loss = beta_nll(out["mu"], out["var"], batch["y"]) + 0.5 * ((out["mu"] - batch["y"]) ** 2).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        if len(va):
            _, yv, mv, _ = predict(model, dl_va, dev)
            r = regression_metrics(yv, mv)["rmse"]
            if r < best:
                best, best_state = r, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state:
        model.load_state_dict(best_state)
    _, yv, mv, sv = predict(model, dl_va, dev)
    q = conformal_quantile(yv, mv, sv, a.alpha) if len(va) > 5 else float("nan")
    ids, y, mu, sd = predict(model, dl_te, dev)
    res = {"demo": True, "note": "BDB2020+ random split, proof-of-run only",
           "config": {"mode": "struct", "esm": a.esm, "esm_model": a.esm_model if a.esm == "esm" else "onehot",
                      "n_train": int(len(tr_df)), "n_test": int(len(te_df)), "epochs": a.epochs},
           "test": {**regression_metrics(y, mu), **uncertainty_metrics(y, mu, sd, q if not math.isnan(q) else None, a.alpha)}}
    with open(f"{a.out}/metrics.json", "w") as fh:
        json.dump(res, fh, indent=2)
    print(json.dumps(res["test"], indent=2))
    return res


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ensaff/pdbfetch_core.py
"""Extract (protein, ligand) from a raw RCSB PDB entry, using the expected ligand from SMILES.

Why this exists: PDBbind requires registration, but the same structures are freely available from RCSB by
PDB ID. The curation PDBbind adds is mainly "which HETATM group is the actual ligand" - crystallisation
additives, ions and waters must be rejected. Here that choice is made by matching the heavy-atom count of
each candidate group against the heavy-atom count of the SMILES recorded in the LP-PDBBind metadata.
"""
from __future__ import annotations

import os
import sys

# waters, ions and the usual crystallisation additives: never the ligand of interest
JUNK = {
    "HOH", "DOD", "WAT", "SO4", "PO4", "GOL", "EDO", "PEG", "PG4", "PGE", "1PE", "2PE", "MPD", "ACT", "ACY",
    "FMT", "DMS", "TRS", "EPE", "MES", "IMD", "BME", "NO3", "CIT", "FLC", "TLA", "IPA", "MOH", "ETA", "BU3",
    "NH4", "AZI", "SCN", "CO3", "BCT", "OXL", "MLI", "SIN", "TAR", "BEN", "CAC", "NHE", "P6G", "12P", "15P",
    "NA", "K", "MG", "ZN", "CA", "MN", "FE", "FE2", "NI", "CU", "CU1", "CD", "CO", "HG", "CL", "BR", "IOD",
    "F", "LI", "RB", "CS", "SR", "BA", "AU", "AG", "PT", "PB", "XE", "KR", "AR", "UNX", "UNL",
}


def heavy_atoms_from_smiles(smiles: str) -> int | None:
    try:
        from rdkit import Chem, RDLogger
        RDLogger.DisableLog("rdApp.*")
        m = Chem.MolFromSmiles(str(smiles))
        return None if m is None else Chem.RemoveHs(m, sanitize=False).GetNumAtoms()
    except Exception:
        return None


def parse_entry(path: str):
    """Returns (protein_lines, {(resname, chain, resseq): [hetatm lines]}) from the first model."""
    prot, het = [], {}
    with open(path, errors="ignore") as fh:
        for line in fh:
            if line.startswith("ENDMDL"):
                break
            if line[:6] in ("ATOM  ", "HETATM") and line[16] not in (" ", "A", "1"):
                continue                 # alternate conformation: keep only the first
            if line.startswith("ATOM"):
                prot.append(line)
            elif line.startswith("HETATM"):
                resn = line[17:20].strip()
                if resn in ("HOH", "DOD", "WAT"):
                    continue
                if resn == "MSE":          # selenomethionine is part of the protein chain
                    prot.append(line)
                    continue
                het.setdefault((resn, line[21], line[22:27]), []).append(line)
    return prot, het


def _is_hydrogen(line: str) -> bool:
    el = line[76:78].strip()
    return el.upper() == "H" if el else line[12:16].strip().startswith("H")


def _xyz(lines):
    import numpy as np
    return np.array([(float(l[30:38]), float(l[38:46]), float(l[46:54]))
                     for l in lines if not _is_hydrogen(l)], dtype=float)


def _contacts(lig_lines, prot_lines, cutoff=5.0):
    """How many protein atoms sit within `cutoff` of this ligand copy - a proxy for how well the
    binding site is defined. Used to break ties when a structure holds several copies of the ligand."""
    import numpy as np
    L = _xyz(lig_lines)
    if len(L) == 0 or not prot_lines:
        return 0
    P = _xyz(prot_lines)
    if len(P) == 0:
        return 0
    lo, hi = L.min(0) - cutoff, L.max(0) + cutoff          # cheap bounding-box prefilter
    P = P[((P >= lo) & (P <= hi)).all(1)]
    if len(P) == 0:
        return 0
    d = np.sqrt(((P[:, None, :] - L[None, :, :]) ** 2).sum(-1))
    return int((d.min(1) < cutoff).sum())


def pick_ligand(het: dict, want: int | None, prot_lines=None):
    """Choose the HETATM group that best matches the expected heavy-atom count; among equally good
    matches (several copies of the same ligand) prefer the copy with the most protein contacts."""
    cands = []
    for key, lines in het.items():
        if key[0] in JUNK:
            continue
        n = sum(1 for l in lines if not _is_hydrogen(l))
        if n < 5:                       # too small to be a drug-like ligand
            continue
        cands.append((key, lines, n))
    if not cands:
        return None, None, "aday yok"
    if want:
        best = min(abs(c[2] - want) for c in cands)
        tied = [c for c in cands if abs(c[2] - want) == best]
        # Several copies of the same ligand: take the first one in file order. A "most protein contacts"
        # tie-break was tried and matched the curated PDBbind choice far less often (78/114 vs 109/114),
        # so file order is what the curation effectively follows.
        key, lines, n = tied[0]
        how = ("tam eslesme" if n == want else f"yakin eslesme ({n} vs {want})") + \
              (f" [{len(tied)} kopya, ilki]" if len(tied) > 1 else "")
        return key, lines, how
    key, lines, n = max(cands, key=lambda c: c[2])
    return key, lines, "en buyuk grup (SMILES yok)"


def extract(entry_path: str, smiles: str | None, out_dir: str):
    want = heavy_atoms_from_smiles(smiles) if smiles else None
    prot, het = parse_entry(entry_path)
    key, lig_lines, how = pick_ligand(het, want, prot)
    if key is None:
        return None
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "protein.pdb"), "w") as fh:
        fh.writelines(prot)
        fh.write("END\n")
    with open(os.path.join(out_dir, "ligand.pdb"), "w") as fh:
        fh.writelines(lig_lines)
        fh.write("END\n")
    return {"resname": key[0], "chain": key[1], "how": how,
            "n_heavy": sum(1 for l in lig_lines if not _is_hydrogen(l)), "want": want,
            "n_protein_atoms": len(prot)}


if __name__ == "__main__":
    print(extract(sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else None, sys.argv[3]))


def ligand_pdb_to_sdf(lig_pdb: str, smiles: str, out_sdf: str) -> str | None:
    """A PDB ligand carries no bond orders; recover them by matching against the SMILES template
    (RDKit AssignBondOrdersFromTemplate). Returns a short status string, or None on failure."""
    from rdkit import Chem, RDLogger
    from rdkit.Chem import AllChem
    RDLogger.DisableLog("rdApp.*")
    mol = Chem.MolFromPDBFile(lig_pdb, removeHs=True, sanitize=False)
    if mol is None:
        return None
    tpl = Chem.MolFromSmiles(str(smiles))
    if tpl is None:
        return None
    try:
        fixed = AllChem.AssignBondOrdersFromTemplate(tpl, mol)
        status = "sablonla eslesti"
    except Exception:
        try:                                   # fall back to the raw connectivity RDKit inferred
            Chem.SanitizeMol(mol, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE,
                             catchErrors=True)
            fixed, status = mol, "sablon basarisiz, ham baglar"
        except Exception:
            return None
    w = Chem.SDWriter(out_sdf)
    w.write(fixed)
    w.close()
    return status


In [ ]:
%%writefile ensaff/fetch_structures.py
r"""Build the structure folder for LP-PDBBind from RCSB PDB - no PDBbind registration needed.

LP-PDBBind ships metadata only (sequence, SMILES, affinity); the 3D files normally come from PDBbind,
which requires registration. The same structures are openly available from RCSB by PDB ID, so this script
downloads them and reproduces the two files the structure-mode dataset expects:

    <out>/<id>/protein.pdb     standard-residue ATOM records (full chains, so ESM sees full context)
    <out>/<id>/ligand.sdf      the bound ligand, bond orders recovered from the LP-PDBBind SMILES

Which HETATM group is the ligand is decided by matching heavy-atom counts against that SMILES. On the 115
BDB2020+ complexes, where a curated ligand is available for comparison, this picked the correct ligand for
114/115 and reproduced the curated coordinates exactly for 109/115 (the other 5 are the same ligand in a
symmetry-equivalent copy). Bond-order assignment succeeded from the template for 105 and fell back to raw
connectivity for 9, with all 114 usable by the featurizer.

    python -m ensaff.fetch_structures --out /content/pdbstruct --workers 8
"""
from __future__ import annotations

import argparse
import os
import random
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

from .data import load_lp_pdbbind
from .pdbfetch_core import extract, ligand_pdb_to_sdf

RCSB = "https://files.rcsb.org/download/{id}.pdb"


def fetch_one(pdb_id: str, smiles: str, out_root: str, retries: int = 3, timeout: int = 30):
    pid = str(pdb_id).lower()
    d = os.path.join(out_root, pid)
    if os.path.exists(os.path.join(d, "ligand.sdf")) and os.path.exists(os.path.join(d, "protein.pdb")):
        return pid, "zaten var"
    raw = os.path.join(out_root, "_raw", f"{pid}.pdb")
    os.makedirs(os.path.dirname(raw), exist_ok=True)
    if not os.path.exists(raw):
        for k in range(retries):
            try:
                req = urllib.request.Request(RCSB.format(id=pid), headers={"User-Agent": "ensaff-research/1.0"})
                with urllib.request.urlopen(req, timeout=timeout) as r, open(raw, "wb") as fh:
                    fh.write(r.read())
                break
            except urllib.error.HTTPError as e:
                if e.code == 404:
                    return pid, "RCSB'de yok (404)"
                time.sleep(1.5 * (k + 1) + random.random())
            except Exception:
                time.sleep(1.5 * (k + 1) + random.random())
        else:
            return pid, "indirilemedi"
    info = extract(raw, smiles, d)
    if info is None:
        return pid, "ligand bulunamadi"
    st = ligand_pdb_to_sdf(os.path.join(d, "ligand.pdb"), smiles, os.path.join(d, "ligand.sdf"))
    if st is None:
        return pid, "sdf uretilemedi"
    try:
        os.remove(raw)                       # raw entry is no longer needed; keeps the folder small
    except OSError:
        pass
    return pid, ("tamam" if "eslesti" in st else "tamam (ham baglar)")


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", default=".")
    p.add_argument("--out", default="/content/pdbstruct")
    p.add_argument("--workers", type=int, default=8)
    p.add_argument("--limit", type=int, default=0)
    p.add_argument("--clean_level", default="CL1")
    a = p.parse_args(argv)
    tabs = load_lp_pdbbind(os.path.join(a.data_dir, "LP_PDBBind.csv"), clean_level=a.clean_level)
    df = pd.concat([tabs[s] for s in ("train", "val", "test")], ignore_index=True)
    if a.limit:
        df = df.head(a.limit)
    os.makedirs(a.out, exist_ok=True)
    print(f"[fetch] {len(df)} kompleks -> {a.out}  ({a.workers} paralel indirme)")
    t0, done = time.time(), []
    with ThreadPoolExecutor(max_workers=a.workers) as ex:
        futs = [ex.submit(fetch_one, r.id, r.smiles, a.out) for r in df.itertuples()]
        for i, f in enumerate(futs):
            done.append(f.result())
            if (i + 1) % 500 == 0:
                print(f"  {i + 1}/{len(df)}  ({time.time() - t0:.0f}s)", flush=True)
    counts = pd.Series([s for _, s in done]).value_counts()
    print("\n[fetch] sonuc:")
    for k, v in counts.items():
        print(f"  {k:<28} {v}")
    usable = sum(v for k, v in counts.items() if k.startswith("tamam") or k == "zaten var")
    print(f"\n[fetch] kullanilabilir: {usable}/{len(df)} (%{100 * usable / max(1, len(df)):.1f})")
    pd.DataFrame(done, columns=["id", "durum"]).to_csv(os.path.join(a.out, "_fetch_log.csv"), index=False)
    return counts


if __name__ == "__main__":
    main()


## Veri — dış setler artık LP val/test'e karşı tekilleştiriliyor

In [ ]:
from ensaff.data import load_lp_pdbbind, load_external_sets
t = load_lp_pdbbind('LP_PDBBind.csv')
print({k: len(v) for k, v in t.items()})
for n, (df, pp, lp) in load_external_sets('.').items():
    print(f'  dis set {n}: n={len(df)}  y={df.y.min():.2f}-{df.y.max():.2f}')

## Yapı dosyaları  *(CPU + internet, GPU yok)*

In [ ]:
# =====================================================================
# 1b/4 YAPI DOSYALARI (RCSB) | CPU + internet (GPU YOK) | 0-30 dk
# Arsiv once YERELE kopyalanir, oradan acilir: tar'i Drive uzerinden acmak
# hem yavas hem de FUSE kopmasina acik.
# =====================================================================
TAR_LOCAL = "/content/pdbstruct.tar"
TAR_DRIVE = os.path.join(DRIVE_ROOT, "pdbstruct.tar")
n0 = len(glob.glob(os.path.join(STRUCT, "*", "ligand.sdf")))
if n0 < 500 and HAS_DRIVE and os.path.exists(TAR_DRIVE):
    print("[fetch] Drive arsivinden geri yukleniyor...")
    try:
        shutil.copyfile(TAR_DRIVE, TAR_LOCAL)
        os.system(f"tar -xf {TAR_LOCAL} -C {STRUCT}")
    except Exception as e:
        print("[fetch] arsiv alinamadi:", e)
os.system(f"python -m ensaff.fetch_structures --out {STRUCT} --workers 12")
n = len(glob.glob(os.path.join(STRUCT, "*", "ligand.sdf")))
print(f"[fetch] hazir yapi: {n}")
if n > 500 and HAS_DRIVE and not os.path.exists(TAR_DRIVE):
    os.system(f"cd {STRUCT} && tar -cf {TAR_LOCAL} .")
    try:
        shutil.copyfile(TAR_LOCAL, TAR_DRIVE); print("[fetch] Drive yedegi tamam")
    except Exception as e:
        print("[fetch] Drive yedegi yazilamadi:", e)
# veri setleri ilk kez indirildiyse Drive'a da yaz (sonraki oturumlar hizli acilsin)
for sub in ("LP_PDBBind.csv", "BDB2020+", "mpro", "EGFR"):
    if os.path.exists(os.path.join(WORK, sub)) and not os.path.exists(os.path.join(DRIVE_ROOT, sub)):
        drive_push(sub)


## A BÖLÜMÜ — üçüncü mimari  *(A100 GPU, High-RAM gerekmez)*

In [ ]:
# =====================================================================
# A1/5 VERI ONBELLEGI | CPU yeterli (GPU KULLANILMAZ) | 20-30 dk, BIR KEZ
# Dugumler: cep kalintilarinin agirlik merkezleri + ligand agir atomlari.
# Dugum ozellikleri yalnizca SINIF kimligi; konum bilgisi ozelliklerde DEGIL,
# yalnizca koordinatlarda. Boylece "geometriyi kaldir" gercekten yalnizca
# geometriyi kaldiriyor.
# =====================================================================
import os, glob, json, shutil
assert os.getcwd() == WORK, "calisma dizini bozulmus -> once 1/4 KURULUM hucresini calistirin"
PAT = f"--protein_pattern '{STRUCT}/{{id}}/protein.pdb' --ligand_pattern '{STRUCT}/{{id}}/ligand.sdf'"
EQCACHE = "/content/cache_eq"; EQDRIVE = os.path.join(DRIVE_ROOT, "cache_eq")
os.makedirs(EQCACHE, exist_ok=True)

def sh(c):
    print(f"\n$ {c}", flush=True); rc = os.system(c); print(f"[exit {rc}]", flush=True); return rc
def sh_once(outdir, c, marker="summary.json"):
    if os.path.exists(os.path.join(outdir, marker)):
        print(f"[atla] {outdir} zaten var"); return 0
    return sh(c)

if HAS_DRIVE and os.path.isdir(EQDRIVE) and not glob.glob(EQCACHE + "/*.npz"):
    try:
        shutil.copytree(EQDRIVE, EQCACHE, dirs_exist_ok=True); print("[eq] onbellek geri yuklendi")
    except Exception as e: print("[eq] onbellek alinamadi:", e)

assert len(glob.glob(os.path.join(STRUCT, "*", "ligand.sdf"))) > 500, "yapi dosyalari yok"
EQB = f"python -m ensaff.equiv --cache {EQCACHE} {PAT} --epochs 30 --bs 16 --workers 2"
# kontrol noktasi yok, sonda adiminda hata verir; amac yalnizca onbellegi kurmak
sh(f"{EQB} --arm eq --out runs/eq --probe-only runs/eq --probes none 2>&1 | tail -15")
print("\n[eq] onbellek:", sorted(os.path.basename(p) for p in glob.glob(EQCACHE + "/*.npz")))
if HAS_DRIVE:
    try: shutil.copytree(EQCACHE, EQDRIVE, dirs_exist_ok=True)
    except Exception as e: print("[eq] yedek yazilamadi:", e)

In [ ]:
# =====================================================================
# A2/5 GERCEK KOORDINATLAR | A100 GPU | High-RAM GEREKMEZ | ~1,5 sa (5 tohum)
# =====================================================================
sh_once("runs/eq", f"{EQB} --arm eq --seeds 0,1,2,3,4 --out runs/eq")
drive_push("runs")
if os.path.exists("runs/eq/probe_report_test.txt"):
    print(open("runs/eq/probe_report_test.txt").read())

In [ ]:
# =====================================================================
# A3/5 SABIT OLCU KOLU | A100 GPU | High-RAM GEREKMEZ | ~50 dk (3 tohum)
# Kenarlar GERCEK konumlardan kurulur, ama butun mesafeler sabitlenir.
# Topoloji kalir, olcu silinir: cift-gosterim modelindeki const:8'in birebir karsiligi.
# =====================================================================
sh_once("runs/eq_const", f"{EQB} --arm eq_const --seeds 0,1,2 --out runs/eq_const")
drive_push("runs")

In [ ]:
# =====================================================================
# A4/5 KARISTIRILMIS KONUM KOLU | A100 GPU | High-RAM GEREKMEZ | ~50 dk (3 tohum)
# Konumlar kutu icinde rastgele: hem olcu hem topoloji silinir, dugum bilesimi korunur.
# =====================================================================
sh_once("runs/eq_scr", f"{EQB} --arm eq_scr --seeds 0,1,2 --out runs/eq_scr")
drive_push("runs")
for t in ("eq", "eq_const", "eq_scr"):
    p = f"runs/{t}/summary.json"
    if os.path.exists(p):
        d = json.load(open(p)); print(f"{t:9s} test RMSE {d['mean']:.4f} +/- {d['sd']:.4f}  {d['per_seed_rmse']}")

In [ ]:
# =====================================================================
# A5/5 SONDA BATARYASI + ESLESMIS KARSILASTIRMALAR | A100 GPU | ~10 dk
# =====================================================================
sh(f"{EQB} --arm eq --out runs/eq --probe-only runs/eq")
drive_push("runs")

# eq / eq_const / eq_scr arasinda eslesmis onyukleme
import numpy as np
from ensaff.ccdyn import paired_bootstrap
def ens(d):
    fs = sorted(glob.glob(os.path.join(d, "seed*", "pred_test.npz")))
    if not fs: return None
    Z = [np.load(f, allow_pickle=True) for f in fs]
    ids = Z[0]["ids"].astype(str); o = np.argsort(ids)
    return ids[o], Z[0]["y"][o], np.mean([z["p"][np.argsort(z["ids"].astype(str))] for z in Z], 0)
A = ens("runs/eq")
if A:
    for tag in ("eq_const", "eq_scr"):
        B = ens(f"runs/{tag}")
        if not B: continue
        common = np.intersect1d(A[0], B[0])
        sa, sb = np.isin(A[0], common), np.isin(B[0], common)
        d = paired_bootstrap(A[1][sa], A[2][sa], B[2][sb])
        print(f"  {tag:9s} vs eq: {d['diff']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]  "
              f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}  n={len(common)}")

## B BÖLÜMÜ — v16'dan kalan iki küçük açık  *(CPU, ~15 dk)*

In [ ]:
# =====================================================================
# B/5 KUCUK DUZELTMELER | CPU yeterli | ~15 dk
#  1) voksel karsilastirmasi ADIL kosullarda: topluluk vs topluluk
#  2) RMSF kolunun SEYRELME kontrolu (125 ozelligi ikiye katla, yeni bilgi yok)
# =====================================================================
import numpy as np, glob, os
from ensaff.ccdyn import paired_bootstrap

# 1) voksel toplulugu kur ve temas sayimina karsi karsilastir
vz = sorted(glob.glob("runs/vx/seed*/pred_test.npz"))
cc = "runs/ccdyn/pred_shell5.npz"
if vz and os.path.exists(cc):
    Z = [np.load(f, allow_pickle=True) for f in vz]
    vids = Z[0]["ids"].astype(str); o = np.argsort(vids)
    vids, vy = vids[o], Z[0]["y"][o]
    vp = np.mean([z["p"][np.argsort(z["ids"].astype(str))] for z in Z], 0)
    c = np.load(cc, allow_pickle=True); cid = c["ids"].astype(str); co = np.argsort(cid)
    cid, cy, cp = cid[co], c["y"][co], c["p"][co]
    common = np.intersect1d(vids, cid)
    sa, sb = np.isin(vids, common), np.isin(cid, common)
    d = paired_bootstrap(vy[sa], vp[sa], cp[sb])
    print(f"[adil] temas sayimi vs voksel TOPLULUGU ({len(vz)} tohum): "
          f"voksel {np.sqrt(((vp[sa]-vy[sa])**2).mean()):.4f}  "
          f"temas {np.sqrt(((cp[sb]-cy[sb])**2).mean()):.4f}  "
          f"fark {d['diff']:+.4f} [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}] "
          f"{'ANLAMLI' if d['significant'] else 'anlamli degil'}  n={len(common)}")
else:
    print("[adil] voksel veya temas tahminleri yok, atlandi")

# 2) seyrelme kontrolu: ccdyn'i yalnizca rmsf kolu icin yeniden kosar (dup dahil)
sh(f"python -m ensaff.ccdyn --arms rmsf --seeds 0,1,2,3,4 {PAT} "
   f"--cache /content/cache_ccdyn --out runs/ccdyn_dup")
drive_push("runs")
if os.path.exists("runs/ccdyn_dup/report.txt"):
    print(open("runs/ccdyn_dup/report.txt").read())

## RAPOR  *(CPU, ~1 dk)*

Aşağıdaki bloğu olduğu gibi kopyalayıp bana gönderin.

In [ ]:
# =====================================================================
# RAPOR | CPU | ~1 dk
# =====================================================================
import os, json, glob, datetime, numpy as np
L = []
def w(s=""):
    print(s); L.append(s)
w("=" * 72); w("EnsAff v17 RAPORU   " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M")); w("=" * 72)

w("\n[A] UCUNCU MIMARI - E(n)-DEGISMEZ GRAF AGI")
for t in ("eq", "eq_const", "eq_scr", "eq_rand"):
    p = f"runs/{t}/summary.json"
    if os.path.exists(p):
        d = json.load(open(p))
        w(f"  {t:9s} tohum basina {d['mean']:.4f} +/- {d['sd']:.4f}  ({len(d['per_seed_rmse'])} tohum)")
    else:
        w(f"  {t:9s} kosu yok")
for f in ("runs/eq/probe_report_test.txt",):
    if os.path.exists(f): w("\n" + open(f).read().rstrip())

w("\n[B] KUCUK DUZELTMELER")
if os.path.exists("runs/ccdyn_dup/report.txt"):
    w(open("runs/ccdyn_dup/report.txt").read().rstrip())
else:
    w("  seyrelme kontrolu kosusu yok")

w("\n[ORTAM]")
try:
    import torch
    w(f"  torch {torch.__version__}  gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'yok'}")
except Exception as e: w(f"  okunamadi: {e}")
w(f"  BUILD {BUILD}")
txt = "\n".join(L)
fn = f"EnsAff_rapor_v17_{datetime.datetime.now():%Y%m%d_%H%M}.txt"
open(fn, "w").write(txt + "\n")
try: drive_push(fn)
except Exception: pass
try:
    from google.colab import files; files.download(fn)
except Exception: pass
print("\n\n### BU DOSYAYI GONDERIN:", fn)